In [950]:
# %config InteractiveShell.xmode = "Context"

In [951]:
# %xmode

In [952]:
import re

from datetime import datetime

import rdflib
from rdflib import Graph, RDF, URIRef, RDFS, OWL, Literal, Namespace, BNode, XSD
from rdflib.collection import Collection

import psycopg2

from collections import defaultdict, Counter, deque

from decimal import Decimal

from copy import deepcopy

from dateutil.parser import parse
from urllib.parse import urlparse

import enchant
import nltk
from nltk.corpus import words

# nltk.download('words')
english_words = set(words.words())
# dict_en = enchant.Dict("en_US")


import os, hashlib, requests, logging, warnings, shutil
from concurrent.futures import ThreadPoolExecutor, as_completed

logging.getLogger("rdflib").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")


#### Evaluation ####
import time
from datetime import datetime
import tempfile
import csv
import os
import pandas as pd

## Evaluation - Part 1

In [953]:
# ---------- TIME MEASUREMENT ----------
TIMESTAMPS = {}

def mark(name: str):
    """Save a timestamp for a named checkpoint."""
    TIMESTAMPS[name] = time.time()

def elapsed(start: str, end: str) -> float:
    """Calculate elapsed seconds between two checkpoints."""
    return TIMESTAMPS[end] - TIMESTAMPS[start]


# ---------- LOGGING ----------
def append_to_file(log_path: str, text: str):
    """Append text to a file without overwriting it."""
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(text + "\n")

# Ontology Shredder

In [954]:
### Anreicherung vom Graph

# -------------------------------------------------------------
# 1. Cache handling + optional reset
# -------------------------------------------------------------
def reset_cache(cache_dir="cache"):
    """Delete the cache directory (used to clear bad HTML responses)."""
    shutil.rmtree(cache_dir, ignore_errors=True)
    os.makedirs(cache_dir, exist_ok=True)


# -------------------------------------------------------------
# 2. Cached dereference (safe RDF-only)
# -------------------------------------------------------------
def cached_dereference(uri: str, cache_dir="cache", timeout=5) -> Graph:
    """
    Download RDF data for a URI with local caching and strict content-type filtering.
    Returns an RDFLib Graph (empty if not dereferenceable or invalid RDF).
    """
    os.makedirs(cache_dir, exist_ok=True)
    cache_file = os.path.join(cache_dir, hashlib.md5(uri.encode()).hexdigest() + ".ttl")

    # Use cached version if available
    if os.path.exists(cache_file):
        g = Graph()
        try:
            g.parse(cache_file)
            return g
        except Exception:
            os.remove(cache_file)  # corrupted cache → re-download

    g = Graph()
    try:
        headers = {"Accept": "text/turtle, application/rdf+xml, application/ld+json;q=0.9"}
        r = requests.get(uri, headers=headers, timeout=timeout)

        # Strictly allow only RDF content
        ct = r.headers.get("Content-Type", "").lower()
        if not any(fmt in ct for fmt in ["turtle", "rdf+xml", "ld+json", "n3"]):
            return Graph()

        if r.ok:
            g.parse(data=r.text)
            g.serialize(cache_file, format="turtle")
    except Exception:
        pass

    return g


# -------------------------------------------------------------
# 3. Parallel dereferencing
# -------------------------------------------------------------
def dereference_many_uris(uris: list[str], max_workers=10, timeout=5) -> dict:
    """
    Parallel dereferencing of multiple URIs with caching.
    Returns: dict {uri -> Graph}
    """
    results = {}
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_uri = {
            executor.submit(cached_dereference, uri, "cache", timeout): uri
            for uri in uris
        }

        for future in as_completed(future_to_uri):
            uri = future_to_uri[future]
            try:
                g = future.result()
                results[uri] = g
            except Exception:
                results[uri] = Graph()

    return results


# -------------------------------------------------------------
# 4. Clean invalid triples (DOCTYPE, non-URI predicates, etc.)
# -------------------------------------------------------------
def clean_invalid_triples(g: Graph) -> Graph:
    """
    Remove invalid triples where predicates are not URIRefs or contain garbage text.
    """
    remove = []
    for s, p, o in g:
        if not isinstance(p, URIRef):
            remove.append((s, p, o))
        else:
            p_str = str(p)
            if "DOCTYPE" in p_str or "xml" in p_str or "version=" in p_str:
                remove.append((s, p, o))
    for triple in remove:
        g.remove(triple)
    return g


# -------------------------------------------------------------
# 5. Combine ontology + external RDF in parallel, count triples
# -------------------------------------------------------------
def load_complete_graph_parallel_clean(local_file, max_workers=15, timeout=5, reset=False):
    """
    Load local ontology + external RDF data in parallel.
    Includes:
      - caching
      - strict RDF content-type filtering
      - invalid triple cleanup
      - triple count summary
    """
    if reset:
        reset_cache()

    combined = Graph()
    combined.parse(local_file)
    initial_triples = len(combined)

    # Identify ontology URI
    ontology_uris = [str(o) for o in combined.subjects(RDF.type, OWL.Ontology)]
    ontology_uri = ontology_uris[0] if ontology_uris else ""

    # Collect external URIs
    all_uris = {str(s) for s in combined.all_nodes() if isinstance(s, URIRef)}
    external_uris = [
        u for u in all_uris
        if ontology_uri not in u and "w3.org" not in u and "xmlschema" not in u
    ]

    print(f"Found {len(external_uris)} external URIs, fetching with {max_workers} threads...\n")

    results = dereference_many_uris(external_uris, max_workers=max_workers, timeout=timeout)

    total_added = 0
    for uri, ext_g in results.items():
        triples_before = len(combined)
        if len(ext_g) > 0:
            combined += ext_g
            added = len(combined) - triples_before
            total_added += added

    # Clean invalid data
    before_clean = len(combined)
    combined = clean_invalid_triples(combined)
    removed = before_clean - len(combined)

    # Summary
    print(f"\n✅ Combined graph complete — total triples: {len(combined)}, new triples: {total_added}")

    return combined


In [955]:
LOG_FILE = "shredder_runtime_log.txt"

mark("start_total")
append_to_file(LOG_FILE, "\n\n===== NEW RUN: " + datetime.now().isoformat() + " =====")


In [956]:
ttl_file = r"C:\Users\ilove\OneDrive\Uni\Master - Philipps Uni\Master Thesis\Ontos für Evaluation\wsb-2.0_reasoning.ttl"
ttl_file2 = r"C:\Users\ilove\OneDrive\Uni\Master - Philipps Uni\Master Thesis\Ontos für Evaluation\Original Datei\wasabi-2-0\rdf\album.ttl"
# ttl_file3 = r"C:\Users\ilove\OneDrive\Uni\Master - Philipps Uni\Master Thesis\Ontos für Evaluation\Original Datei\wasabi-2-0\rdf\artist.ttl"
# ttl_file = r"C:\Users\ilove\OneDrive\Uni\Master - Philipps Uni\Master Thesis\Ontos für Evaluation\geo-all_reasoning.ttl"
# ttl_file = r"C:\Users\ilove\OneDrive\Uni\Master - Philipps Uni\Master Thesis\Ontos für Evaluation\FISHO_resoning.ttl"
# ttl_file = r"C:\Users\ilove\OneDrive\Uni\Master - Philipps Uni\Master Thesis\Ontos für Evaluation\hso_reasoning.ttl"
# ttl_file = r"C:\Users\ilove\OneDrive\Uni\Master - Philipps Uni\Master Thesis\Ontos für Evaluation\MOSAIC_reasoning.ttl"
# ttl_file = r"C:\Users\ilove\OneDrive\Uni\Master - Philipps Uni\Master Thesis\Ontos für Evaluation\mwo_rasoning.ttl"

# g = Graph()
# g.parse(ttl_file, format="turtle")


tbox = Graph()
tbox.parse(ttl_file, format="turtle")
abox1 = Graph()
abox1.parse(ttl_file2, format="turtle")

g = Graph()
for triple in tbox:
    g.add(triple)
for triple in abox1:
    g.add(triple)
print(f"📦 Loaded graph with {len(g)} triples from {ttl_file} and {ttl_file2}")

mark("ontology_loaded")


PROPERTY_INFO_MAP = {}
CLASS_PROPERTY_MAP = {}
CLASS_INFO_MAP = {}

# a = load_complete_graph_parallel_clean(ttl_file, max_workers=15, timeout=5, reset=False)
# g = a

# output_file = f"{ttl_file.replace(".ttl", "_combined2")}.ttl"
# g.serialize(output_file, format="turtle")
# print(f"💾 Saved combined graph to: {output_file}")



📦 Loaded graph with 2982804 triples from C:\Users\ilove\OneDrive\Uni\Master - Philipps Uni\Master Thesis\Ontos für Evaluation\wsb-2.0_reasoning.ttl and C:\Users\ilove\OneDrive\Uni\Master - Philipps Uni\Master Thesis\Ontos für Evaluation\Original Datei\wasabi-2-0\rdf\album.ttl


In [957]:
# def save_graph_turtle(graph: Graph, filepath: str):
#     """
#     Serialize an RDF graph as Turtle and save safely to a file.
#     Handles both str and bytes returned by rdflib.
#     """
#     data = graph.serialize(format="turtle")

#     # Ensure bytes
#     if isinstance(data, str):
#         data = data.encode("utf-8")

#     with open(filepath, "wb") as f:
#         f.write(data)

#     print(f"Graph successfully saved to: {filepath}")


# save_graph_turtle(a, "reasoning_geo-all_combined.ttl")

## α. remove_prefix und add_underscore_before_caps

In [958]:
def analyze_prefix_usage(graph = g):
    """
    Analyze a RDF/OWL file and classify prefixes into internal/external.
    The detection is robust against incomplete URIs, varying namespace depth,
    and domain-level variations (e.g. /vocab/ vs /cbo/).
    """
    g = graph

    db_schema = None
    
    ontology_uris = [str(o) for o in g.subjects(RDF.type, OWL.Ontology)]
    ontology_uri = ontology_uris[0] if ontology_uris else None

    prefix_map = {str(ns): prefix for prefix, ns in g.namespaces()}

    used_namespaces = set()

    for s, p, o in g:
        for term in (s, p, o):
            if isinstance(term, URIRef):
                uri = str(term)
                parsed = urlparse(uri)

                # Safe extraction of the namespace "base"
                if "#" in uri:
                    base = uri.rsplit("#", 1)[0] + "#"
                elif parsed.path and parsed.path != "/":
                    base = uri.rsplit("/", 1)[0] + "/"
                else:
                    base = f"{parsed.scheme}://{parsed.netloc}/"

                used_namespaces.add(base)

    all_namespaces = set(prefix_map.keys()) | used_namespaces
    
    # Helper: compare by domain 
    def same_domain(u1, u2):
        p1, p2 = urlparse(u1), urlparse(u2)
        return p1.netloc == p2.netloc and bool(p1.netloc)

    # Determine domain root of ontology
    domain_root = None
    if ontology_uri:
        parsed_base = urlparse(ontology_uri)
        db_schema = (parsed_base.netloc).rsplit(".", 1)[0]
        domain_root = f"{parsed_base.scheme}://{parsed_base.netloc}/"

    internal = []
    external = []

    for ns in all_namespaces:
        if any(str(ns).startswith(u) or u.startswith(str(ns)) or same_domain(str(ns), u)
               for u in used_namespaces):

            # Case 1: direct ontology namespace
            if ontology_uri and str(ns).startswith(ontology_uri):
                internal.append(ns)

            # Case 2: same domain as ontology (e.g., /vocab/, /schema/)
            elif domain_root and str(ns).startswith(domain_root):
                internal.append(ns)

            else:
                external.append(ns)

    return {
        "ontology_uri": ontology_uri,
        "domain_root": domain_root,
        "internal": internal,
        "external": external,
        "used_namespaces": used_namespaces,
        "db_schema": db_schema,
    }


ALL_NAMESPACES = analyze_prefix_usage()
ONTOLOGY_URI = ALL_NAMESPACES["ontology_uri"]
PREFIXES_TO_REMOVE = ALL_NAMESPACES["used_namespaces"]
INTERNAL_PREFIXES = ALL_NAMESPACES["internal"]
EXTERNAL_PREFIXES = ALL_NAMESPACES["external"]

In [959]:
def make_db_schema_from_uri(uri = ONTOLOGY_URI):
    """
    Generate a clean, SQL-safe schema name from an ontology URI.
    Fallback priority:
      1. Last meaningful path segment (without file extensions)
      2. Domain name without tld (and no www)
      3. Fragment identifier
    """
    parsed = urlparse(uri)
    netloc = parsed.netloc
    path = parsed.path.strip("/")
    fragment = parsed.fragment

    if netloc.startswith("www."):
        netloc = netloc[4:]

    base = None

    if path:
        parts = [p for p in path.split("/") if p]
        last_segment = parts[-1]

        # Remove known file extensions
        last_segment = re.sub(r"\.(owl|rdf|ttl|xml|jsonld|nt|n3)$", "", last_segment, flags=re.IGNORECASE)

        # If too short or generic, use previous segment instead
        if not re.match(r"^[A-Za-z0-9_-]{3,}$", last_segment):
            if len(parts) > 1:
                last_segment = parts[-2]
            else:
                last_segment = netloc.split(".")[0]
        base = last_segment
    else:
        base = netloc.split(".")[0]

    schema = re.sub(r"[^A-Za-z0-9_]", "_", base.lower())

    if not schema and fragment:
        schema = re.sub(r"[^A-Za-z0-9_]", "_", fragment.lower())

    if not schema[0].isalpha():
        schema = "s_" + schema

    return schema

In [960]:
def is_internal(elem, internal_prefixes = INTERNAL_PREFIXES):
    """   
    Checks if an RDF element (URIRef) belongs to an internal prefix.

    Considers:
        - Main ontology URI
        - All internal prefixes from def analyze_prefix_usage
        - Related namespaces with the same domain root
    """
    if not isinstance(elem, URIRef):
        return False

    uri = str(elem)

    # direkte Prüfung: kommt URI aus bekanntem internen Namespace?
    if any(uri.startswith(ns) for ns in internal_prefixes):
        return True

    return False


def is_external(elem, external_prefixes = EXTERNAL_PREFIXES):
    """
    Checks if an RDF element (URIRef) belongs to an external prefix.

    Considers:
        - Main ontology URI
        - All external prefixes from def analyze_prefix_usage
        - Related namespaces with the same domain root
    """
    if not isinstance(elem, URIRef):
        return False

    uri = str(elem)

    # direkte Prüfung: kommt URI aus bekanntem internen Namespace?
    if any(uri.startswith(ns) for ns in external_prefixes):
        return True

    return False


def is_deprecated(g, elem):
    """
    Checks if an RDF element (URIRef) is marked as obsolete.
    """
    if (elem, OWL.deprecated, Literal(True)) in g:
        return True
    if (elem, RDF.type, OWL.DeprecatedClass) in g:
        return True
    if (elem, RDF.type, OWL.DeprecatedProperty) in g:
        return True
    return False

def remove_prefix(value, prefixes_to_remove = PREFIXES_TO_REMOVE):
    """
    Removes defined prefixes (namespaces) from a URI.

    The first matching namespace from 'prefixes_to_remove' is removed from 'value'.

    :param value: str - complete URI
    :param prefixes_to_remove: List[str] - namespace URIs to remove
    :return: str - URI without namespace prefix
    """
    value = str(value).strip()
    candidates = set()
    candidates.add(value)
    
    for prefix in prefixes_to_remove:
        if value.startswith(prefix):
            val = value[len(prefix):].strip()
            candidates.add(val)
    return sorted(candidates, key=lambda x: (len(x)))[0]


def find_all_caps_words(value):
    """
    Returns all substrings containing two or more consecutive uppercase letters.

    :param value: str - Input text
    :return: List[str] - List of found uppercase substrings
    """
    matches = []
    for m in re.finditer(r'[A-Z]{2,}', value):
        end = m.end()
        # Prüfe: Kommt direkt nach dem Match ein Kleinbuchstabe?
        if end < len(value) and value[end].islower():
            # Letzten Buchstaben entfernen
            matches.append(m.group(0)[:-1])
        else:
            matches.append(m.group(0))
    return matches


def split_around_keywords(value, keywords):
    """
    Splits a text at each occurrence of a keyword.

    The text is split before and after each found keyword. Keywords remain as separate entries.
    
    :param value: str - Input text
    :param keywords: List[str] - List of keywords to split
    :return: str or List[str] - Original text (if no matches) or list of split sections
    """

    if not keywords:
        return value
    else:
        # Baue einen Regex-Ausdruck, der alle Schlüsselwörter gruppiert 
        pattern = '|'.join(re.escape(k) for k in keywords)
        parts = re.split(f'({pattern})', value)
        return [part.strip() for part in parts if part]  # leere Teile entfernen


def add_underscore_before_caps(keywords, parts):
    """   
    Converts a list of words into a snakecase string.

    Inserts underscores, removes spaces, and converts everything to lowercase.
    Keywords can be included in the conversion.

    :param parts: List[str] - List of substrings
    :param keywords: List[str] - Optional keywords to influence the conversion
    :return: str - Snakecase string
    """
    if isinstance(parts, str):
        parts = [parts]  # String zu Liste machen

    result = []

    for part in parts:
        if part in keywords:
            part = part.lower()
            result.append(part)
        else:
            # part = part.strip()
            part = part.replace(" ", "_")
            # Dann '_' vor Großbuchstaben einfügen, falls nicht am Anfang oder bereits ein '_' davor steht
            part = re.sub(r'(?<!^)(?<!_)([A-Z])', r'_\1', part)  
            part = part.lower()  # Alles in Kleinbuchstaben umwandeln
            result.append(part)
    return '_'.join(result)


def clean_value(value):
    """
    Converts any string to Snakecase.

    Spaces are replaced, uppercase letters are separated, and everything is converted to lowercase.

    :param value: str - Input string (e.g., URI, Label, etc.)
    :return: str - Converted Snakecase string
    """
    value = remove_prefix(value)
    keywords = find_all_caps_words(value)  # z. B. ['ID', 'HTML']
    parts = split_around_keywords(value, keywords)
    clean_value = add_underscore_before_caps(keywords, parts)
    clean_value = re.sub(r"[^\w]", "", clean_value) # TODO Was macht das hier??

    return clean_value


## Alle Klassen, Properties und Individuen

In [961]:
def get_all_classes(graph):#, external_namespaces = EXTERNAL_PREFIXES):
    """
    Search the RDFLib graph for OWL classes that are not system-internal 
    or technical classes (e.g., OWL constructs).

    :param graph: rdflib.Graph - RDFLib graph of the loaded ontology
    :return: Set[rdflib.URIRef] - URIs of all non-technical classes
    """

    g = graph

    # Alle Klassen, die explizit als owl:Class oder rdfs:Class deklariert sind
    explicit_class = set(g.subjects(RDF.type, RDFS.Class)).union(
        g.subjects(RDF.type, OWL.Class)
    )
    # Klassen aus rdfs:domain und rdfs:range
    domain_range_class = set(g.objects(None, RDFS.domain)).union(
        g.objects(None, RDFS.range)
    )
    # Klassen, die als Ziel oder Start von rdfs:subClassOf auftauchen
    super_class = set(g.objects(None, RDFS.subClassOf)).union(
        g.subjects(RDFS.subClassOf, None)
    )
    # Klassen, die als Ziel oder Start von owl.equivalentClass auftauchen
    equivalent_class = set(g.objects(None, OWL.equivalentClass)).union(
        g.subjects(OWL.equivalentClass, None)
    )
    # Klassen die Ziel oder Start von owl:complementOf auftauchen
    complement_class = set(g.subjects(OWL.complementOf, None)).union(
        g.objects(None, OWL.complementOf)
    ) 
    # Klassen aus owl:someValuesFrom, owl:allValuesFrom, owl:onClass
    restriction_targets = set(g.objects(None, OWL.someValuesFrom)).union(
        g.objects(None, OWL.allValuesFrom),
        g.objects(None, OWL.onClass)
    )

    # Klassen die min einmal als rdf:type auftauchen
    type_class = set(g.objects(None, RDF.type))

    # Klassen mit min einer Instanz
    cls_with_instances = {}
    for _, _, cls in g.triples((None, RDF.type, None)):
        if isinstance(cls, URIRef):
            cls_with_instances[cls] = cls_with_instances.get(cls, 0) + 1
    cls_with_inst = set(cls_with_instances.keys())

    # Klassen die innerhalb einer Liste stehen
    in_bnods = set(g.objects(None, RDF.first))
    cls_in_bnode = set()
    for cls in in_bnods:
        if cls not in g.subjects(RDF.type, None) and isinstance(cls, URIRef):
            cls_in_bnode.add(cls)   

    # Klassen als Start oder Ziel von owl:disjointWith und owl:disjointUnionOf
    disjoint_class = set(g.subjects(OWL.disjointWith, None)).union(
        g.objects(None, OWL.disjointWith),
        g.subjects(OWL.disjointUnionOf, None),
        g.objects(None, OWL.disjointUnionOf)
    )

    all_class = explicit_class.union(
        domain_range_class,
        super_class,
        equivalent_class,
        restriction_targets,
        complement_class,
        type_class,
        cls_with_inst,
        cls_in_bnode,
        disjoint_class
    )
    g.subject_objects
    
    # Präfixe aller technischen Klassen
    TECHNICAL_NAMESPACES = (
        "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
        "http://www.w3.org/2000/01/rdf-schema#",
        "http://www.w3.org/2002/07/owl#",
        "http://www.w3.org/2001/XMLSchema#",
        "http://www.w3.org/XML/1998/namespace",
        "http://www.w3.org/2003/06/sw-vocab-status/ns#",
        "http://purl.org/vocab/vann/",
        "http://protege.stanford.edu/plugins/owl/protege#",
        "http://www.owl-ontologies.com/2005/08/07/xsp.owl#",
        "http://www.w3.org/ns/shacl#",
        "http://www.w3.org/ns/rdfa#",
        "http://www.w3.org/ns/formats/",
        "http://www.w3.org/ns/prov#",
        "http://www.w3.org/ns/sparql-service-description#",
        "http://spinrdf.org/spin#",
        "http://www.w3.org/2003/11/swrl#",
        "http://www.w3.org/2003/11/swrlb#"
    )
    
    # remove all classes with external prefixes, blind nodes and costum datatypes
    all_classes = set(cls for cls in all_class 
                      if not str(cls).startswith(tuple(TECHNICAL_NAMESPACES)) # technische Klassen
                      and not isinstance(cls, BNode) # Blind Nodes
                      and cls not in set(g.subjects(RDF.type, RDFS.Datatype)) # Costum Datatypes
                    )
    if OWL.Nothing in all_class:
        all_classes.add(OWL.Nothing)
    
    return all_classes

In [962]:
def get_all_properties(graph): #, external_namespaces = EXTERNAL_PREFIXES):
    """  
    Collects all non-technical properties of the ontology.

    Filters all usable object and data type properties that are not OWL 
    or RDF-internal helper constructs.

    :param graph: rdflib.Graph - RDFLib graph of the loaded ontology
    :return: Set[rdflib.URIRef] - URIs of all non-technical properties
    
    """

    g = graph

    # Alle Properties die in Triplen genutzt werden
    used_prop = {}
    for _, p, _ in g:
        if isinstance(p, URIRef):
            used_prop[p] = used_prop.get(p, 0) + 1
    
    used_props = set(used_prop.keys())
    
    # alle Properties die Typisiert sind.
    explicit_props = set(g.subjects(RDF.type, OWL.ObjectProperty)).union(
        g.subjects(RDF.type, OWL.DatatypeProperty),
        # g.subjects(RDF.type, OWL.AnnotationProperty),
        g.subjects(RDF.type, RDF.Property)
    )

    # alle Properties die eine Domain und/oder Range haben.
    domain_range_props = set(g.subjects(RDFS.domain, None)).union(
        g.subjects(RDFS.range, None)
    )

    # alle Properties die in einem Objekt stehen
    object_properties = set(g.objects(None, RDFS.subPropertyOf)).union(
        g.objects(None, OWL.equivalentProperty),
        g.objects(None, OWL.inverseOf),
        g.objects(None, OWL.propertyDisjointWith)
    )

    # alle Properties die in einem Subjekten stehen
    subject_properties = set(g.subjects(None, RDFS.subPropertyOf)).union(
        g.subjects(None, OWL.equivalentProperty),
        g.subjects(None, OWL.inverseOf),
        g.subjects(None, OWL.propertyDisjointWith)
    )

    # alle Properties die nur als Prädikat auftauchen, d.h. keine eigenen Triple haben
    # können durch dereferenziert passieren und wären eigentlich Annotation Properties 
    # werden aber nicht explizit als solche in g gekennzeichnet
    only_as_predicat = used_props.difference(
        explicit_props, 
        domain_range_props, 
        object_properties, 
        subject_properties
    )

         
    all_props = used_props.union(
        explicit_props, 
        domain_range_props, 
        object_properties, 
        subject_properties
    )

    # Präfixe von technischen Properties
    TECHNICAL_NAMESPACES = (
        "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
        "http://www.w3.org/2000/01/rdf-schema#",
        "http://www.w3.org/2002/07/owl#",
        "http://www.w3.org/2001/XMLSchema#",
        "http://www.w3.org/XML/1998/namespace",
        "http://www.w3.org/2003/06/sw-vocab-status/ns#",
        "http://purl.org/vocab/vann/",
        "http://protege.stanford.edu/plugins/owl/protege#",
        "http://www.owl-ontologies.com/2005/08/07/xsp.owl#",
        "http://www.w3.org/ns/shacl#",
        "http://www.w3.org/ns/rdfa#",
        "http://www.w3.org/ns/formats/",
        "http://www.w3.org/ns/prov#",
        "http://www.w3.org/ns/sparql-service-description#",
        "http://spinrdf.org/spin#",
        "http://www.w3.org/2003/11/swrl#",
        "http://www.w3.org/2003/11/swrlb#"
    )

    # remove all properties with external prefixes, blind nodes and annotation properties
    all_properties = set(
        prop for prop in all_props 
        if not str(prop).startswith(tuple(TECHNICAL_NAMESPACES)) # technische Properties
        and not isinstance(prop, BNode) # Blind Nodes
        and prop not in set(g.subjects(RDF.type, OWL.AnnotationProperty)) # AnnotationProperties
        and prop not in only_as_predicat
        # and prop not in unused_properties
    )

    unmarked_annotation_properties = set(
        prop for prop in only_as_predicat
        if not str(prop).startswith(tuple(TECHNICAL_NAMESPACES))
        and prop not in set(g.subjects(RDF.type, OWL.AnnotationProperty))
    )
    
    all_annotation_properties = set(g.subjects(RDF.type, OWL.AnnotationProperty)).union(
        unmarked_annotation_properties,
        # anno_property
    )

    return all_properties, all_annotation_properties

In [963]:
def get_all_individuals(graph, all_classes, all_properties):
    """
    Collects all non-technical individuals of the ontology.
    
    :param graph: rdflib.Graph - RDFLib graph of the loaded ontology
    :param all_classes: Set[rdflib.URIRef] - URIs of all non-technical classes
    :param all_properties: Set[rdflib.URIRef] - URIs of all non-technical properties
    :return: Set[rdflib.URIRef] - URIs of all non-technical individuals
    """
    g = graph
    all_indis = set()
    
    # Sammelt alle Subjekte die als Typ entweder eine Klasse aus all_classes oder owl:NamedIndividual haben  
    for s, o in g.subject_objects(RDF.type):
        if not isinstance(s, URIRef):
            continue
        if o in all_classes or o == OWL.NamedIndividual:
            all_indis.add(s)

    for prop in all_properties:
        for s, o in g.subject_objects(prop):
            if isinstance(s, URIRef):
                all_indis.add(s)
            if isinstance(o, URIRef):
                all_indis.add(o)
       
    # Präfixe aller technischen Klassen
    TECHNICAL_NAMESPACES = (
        "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
        "http://www.w3.org/2000/01/rdf-schema#",
        "http://www.w3.org/2002/07/owl#",
        "http://www.w3.org/2001/XMLSchema#",
        "http://www.w3.org/XML/1998/namespace",
        "http://www.w3.org/2003/06/sw-vocab-status/ns#",
        "http://purl.org/vocab/vann/",
        "http://protege.stanford.edu/plugins/owl/protege#",
        "http://www.owl-ontologies.com/2005/08/07/xsp.owl#",
        "http://www.w3.org/ns/shacl#",
        "http://www.w3.org/ns/rdfa#",
        "http://www.w3.org/ns/formats/",
        "http://www.w3.org/ns/prov#",
        "http://www.w3.org/ns/sparql-service-description#",
        "http://spinrdf.org/spin#",
        "http://www.w3.org/2003/11/swrl#",
        "http://www.w3.org/2003/11/swrlb#"
    )

   
    # remove all individuals with technical prefixes
    all_individuals = set(ind for ind in all_indis 
                        if not str(ind).startswith(TECHNICAL_NAMESPACES) # technische Properties
                        )
    
    return all_individuals


In [964]:
def get_elements_by_group(graph, all_classes, all_properties, all_individuals):
    """    
    Collects all classes, properties, and individuals in a dictionary

    :param all_classes: Set[rdflib.URIRef] - URIs of all non-technical classes
    :param all_properties: Set[rdflib.URIRef] - URIs of all non-technical properties
    :param all_classes: Set[rdflib.URIRef] - URIs of all non-technical individuals
    :return: dict with the following keys:
        {
            "class": List[rdflib.URIRef],
            "classKey": List[rdflib.URIRef],
            "property": List[rdflib.URIRef],
            "propertyKey": List[rdflib.URIRef],
            "individual": List[rdflib.URIRef]
        }
    """
    g = graph
    
    elements_by_group = {
        "class": list(all_classes),
        "classKey": list(all_classes), 
        "property": list(all_properties),
        "propertyKey": list(all_properties),
        "individual": list(all_individuals)
    }

    return elements_by_group


## 3. Restrictionspaket

### Properties

Bestimmt für Properties alle Restrictionskonstrukte in dennen sie genutzt werden, die Klasse die daraus enstehen und welchen Wert die Property hat

In [965]:
def extract_property_restrictions(graph):
    """
    Extrahiert alle owl:hasValue-Verwendungen aus owl:Restriction-Knoten.
    Gibt für jede Property alle Klassen zurück, in denen sie per hasValue verwendet wurde,
    zusammen mit dem jeweiligen Wert.

    :return: Dictionary {Property: [{"class": ..., "predicate", ...", value": ...}, ...]}
    """

    g = graph

    result = {}

    predicates = [
        OWL.hasValue, 
        OWL.someValuesFrom, 
        OWL.allValuesFrom,
        OWL.cardinality, 
        OWL.minCardinality, 
        OWL.maxCardinality,
        OWL.qualifiedCardinality, 
        OWL.minQualifiedCardinality, 
        OWL.maxQualifiedCardinality,
        OWL.hasSelf
    ]

    restriction_location =  set(g.triples((None, OWL.equivalentClass, None))).union(
        g.triples((None, RDFS.subClassOf, None))
    )

    for cls, location, equivalent_class in restriction_location:
        # Fall 1: Direkte Restriction
        if (equivalent_class, RDF.type, OWL.Restriction) in g:
            prop = g.value(equivalent_class, OWL.onProperty)
            for pred in predicates:
                value = g.value(equivalent_class, pred)
                if prop and value:
                    entry = {
                        "definedClass": cls,
                        "predicate": pred,
                        "value": value,
                        "restrictionLocation": location, 
                        "logicalContainer": None
                    }
                    if pred in (OWL.qualifiedCardinality, OWL.minQualifiedCardinality, OWL.maxQualifiedCardinality):
                        qual_class = g.value(equivalent_class, OWL.onClass)
                        if qual_class:
                            entry["onClass"] = qual_class
                    result.setdefault(prop, []).append(entry)

        # Fall 2: Komplexe Klassen mit owl:intersectionOf, unionOf etc.
        for list_pred in [OWL.intersectionOf, OWL.unionOf, OWL.complementOf]:
            if (equivalent_class, list_pred, None) in g:
                collection = g.value(equivalent_class, list_pred)
                while collection:
                    node = g.value(collection, RDF.first)
                    collection = g.value(collection, RDF.rest)

                    if (node, RDF.type, OWL.Restriction) in g:
                        prop = g.value(node, OWL.onProperty)
                        for pred in predicates:
                            value = g.value(node, pred)
                            if prop and value:
                                entry = {
                                    "definedClass": cls,
                                    "predicate": pred,
                                    "value": value,
                                    "restrictionLocation": location, 
                                    "logicalContainer": list_pred
                                }
                                if pred in (OWL.qualifiedCardinality, OWL.minQualifiedCardinality, OWL.maxQualifiedCardinality):
                                    qual_class = g.value(node, OWL.onClass)
                                    if qual_class:
                                        entry["onClass"] = qual_class
                                result.setdefault(prop, []).append(entry)

    return result

### Klassen

Findet alle Class Axiome zu einer angegebenen Klasse und sammelt alle Infos zu den genutzten Restrictions. \
Außerdem unterschiedet er ob es sich beim Objekt der subClassOf und equivalentClass Beziehung um eine echte Klasse oder ein BNode/Class Expression handelt. 

In [966]:
# def extract_class_axioms(g, cls, relation):
#     """
#     Verallgemeinerte Extraktion von OWL-Klassenaussagen, z. B. subClassOf oder equivalentClass.
#     Liefert sowohl explizite Klassen als auch OWL-Restriktionen (direkt und in RDF-Listen).
    
#     :param g: rdflib.Graph der geladenen Ontologie
#     :param cls: URIRef der Klasse
#     :param relation: RDFS.subClassOf oder OWL.equivalentClass
#     :return: [{ "type": "unionOf", "restrictions": [...] }, ...]
#     """
#     explicit_classes = set()
#     restrictions = []

#     if relation == OWL.disjointUnionOf:
#         parts = list(Collection(g, g.value(cls, OWL.disjointUnionOf)))
#         all_uri = all(isinstance(p, URIRef) for p in parts)
#         is_class = []
#         if all_uri:
#             explicit_classes = parts
#         else:
#             for target in parts:
#                 if isinstance(target, BNode):
#                     parse_complex_class_construct(g, target, restrictions)
#                 if isinstance(target, URIRef):
#                     is_class.append(target)
#             restrictions.append({"disjointUnionOfClasses": is_class})
    
#     else:
#         for target in g.objects(cls, relation):  
#             if isinstance(target, URIRef):
#                 explicit_classes.add(target)           
#             elif isinstance(target, BNode):
#                 parse_complex_class_construct(g, target, restrictions)

#     return explicit_classes, restrictions


# def parse_restriction(g, node):
#     """
#     Extrahiert OWL-Restriction-Details aus einem BNode.
#     """
#     restriction = {
#         "owl:onProperty": g.value(node, OWL.onProperty),
#         "owl:onClass": g.value(node, OWL.onClass),
#         "owl:hasValue": g.value(node, OWL.hasValue),
#         "owl:someValuesFrom": g.value(node, OWL.someValuesFrom),
#         "owl:allValuesFrom": g.value(node, OWL.allValuesFrom),
#         "owl:cardinality": g.value(node, OWL.cardinality),
#         "owl:minCardinality": g.value(node, OWL.minCardinality),
#         "owl:maxCardinality": g.value(node, OWL.maxCardinality),
#         "owl:qualifiedCardinality": g.value(node, OWL.qualifiedCardinality),
#         "owl:minQualifiedCardinality": g.value(node, OWL.minQualifiedCardinality),
#         "owl:maxQualifiedCardinality": g.value(node, OWL.maxQualifiedCardinality),
#     }
#     return {k: v for k, v in restriction.items() if v}


# def parse_complex_class_construct(g, target, restrictions):
#     """_summary_

#     :param g: rdflib.Graph der geladenen Ontologie
#     :type g: _type_
#     :param target: _description_
#     :type target: _type_
#     :param restrictions: _description_
#     :type restrictions: _type_
#     :return: _description_
#     :rtype: _type_
#     """

#     # Fall 1: direkte Restriction
#     if (target, RDF.type, OWL.Restriction) in g:
#         restrictions.append({
#             "type": "singleRestriction",
#             "restrictions": [parse_restriction(g, target)]
#         })

#     # Fall 2: unionOf / intersectionOf
#     for list_pred in [OWL.unionOf, OWL.intersectionOf, OWL.complementOf]:
#         if (target, list_pred, None) in g:
#             collection = g.value(target, list_pred)
#             if isinstance(collection, BNode):
#                 restriction_list = {
#                     "type": list_pred,  # "unionOf" oder "intersectionOf"
#                     "classes": [],
#                     "restrictions": []
#                 }
#                 for node in Collection(g, collection):
#                     if isinstance(node, URIRef):
#                         restriction_list["classes"].append(node)
#                     elif (node, RDF.type, OWL.Restriction) in g:
#                         restriction_list["restrictions"].append(parse_restriction(g, node))
#                 if restriction_list["classes"] or restriction_list["restrictions"]:
#                     restrictions.append(restriction_list)

#     return restrictions

Sucht für alle subClass- und equivalentClass Restrictions einer Klasse die raus mit Cardinality-Angaben und grupiert diese nach den Properties

In [967]:
# def extract_cardinality_constraints(sub_class_restrictions, equivalent_restrictions):
#     """_summary_

#     :param sub_class_restrictions: _description_
#     :type sub_class_restrictions: _type_
#     :param equivalent_restrictions: _description_
#     :type equivalent_restrictions: _type_
#     :return: _description_
#     :rtype: _type_
#     """
#     cardinality_list = []
#     result = {}

#     axiom_blocks = (sub_class_restrictions or []) + (equivalent_restrictions or [])

#     for block in axiom_blocks:
#         for r in block.get("restrictions", []):
#             cardinality_keys = [
#                 "owl:cardinality", "owl:minCardinality", "owl:maxCardinality",
#                 "owl:qualifiedCardinality", "owl:minQualifiedCardinality", "owl:maxQualifiedCardinality"
#             ]
#             if any(k in r for k in cardinality_keys):
#                 cardinality_list.append(r)


#     for entry in cardinality_list:
#         prop = entry.get("owl:onProperty")
#         if prop:
#             if prop in result:
#                 result[prop].append(entry)
#             else:
#                 result[prop] = [entry]
#     return result

Clustert wo und wie eine Klasse in ClassExpressions verwendet wird.

In [968]:
# def collect_class_usages(g):
#     """
#     Sammelt, wo und wie jede Klasse in OWL-Klassenaussagen verwendet wird.

#     :param g: rdflib.Graph der geladenen Ontologie
#     :param all_classes: Menge aller Klassen-URIs
#     :return: Dict[class_uri] = [Verwendungsinformationen]
#     """
    
#     usage_map = {}

#     for cls in get_all_classes(g):
#         for rel in [RDFS.subClassOf, OWL.equivalentClass, OWL.complementOf, OWL.disjointUnionOf]:
#             explicit, restrictions = extract_class_axioms(g, cls, rel)

#             # explizite Zielklassen (z. B. ex:A subClassOf ex:B)
#             for target in explicit:
#                 if target not in usage_map:
#                     usage_map[target] = []
#                 usage_map[target].append({
#                     "usedIn": cls,
#                     "relation": rel,
#                     "context": "explicit"
#                 })

#             for block in restrictions:
#                 # Klassen aus unionOf / intersectionOf
#                 for c in block.get("classes", []):
#                     if isinstance(c, URIRef):
#                         if c not in usage_map:
#                             usage_map[c] = []
#                         usage_map[c].append({
#                             "usedIn": cls,
#                             "relation": rel,
#                             "context": block["type"]
#                         })

#                 # Klassen aus Restriktionen (someValuesFrom, allValuesFrom, onClass)
#                 for r in block.get("restrictions", []):
#                     for key in ["owl:someValuesFrom", "owl:allValuesFrom", "owl:onClass"]:
#                         if key in r:
#                             target_class = r[key]
#                             if isinstance(target_class, URIRef):
#                                 if target_class not in usage_map:
#                                     usage_map[target_class] = []
#                                 usage_map[target_class].append({
#                                     "usedIn": cls,
#                                     "relation": rel,
#                                     "context": key
#                                 })
                
#                 # Klassen aus owl:disjointUnionOf
#                 for d in block.get("disjointUnionOfClasses", []):
#                     if isinstance(d, URIRef):
#                         if d not in usage_map:
#                             usage_map[d] = []
#                         usage_map[d].append({
#                             "usedIn": cls,
#                             "relation": rel
#                         })


#                     # Absicherung: Falls mal URIRefs direkt in "restrictions" auftauchen
#                     if isinstance(r, URIRef):
#                         if r not in usage_map:
#                             usage_map[r] = []
#                         usage_map[r].append({
#                             "usedIn": cls,
#                             "relation": rel,
#                             "context": "directRestrictionClass"
#                         })

#     return usage_map

## 1. Basispaket

### R1 - Klassenabbildung


siehe <b> Tabellen Infos zusammentragen </b>

### R10 - Individuen-Initialisierung

siehe <b> SQL-Generierung - Einträge </b>

### R4 - SubClassOf (einfach)


In [969]:
def build_sub_elemenet_map(graph, sub_element_type, equivalent_group, all_elements):
    """
    Builds a hierarchical mapping of sub- and super-elements within the ontology.

    This function detects and structures hierarchical relationships such as 
    subclass or subproperty connections between ontology elements. It also 
    handles equivalent elements to maintain transitive and symmetric links 
    across the hierarchy.

    :param graph (rdflib.Graph): RDF graph containing ontology triples.
    :param sub_element_type (rdflib.term.URIRef): Predicate used to identify hierarchical relationships 
                                                  (e.g., rdfs:subClassOf or rdfs:subPropertyOf).
    :param equivalent_group (dict): Mapping of equivalent elements.
    :param all_elements (set): Set of all relevant ontology elements (classes or properties) to consider.
    :return (tuple[dict, dict, dict]): 
        - **super_element_of_dist (dict)**: Distances from each element to its super-elements.
        - **sub_element_of_dist (dict)**: Distances from each element to its sub-elements.
        - **new_equivalent_group (dict)**: Newly discovered equivalences among elements based on mutual hierarchy.

    """
    g = graph

    sub_element_of = defaultdict(set)
    super_element_of = defaultdict(set)
    super_equivalents = defaultdict(set)

    
    for s, o in g.subject_objects(sub_element_type):
        if s not in all_elements or o not in all_elements:
            continue
        if not isinstance(s, URIRef) or not isinstance(o, URIRef):
            continue
            
        if any(is_deprecated(g, x) for x in (s,o)):
            continue
       
        # avoids semanticly wrong edges between a normal property and a annotation property
        if any((x, RDF.type, OWL.AnnotationProperty) in g for x in (s,o)):
            continue

        if s != o:
            sub_element_of[s].add(o)
            sub_element_of[s] |= equivalent_group.get(o, set())
            super_element_of[o].add(s)
            super_element_of[o] |= equivalent_group.get(s, set())      

    
    new_equivalent_group = defaultdict(set)
    super_equivalents = dict(super_element_of)

    for a, supers in super_element_of.items():
        for b in supers:
            if a in super_element_of.get(b, set()) and a != b:
                new_equivalent_group[a].add(b)
                new_equivalent_group[b].add(a)
    
    
    super_element_of_dist = {}
    sub_element_of_dist = {}
    
    for elem in all_elements:
        if not sub_element_of.get(elem) and not super_element_of.get(elem):
            continue
        
        def dist_from_elem_to_sub_super_elem(elem, sub_element_of_map):
            dist = {elem: 0}
            quere = deque([elem])
            while quere:
                u = quere.popleft()
                for p in sub_element_of_map.get(u, ()):
                    if p not in dist:
                        dist[p] = dist[u] + 1
                        quere.append(p)
            dist.pop(elem)
            return dist
        
        super_elem_dist = dist_from_elem_to_sub_super_elem(elem, super_element_of)
        if super_elem_dist:
            super_element_of_dist[elem] = super_elem_dist
        
        sub_elem_dist = dist_from_elem_to_sub_super_elem(elem, sub_element_of)
        if sub_elem_dist:
            sub_element_of_dist[elem] = sub_elem_dist
        
    return super_element_of_dist, sub_element_of_dist, dict(new_equivalent_group)

In [970]:
def build_sub_class_map(graph, equivalent_class_group, all_classes):
    """
    Constructs the subclass hierarchy map for all ontology classes.

    It determines the hierarchical structure of all classes.

    :param graph (rdflib.Graph): RDF graph containing ontology triples.
    :param equivalent_class_group (dict): Mapping of equivalent classes.
    :param all_classes (set): Set of all ontology classes to consider in the hierarchy.
    :return (tuple[dict, dict, dict]): 
        Contains the super-class distances, sub-class distances, and newly inferred 
        class equivalences.
    """
    return build_sub_elemenet_map(graph, RDFS.subClassOf, equivalent_class_group, all_classes)

### R6/R7 - Functional Properties


In [971]:
def get_functional_properties(graph, all_properties):
    """  
    Bestimmt, ob eine Property Single-Valued ist und auf welche Weise.

    Eine Property gilt als Single-Valued / Funktional, wenn:
    - sie empirisch pro Subjekt nur einen Objekt besitzt,
    - sie als owl:FunctionalProperty deklariert ist,
    - sie als einzige Property in einem owl:hasKey-Objekt steht,
    - ihre inverse als owl:InverseFunctionalProperty deklariert ist
    - ihre equivalente als owl:FunctionalProperty deklariert ist
    - ihre super Property als owl:FunctionalProperty deklariert ist 
    
    :param graph: rdflib.Graph - RDFLib-Graph der geladenen Ontologie
    :param prop: rdflib.URIRef - URI der zu prüfenden Property
    :return: Dict[List[str]] - Set der zutreffenden Gründe für Single-Valued-Status
    
    
    Determines which properties are single-valued (functional) and for what reason.

    The function analyses each property in the ontology and identifies different 
    reasons why it may be functional, either declared explicitly via OWL semantics 
    or inferred empirically from the data.

    :param graph (rdflib.Graph): RDF graph representing the ontology.
    :param all_properties (set): All property URIs in the ontology.
    :return (tuple[dict, dict]): 
        - **functional_types (dict)**: For each property, the detected functional types 
          (e.g., 'functional', 'empirically functional', 'key functional').
        - **used_as_functional (dict)**: Boolean flags indicating whether each property 
          is effectively treated as single-valued.
    """

    g = graph
    functional_types = {}
    used_as_functional = {}
    
    for prop in all_properties:
        # alle Subjekte von Triplen mit der Property als Prädikat
        subjects_with_prop = set(g.subjects(prop, None))
        prop_functional_types = set()
        prop_used_as_functional = False
        # prop_single

        # owl:hasKey prüfen
        for s, o in g.subject_objects(OWL.hasKey):
            keys = list(Collection(g, o))
            if len(keys) == 1 and keys[0] == prop:
                prop_functional_types.add("key functional")
                break  # reicht, wenn einmal gefunden
        
        # if the inverse of a property is inverse functional than the property is functional
        for s, o in g.subject_objects(OWL.inverseOf):
            if o == prop and (s, RDF.type, OWL.InverseFunctionalProperty) in g:
                prop_functional_types.add("functional")
            elif s == prop and (o, RDF.type, OWL.InverseFunctionalProperty) in g:
                prop_functional_types.add("functional")  
        
        # if a equivalent has type owl:FunctionalProperty than the property is functional
        for s, o in g.subject_objects(OWL.equivalentProperty):
            if s == prop and (o, RDF.type, OWL.FunctionalProperty) in g:
                prop_functional_types.add("functional")
            if o == prop and (s, RDF.type, OWL.FunctionalProperty) in g:
                prop_functional_types.add("functional")
        
        # if a super Property has type owl:FunctionalProperty than the property is functional
        for o in g.objects(prop, RDFS.subPropertyOf):
            if (o, RDF.type, OWL.FunctionalProperty) in g:
                prop_functional_types.add("functional")
        
        # owl:FunctionalProperty prüfen
        if (prop, RDF.type, OWL.FunctionalProperty) in g:
            prop_functional_types.add("functional")

        # no triples given but functional in any way
        if len(subjects_with_prop) == 0 and prop_functional_types:
            prop_used_as_functional = True
        
        # prüfen ob die Property je Subjekt nur max. 1 Objekt hat.
        elif subjects_with_prop or "functional" in prop_functional_types:
            is_empirically_functional = True
            for s in subjects_with_prop:
                values = list(g.objects(s, prop))
                if len(values) > 1:
                    is_empirically_functional = False
                    break
            if is_empirically_functional:
                prop_functional_types.add("empirically functional")
                prop_used_as_functional = True
        
        if prop_functional_types:
            functional_types[prop] = prop_functional_types
            
        used_as_functional[prop] = prop_used_as_functional

    return functional_types, used_as_functional

### R11 - Annotations

fasst alle Annotations einer Klasse/Property/Individuum zusammen

In [972]:
def get_element_annotations(graph, element, all_annotation_properties):
    """          
    Collects all annotation values for a given ontology element.

    The function extracts all annotations attached to a specific class, 
    property, or individual, including standard ones such as rdfs:label 
    and rdfs:comment, as well as custom user-defined annotation properties. 

    :param graph (rdflib.Graph): RDF graph containing the ontology.
    :param element (rdflib.URIRef): URI of the ontology element (class, property, or individual).
    :param all_annotation_properties (set): Set of known annotation property URIs.
    :return (dict): A dictionary containing all annotations for the element, 
                    where keys are annotation properties and values are sets 
                    or nested dictionaries of literal values. Used for generating 
                    SQL comments and metadata tables.
    """
    g = graph

    annotations = defaultdict(set)

    for _, p, o in g.triples((element, None, None)):
        if isinstance(o, URIRef):
            o_new = o
        else:
            o_new = remove_prefix(o).replace("'", "''") # ' wird mit '' ersetzt, sodass am Ende im String ' erhalten bleibt.

        if p == RDFS.label: # AnnotationProperty
            annotations[RDFS.label].add(o_new)

        elif p == RDFS.comment: # AnnotationProperty
            annotations[RDFS.comment].add(o_new)

        elif p == RDFS.seeAlso: # AnnotationProperty
            annotations[RDFS.seeAlso].add(o_new)

        elif p == RDFS.isDefinedBy: # AnnotationProperty
            annotations[RDFS.isDefinedBy].add(o_new)

        elif p == OWL.sameAs: # for individuals only
            annotations[OWL.sameAs].add(o_new)

        # elif p == OWL.versionInfo: # no semantic meaning
        #     annotations[OWL.versionInfo].add(str(o))  # stringified

        # elif p == OWL.priorVersion: # no semantic meaning
        #     annotations[OWL.priorVersion].add(o_new)

        # elif p == OWL.backwardCompatibleWith: # no semantic meaning
        #     annotations[OWL.backwardCompatibleWith].add(o_new)

        # elif p == OWL.incompatibleWith: # no semantic meaning
        #     annotations[OWL.incompatibleWith].add(o_new)

        elif p == OWL.deprecated:
            annotations[OWL.deprecated].add(str(o))  # stringified

        elif p == OWL.AllDifferent: # for individuals only
            annotations[OWL.AllDifferent].add(o_new)

        elif p == OWL.differentFrom: # for individuals only
            annotations[OWL.differentFrom].add(o_new)
        
        else:
            for anno_prop in all_annotation_properties:
                if p == anno_prop:
                    annotations[anno_prop].add(o_new)

    return dict(annotations)

In [973]:
def get_annotation_map(graph, context_dict):
    """
    Aggregates all annotation data across ontology classes, properties, and individuals.

    This function applies 'get_element_annotations' to every ontology element type 
    and merges the results into a single dictionary.

    :param graph (rdflib.Graph): RDF graph representing the ontology.
    :param context_dict (dict): Context dictionary including the all classes, properties, 
                                individuals, and annotation properties.
    :return (dict): Nested dictionary containing annotation mappings for all ontology elements,
                    structured by element type ("class", "property", "individual").
                    Example:
                    {
                        "class": { class_uri: {...} },
                        "property": { property_uri: {...} },
                        "individual": { individual_uri: {...} }
                    }
    """
    g = graph
    
    all_classes = context_dict["all_classes"]
    all_properties = context_dict["all_properties"]
    all_individuals = context_dict["all_individuals"]
    all_annotation_properties = context_dict["all_annotation_properties"]
    
    class_annotations = {}
    property_annotations = {}
    individual_annotations = {}
    
    all_annotations = {}
    
    def get_all_element_annotations(all_elements, element_annotations):
        for elem in all_elements:
            element_annotations[elem] = get_element_annotations(g, elem, all_annotation_properties)
        return element_annotations
    
    all_annotations["class"] = get_all_element_annotations(all_classes, class_annotations)
    all_annotations["property"] = get_all_element_annotations(all_properties, property_annotations)
    all_annotations["individual"] = get_all_element_annotations(all_individuals, individual_annotations)
    
    return all_annotations

### R9 - Datentyp-Inferenz

Bestimmt für eine Property Datentypen anhand bestimmter Keywords in der URI, in Labeln oder in Kommentaren. Und zählt wie oft diese jeweils ausgewehlt werden.

In [974]:
def infer_datatypes_by_keywords(graph, prop, context_dict):
    """
    Infers possible XSD datatypes for a property based on keyword matching.

    Both the URI (without prefix) and textual annotations (labels, comments, etc.)
    are scanned for predefined keywords (e.g., "date", "price", "flag") that suggest
    an appropriate XSD datatype. The result is a frequency counter indicating
    how often each datatype was matched. This heuristic supports datatype inference
    in the Ontology Shredder when no explicit range or literal type is given.

    :param graph (rdflib.Graph): RDF graph containing the ontology.
    :param prop (rdflib.URIRef): URI of the property being analyzed.
    :param context_dict (dict): Context dictionary including the annotation map.
    :return (collections.Counter): Counts of detected XSD datatypes, e.g.
                                   Counter({XSD.string: 3, XSD.integer: 1}).
    """
    g = graph
    
    keyword_counts = Counter()
    labels_comments = []
    
    prop_annotations = context_dict["annotation_map"]["property"][prop]

    # for annotations in prop_annotations:
    for keys, annotation in prop_annotations.items(): 
        if annotation != None:
            if keys != "customAnnotations":
                labels_comments.extend(annotation)
            if keys == "customAnnotations":
                for custom_anno, value in annotation.items():
                    labels_comments.extend(value)
                        

    p_clean = remove_prefix(prop).lower()
    labels_comments.append(p_clean)

    for keywords in labels_comments:
        keywords = keywords.lower()

        if "date" in keywords:
            keyword_counts[XSD.date] += 1

        if "time" in keywords:
            keyword_counts[XSD.time] += 1

        if "year" in keywords:
            keyword_counts[XSD.gYear] += 1

        if "time" in keywords and "date" in keywords:
            keyword_counts[XSD.dateTime] += 1

        if any(k in keywords for k in ["id", "uri", "url", "link"]):
            keyword_counts[XSD.anyURI] += 1

        if any(k in keywords for k in ["count", "number", "total", "size"]):
            keyword_counts[XSD.integer] += 1

        if any(k in keywords for k in ["flag", "valid", "enabled"]):
            keyword_counts[XSD.boolean] += 1

        if any(k in keywords for k in ["price", "amount", "value", "score"]):
            keyword_counts[XSD.decimal] += 1

        if any(k in keywords for k in ["name", "label", "title", "code", "type", "category", "about"]):
            keyword_counts[XSD.string] += 1
    
    return keyword_counts

Bestimmt den Datatype eines Objektswerts

In [975]:
def infer_datatypes_by_value(object):
    """
    Determines the most likely XSD datatype(s) of a literal value by pattern recognition.

    The function compares the given value against predefined regular expressions
    and semantic patterns (e.g., numeric, date, boolean) to identify matching
    datatypes. Multiple matches may be returned if a value fits several patterns.
    Used by the Ontology Shredder to infer datatypes directly from instance data.

    :param object (str | rdflib.Literal): Literal or string value to analyze.
    :return (set): Set of matching XSD datatype URIs, e.g.
                   {XSD.date, XSD.dateTime}.
    """
    byValue = set()

    # Sprachliteral (z. B. "Beispiel"@de)
    if isinstance(object, Literal) and object.language:
        byValue.add(XSD.language)

    o_clean = remove_prefix(object).strip()

    # Boolean (auch einfache Varianten wie 0/1, yes/no)
    if o_clean.lower() in {"true", "false", "1", "0", "yes", "no", "on", "off"}:
        byValue.add(XSD.boolean)

    # gYear (vierstellige Zahl und startet mit 19 oder 20)
    if re.fullmatch(r"(19|20)\d{2}", o_clean):
        byValue.add(XSD.gYear)

    # gYearMonth (z. B. 2023-05)
    if re.fullmatch(r"(19|20)\d{2}-\d{2}", o_clean):
        byValue.add(XSD.gYearMonth)

    # Integer (ganze Zahl mit optionalem Minus)
    if re.fullmatch(r"-?\d+", o_clean):
        byValue.add(XSD.integer)

    # Decimal (mit Punkt, optional negativ)
    elif re.fullmatch(r"-?\d+\.\d+", o_clean):
        byValue.add(XSD.decimal)

    # Double (Exponentialschreibweise z. B. 1.23e4)
    elif re.fullmatch(r"-?\d+\.\d+[eE][+-]?\d+", o_clean):
        byValue.add(XSD.double)

    # Time (z. B. 12:30 oder 12:30:00)
    if re.fullmatch(r"\d{2}:\d{2}(:\d{2}(\.\d{1,6})?)?", o_clean):
        byValue.add(XSD.time)

    # AnyURI (http/https-URL)
    if re.match(r"^https?://[^\s]+$", object):
        byValue.add(XSD.anyURI)
    
    # Duration (z. B. PT2H30M, P1Y)
    if re.fullmatch(r"P(T)?(\d+[YMDHMS])+", o_clean):
        byValue.add(XSD.duration)

    # Date / DateTime (z. B. 2023-01-01 oder 2023-01-01 12:30:00)
    dt = 0
    try:
        dt = parse(o_clean, fuzzy=False)  
    except Exception:
        dt = None
    if dt is not None:
        has_time = dt.hour != 0 or dt.minute != 0 or dt.second != 0
        if has_time:
            byValue.add(XSD.dateTime)     
        else: 
            byValue.add(XSD.date)

    # default Wert String      
    if not byValue:
        byValue.add(XSD.string) 

  
    return byValue
    


Bestimmt Datatypes der Individuen einer Property. Per Range, per Keywords, per Wert des Literals, per Wert des Objekts

In [976]:
def extract_property_datatypes(graph, prop, class_expressions, context_dict):
    """
    Collects all potential datatypes of a property across multiple inference categories.

    For each property, the function aggregates datatype hints from five sources:
    range declarations, literal type annotations, object values, OWL restrictions,
    and keyword analysis. The results are stored per category together with a
    combined overall count. This information guides SQL type selection in the
    Ontology Shredder's mapping phase.

    :param graph (rdflib.Graph): RDF graph representing the ontology.
    :param prop (rdflib.URIRef): Property for which datatypes are inferred.
    :param class_expressions (dict): Mapping of class expressions (someValuesFrom, allValuesFrom, …).
    :param context_dict (dict): Shared ontology context including equivalence groups.
    :return (dict): Dictionary summarizing all detected datatypes per inference method and combined,
                    e.g.:
                    {
                        prop: {
                            "byRange": CounterDict,
                            "byLiteral": CounterDict,
                            "byValue": CounterDict,
                            "byRestriction": CounterDict,
                            "byKeyword": CounterDict,
                            "combinedCount": CounterDict
                        }
                    }
    """ 

    g = graph
    used_in_class_expression = class_expressions
    equivalent_prop_group = context_dict["equivalent_property_group"].get(prop, [prop])
    # super_classes = context_dict["sub_property_of_dist"].get(prop, set())
    all_classes = context_dict["all_classes"]
    datatypes = {}

    byLiteral = Counter()
    byValue = Counter()
    byRestriction = Counter()
    byRange = Counter()
    
    for p in equivalent_prop_group:
        # Alle Objeket auf die die Property p zeigt
        objects_referred_by_prop = set(o for _, _, o in g.triples((None, p, None)))

        # by rdfs:range
            # bestimmt die Range der Property p
        # range_ = g.objects(p, RDFS.range)
        for r in g.objects(p, RDFS.range):
            if r in all_classes:
                continue
            if isinstance(r, BNode) and not (r, RDF.type, RDFS.Datatype) in g:
                continue
            byRange[r] += 1
            
        # by Keywords
            # überprüft die URI, Labels (rdfs:label/andere Label) der Property p auf festgelegte Keywords
            # besonders mit Domainwissen zur Ontologie kann dies ein guter Weg sein einen Datatype zu bestimmen
        byKeywords = infer_datatypes_by_keywords(g, p, context_dict)  # bleibt ein Set

        # by Restrictions
            # Schaut ob bei someValuesFrom und allValuesFrom ein Datatype angegeben ist.
        if p in used_in_class_expression:
            for class_expression in used_in_class_expression[p]:
                if class_expression["predicate"] in (OWL.someValuesFrom, OWL.allValuesFrom, OWL.hasValue):
                    value = class_expression["value"]
                    if g.value(value, RDF.type) == RDFS.Datatype:
                        byRestriction[value] += 1
                    if str(value).startswith(str(XSD)):
                        byRestriction[value] += 1
                    if isinstance(value, Literal):
                        byRestriction[value.datatype] += 1
        

        for o in objects_referred_by_prop:
            # by Literal Datatype
                # Bestimmt den Typ von Literalen sofern diese auch wirklich einen Typ angehängt haben
            if isinstance(o, Literal): 
                if o.datatype is not None:
                    byLiteral[o.datatype] += 1
                if o.language is not None:
                    byLiteral[XSD.language] += 1

            # by Value
                # Bestimmt je Objekt anhand der tatsächlichen Werte passende Datentypen
            inferred = infer_datatypes_by_value(o)
            if isinstance(inferred, set):
                for dtype in inferred:
                    byValue[dtype] += 1
            elif inferred:  # falls infer_datatypes_by_value nur einen Typ zurückgibt
                byValue[inferred] += 1


        # Add combined count
        combined = Counter()
        for sub_counter in [byRange, byLiteral, byValue, byRestriction, byKeywords]:
            combined.update(sub_counter)

        # Zähle Range, wenn vorhanden
        # for r in range_:
        #     if str(r).startswith(str(XSD)):
        #         combined[r] += 1
        #     if g.value(r, RDF.type) == RDFS.Datatype:
        #         combined[r] += 1


    datatypes[prop] = {
        "byRange": byRange if byRange else None,
        "byLiteral": byLiteral if byLiteral else None,
        "byValue": byValue if byValue else None,
        "byRestriction": byRestriction if byRestriction else None,
        "byKeyword": byKeywords if byKeywords else None,
        "combinedCount": combined if combined else None
    }

    return datatypes


In [977]:
def extract_custom_datatypes(graph):
    """
    Extrahiert benutzerdefinierte rdfs:Datatype-Definitionen mit owl:withRestrictions.

    Dabei werden die Basistypen (z. B. xsd:string) und die Facetten (z. B. maxLength, pattern) extrahiert.

    :param g: rdflib.Graph - RDFLib-Graph der geladenen Ontologie
    :return: Dictionary 
            { Datatype URI1: {
                "base": URI des XSD Datatyps, 
                "restrictions": {facet1: value1, ... }, 
                "maxLen": int | None }, 
              Datatype URI2: {...},
              ...
            }
    
    Extracts custom rdfs:Datatype definitions with restrictions from the ontology.

    The function identifies user-defined datatypes described with
    owl:onDatatype and owl:withRestrictions or owl:oneOf constructs.
    It records their base XSD type and constraint facets (e.g., maxLength, pattern)
    so the Ontology Shredder can later translate them into precise SQL column types.

    :param graph (rdflib.Graph): RDF graph containing the ontology.
    :return (dict): Mapping of each custom datatype URI to its base type and restrictions, e.g.:
                    {
                        ex:AgeType: {
                            "base": XSD.integer,
                            "restrictions": {XSD.minInclusive: 18},
                            "maxLen": None
                        }
                    }
    
    """

    g = graph

    datatypes = {}
    
    for datatype in g.subjects(RDF.type, RDFS.Datatype):
        base = g.value(datatype, OWL.onDatatype)
        restrictions = {}
        if base:
            for rest_node in g.items(g.value(datatype, OWL.withRestrictions)):
                for facet, value in g.predicate_objects(rest_node):
                    if isinstance(value, Literal):
                        restrictions[facet] = value
            
            if restrictions:
                datatypes[datatype] = {
                    "base": base,
                    "restrictions": restrictions
                }

        one_of = list(g.items(g.value(datatype, OWL.oneOf)))

        if one_of:
            inferred_types = set()
            for rest_node in one_of:
                if isinstance(rest_node, Literal):
                    inferred_types.add(rest_node.datatype or XSD.string)
                else:
                    inferred_types.app(XSD.anyURI)
            if len(inferred_types) == 1:
                base = inferred_types.pop()
            elif XSD.anyURI in inferred_types:
                base = XSD.anyURI
            else:
                base = XSD.string 
        
            max_len = max(len(elem) for elem in one_of)  

            datatypes[datatype] = {
                "base": base,
                "restrictions": one_of,
                "maxLen": max_len if base in (XSD.string) else None
            }
    
    return datatypes

Nimmt einen XSD Datatype und übersetzt ihn in einen SQL Datatype.

In [978]:
def map_xsd_to_sql(graph, prop, xsd_type, context_dict):
    """
    Maps an XSD datatype (or custom restricted datatype) to an appropriate SQL datatype.

    The function converts standard and custom XSD datatypes into SQL types.
    For restricted custom datatypes, it also considers facets such as maxLength or numeric precision.
    This ensures datatype fidelity between ontology and relational schema.

    :param graph (rdflib.Graph): RDF graph representing the ontology.
    :param prop (rdflib.URIRef): Property whose datatype is being mapped.
    :param xsd_type (rdflib.URIRef): XSD or custom datatype URI.
    :param context_dict (dict): Context containing all discovered custom datatypes.
    :return (str | dict): SQL datatype string (e.g. "VARCHAR(100)") or
                          detailed mapping { "sql_datatype": ..., "restrictions": ... }.
    """
    g = graph
    
    xsd_to_sql = {
    # aus byValue und byKeywords:
        XSD.string: "VARCHAR",
        XSD.boolean: "BOOLEAN",
        XSD.integer: "INTEGER",
        XSD.decimal: "NUMERIC",
        XSD.double: "DOUBLE PRECISION",
        XSD.date: "DATE",
        XSD.time: "TIME",
        XSD.dateTime: "TIMESTAMP",
        XSD.duration: "INTERVAL",
        XSD.gYear: "NUMERIC(4,0)",
        XSD.gYearMonth: "VARCHAR(7)",
        XSD.anyURI: "VARCHAR",
        XSD.language: "VARCHAR",
        
        # Zusätzliche mögliche Datentypen für byRange und byLiteral:
        XSD.positiveInteger: "INTEGER",
        XSD.positiveInt: "INTEGER",
        XSD.positiveInteger: "INTEGER",
        XSD.nonNegativeInteger: "INTEGER",
        XSD.negativeInteger: "INTEGER",
        XSD.byte: "SMALLINT",
        XSD.short: "SMALLINT",
        XSD.int: "INTEGER",
        XSD.long: "BIGINT",
        XSD.unsignedByte: "SMALLINT",
        XSD.unsignedShort: "SMALLINT",
        XSD.unsignedInt: "INTEGER",
        XSD.unsignedLong: "INTEGER",
        XSD.float: "REAL",
        XSD.token: "VARCHAR",
        XSD.normalizedString: "VARCHAR",
        XSD.hexBinary: "BYTEA",
        XSD.Name: "VARCHAR",
        XSD.NCName: "VARCHAR",
        XSD.NMTOKEN: "VARCHAR",
        XSD.gDay: "VARCHAR(7)",
        XSD.gMonth: "VARCHAR(4)",
        XSD.gMonthDay: "VARCHAR(8)",
        RDFS.Literal: "VARCHAR"
    }

    # Prüft ob es sich um einen XSD Datatype oder ein Literal handelt
    if str(xsd_type).startswith(str(XSD)) or xsd_type == RDFS.Literal:
        sql_type = xsd_to_sql.get(xsd_type, "VARCHAR") # Mapping des XSD.Datatypes auf ein SQL Datatype
    else:
        sql_type = (xsd_type, "rdfs:Datatype")

    # Holt alle custom Datatypes der Ontology
    custom_datytype = context_dict["custom_datatypes"]
    
    # Bestimmt den Type und Restrictions der Custom Datatypes/RDFS Datataypes
    if sql_type[1] == "rdfs:Datatype":
        custom_type = custom_datytype.get(sql_type[0]) # passendes Schlüssel Value Paar aus dem Custom Datatype Dictionary
        if custom_type:
            sql_datatype = xsd_to_sql.get(custom_type["base"], "VARCHAR")
            length = custom_type.get("maxLen")
            sql_type = { 
                "sql_datatype": f"{sql_datatype}({length})" if length else sql_datatype,
                "restrictions": custom_type["restrictions"] 
            }
        else:
            sql_type = "VARCHAR"

    # Bestimmung der maximal Länge für VARCHAR
    if sql_type == "VARCHAR":
        values =  g.objects(None, prop)
        max_len = max((len(value) for value in values), default=0)*1.2
        if max_len > 255:
            sql_type = "VARCHAR"
        elif max_len <= 50:
            sql_type = "VARCHAR(50)"
        elif max_len <= 100:
            sql_type = "VARCHAR(100)"
        else:
            sql_type = "VARCHAR(255)"

    # Bestimmung der maximal Länge für NUMERIC
    if sql_type == "NUMERIC":
        values =  set(g.objects(None, prop))
        scale = set()
        before_decimal = set()
        if values:
            for value in values:
                dec = Decimal(str(value))  # vermeidet float-Rundungsfehler
                sign, digits, exponent = dec.as_tuple()
                digits = list(digits)
                
                scale.add(-exponent if exponent < 0 else 0)
                before_decimal.add(len(digits) + exponent)
            
            s = max(scale)
            p = max(before_decimal) + s
            sql_type = (f"NUMERIC({p}, {s})")
        else:
            sql_type = (f"NUMERIC")

    return sql_type

Bestimmt für eine Property aus allen gefundenen XSD Datatypen einen finalen XSD Datatype.

In [979]:
def recommended_datatype(graph, prop, context_dict):
    """                       
    Selects the most appropriate XSD and SQL datatype for a property using a priority heuristic.

    Datatypes derived from multiple inference sources are compared in order of confidence:
    1 = by Range, 2 = by Literal, 3 = by Value, 4 = by Restriction, 5 = by Keyword.
    The first category that yields a unique or dominant datatype determines the recommendation.

    :param graph (rdflib.Graph): RDF graph representing the ontology.
    :param prop (rdflib.URIRef): Property for which the datatype is chosen.
    :param context_dict (dict): Context containing class expressions and mapping utilities.
    :return (dict): Recommended datatype information:
                    {
                        "XSDdatatype": rdflib.URIRef,
                        "SQLdatatype": str,
                        "source": str | None
                    }
                    If no type can be inferred, defaults to VARCHAR.
    
    """
    g = graph
    usedInClassExpression = extract_property_restrictions(g)

    all_datatypes = extract_property_datatypes(g, prop, usedInClassExpression, context_dict)

    for props, info in all_datatypes.items():
        combined = info["combinedCount"]
        byRange = info["byRange"]
        byLiteral = info["byLiteral"]
        byValue = info["byValue"]
        byRestriction = info["byRestriction"]
        byKeywords = info["byKeyword"]

    
    recommended = {}

    for source_name, type_source in [
        ("byRange", byRange),
        ("byLiteral", byLiteral),
        ("byValue", byValue),
        ("byRestriction", byRestriction),
        # ("byKeyword", byKeywords)
    ]:  
        # Wenn mehr als einen Datatype für eine Kategorie bestimmt wurde
        # if type_source and isinstance(type_source, dict) and len(type_source) >= 2:
        #     top2 = Counter(type_source).most_common(2) # bestimmt die beiden Typen die am häufigsten vorkommen
        #     if top2[0][1] == top2[1][1]: # Vergleicht die Anzahl der beiden häufigsten Datatypes
        #         top_combined = combined and Counter(combined).most_common(1)[0][0]
        #         top2_keys = (top2[0][0], top2[1][0])
        #         value = top_combined if top_combined in top2_keys else top2[0][0]
        #         recommended = {
        #             "XSDdatatype": value,
        #             "SQLdatatype": map_xsd_to_sql(graph, prop, value, context_dict),
        #             "source": source_name
        #         }
        #         break # Wenn ein Datatype gefunden wurde wird die FOR-Schleife beendet
        
        if type_source and isinstance(type_source, dict) and len(type_source) >= 2:
            counts = Counter(type_source)
            top_all = counts.most_common()  # alle Typen nach Häufigkeit sortiert
            max_count = top_all[0][1]       # höchste Häufigkeit
            top_keys = [k for k, v in top_all if v == max_count]  # alle gleich häufigen Typen

            if len(top_keys) > 1:
                top_combined = Counter(combined).most_common(1)[0][0] if combined else None
                value = top_combined if top_combined in top_keys else top_keys[0]
            else:
                value = top_keys[0]

            recommended = {
                "XSDdatatype": value,
                "SQLdatatype": map_xsd_to_sql(graph, prop, value, context_dict),
                "source": source_name
            }

            break
        
        # Wenn nur einen Type für eine Kategorie bestimmt wurde
        elif type_source:
            value = Counter(type_source).most_common(1)[0][0]
            recommended = {
                "XSDdatatype": value,
                "SQLdatatype": map_xsd_to_sql(graph, prop, value, context_dict),
                "source": source_name
            }
            break

    if not recommended:
        recommended = {
            "XSDdatatype": "No datatype found",
            "SQLdatatype": "VARCHAR", 
            "source": None
        }

    return recommended

### R2/R8 - Klassen-Naming/Property-Naming

Hilfstabellen für get_sql_name()

In [980]:
def is_known_word(word):
    """
    Checks if a given string is a valid English word.

    Used to evaluate the semantic quality of ontology labels and 
    to generate more meaningful names in the Ontology Shredder.

    :param word (str): Word to check.
    :return (bool): True if the word exists in the English word list, otherwise False.
    """
    word = word.lower()
    return (
        word in english_words
    )


def split_camel_case(value):
    """
    Splits a CamelCase string into a list of individual words.

    :param value (str): CamelCase input string.
    :return (list[str]): List of substrings obtained from the split.
    """
    return re.findall(r'[A-ZÄÖÜ]?[a-zäöüß]+|[A-ZÄÖÜ]+(?=[A-ZÄÖÜ]|$)', value)


def is_label_meaningful(label):
    """
    Determines if a label is semantically meaningful based on its components.

    The label is split into words, and at least half must be recognized as 
    valid English words. This heuristic helps decide whether a label is 
    suitable for generating SQL-friendly names in the Ontology Shredder.

    :param label (str): Label or identifier to evaluate.
    :return (bool): True if at least 50% of the parts are valid English words.
    """
    parts = split_camel_case(clean_value(label))
    if not parts:
        return False
    real_words = [w for w in parts if is_known_word(w)]
    return len(real_words) / len(parts) >= 0.5

Sammelt alle Properties zusammen die irgendwie Lable im Namen haben

In [981]:
def get_all_lable_properties(graph, preferred_keywords):
    """
    Collects all annotation properties that include the substring "label".

    The function searches the ontology for properties whose URI or label 
    contains "label" and ranks them by relevance. Preferred variants 
    (e.g., "preferredLabel", "prefLabel") are prioritized.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :return (list[rdflib.URIRef]): Ordered list of property URIs containing "label", 
                                   with preferred ones first.
    """
    g = graph

    rdfs_label = [RDFS.label]
    preferred_label = []
    other_label = []

    all_label = []

    # preferred_keywords = ["preferred", "pref", "pf", "prefer", "preferable", "fav", "favour", "favourite"]

    for s in g.subjects(RDF.type, OWL.AnnotationProperty):
        if "label" in s.lower() and any(p in s.lower() for p in preferred_keywords):
            preferred_label.append(s) # Liste alle bevorzugten Lable-Properties
        if "label" in s.lower() and not any(p in s.lower() for p in preferred_keywords):
            other_label.append(s) # Liste alle anderen Lable-Properties
    
    all_label.extend(preferred_label)
    all_label.extend(rdfs_label)
    all_label.extend(other_label)

    return all_label

In [982]:
def generate_unique_names(graph, elements_by_group, preferred_keywords, schema):
    """
    Generates globally unique and human-readable names for ontology elements.

    Each class, property, and individual receives a unique SQL-safe name, 
    preferably derived from labels or, if missing, from URI fragments. 
    Reserved SQL keywords are automatically suffixed. 
    This ensures name consistency across all generated SQL objects.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param elements_by_group (dict): Groups of ontology elements, e.g.:
        {
            "class": [...],
            "property": [...],
            "individual": [...]
        }
    :return (dict): Dictionary of unique names for each ontology group:
        {
            "class": {classURI: "class_name", ...},
            "property": {propertyURI: "property_name", ...},
            "individual": {indURI: "individual_name", ...}
        }
    
    """
    g = graph
    
    label_predicates = get_all_lable_properties(g, preferred_keywords)
    
    def generate_candidate_labels(uri):
        """Returns all potential name candidates for a URI based on labels and URI fragments."""
        labels = []
        for pred in label_predicates:
            for lbl in g.objects(uri, pred):
                if is_label_meaningful(lbl) and len(clean_value(lbl)) <= 40: # damit Lable nicht zu lang werden
                    match = re.match(r'^(\d+)([A-Z]?[a-z]*)?(.*)', lbl)
                    if match:
                        nummer = match.group(1)  # die Zahlen am Anfang
                        first_word = match.group(2) or ''
                        rest = match.group(3)    # der Rest des Strings
                        labels.append(clean_value(rest + "_" + nummer+first_word))
                    else:
                        labels.append(clean_value(lbl))       
        # Fallback auf gereinigte URI
        labels.append(clean_value(uri))
        return labels

    # Hauptspeicher für Namen
    names = {group: {} for group in elements_by_group}
    used_names = {group: set() for group in elements_by_group}

    for group, elements in elements_by_group.items():
        for uri in elements:
            # Alle Namenskandidaten sammeln
            if group not in ["individual", "classKey", "propertyKey"]:
                candidates = generate_candidate_labels(uri)
            else:
                candidates = [remove_prefix(uri)]

            # Wähle ersten eindeutigen Kandidaten
            final_name = None
            for candidate in candidates:
                if candidate not in used_names[group]:
                    final_name = candidate
                    break

            # Wenn immer noch kein eindeutiger Name gefunden
            if final_name is None:
                for candidate in candidates:
                    prefix_name = f"{schema}_{candidate}"
                    if prefix_name not in used_names[group]:
                        final_name = prefix_name
                        break
                for name in [f"{clean_value(uri)}", f"{schema}_{clean_value(uri)}"]:
                    if name not in used_names[group]:
                        final_name = name
                        break 
                
                suffix = 1
                while f"{candidates[0]}_{suffix}" in used_names[group]:
                    suffix += 1
                final_name = f"{candidates[0]}_{suffix}"

            # von SQL reservierte Begriffe, können nicht für Tabellen oder Attribute genutzt werden 
            pg_reserved_words = {
                "user", "group", "order", "select", "table", "insert", "update", "delete",
                "from", "where", "join", "limit", "offset", "view", "index", "primary",
                "foreign", "references", "constraint", "key"
            }
            if final_name in pg_reserved_words:
                if group in ["class"]:
                    final_name += "_cls"
                elif group in ["property"]:
                    final_name += "_prop"
                elif group in ["individual"]:
                    final_name += "_ind"
                else:
                    final_name += "_"
                
            # Speichern
            names[group][uri] = final_name
            used_names[group].add(final_name)
            
    return names

In [983]:
# def get_sql_name(element, group, unique_names_by_group):
#     """
#     Returns the unique, SQL-safe name assigned to an ontology element.

#     Fetches the resolved name for a given class, property, or individual
#     from the central name registry produced by the Ontology Shredder.

#     :param element (rdflib.URIRef): URI of the ontology element.
#     :param group (str): Element group ("class", "property", or "individual").
#     :param unique_names_by_group (dict): Mapping of element URIs to unique names.
#     :return (str): SQL-compliant, unique name corresponding to the element.
#     """
    
#     unique_names = unique_names_by_group.get("unique_names_by_group", unique_names_by_group)
#     for uri, sql_name in unique_names.get(group).items():
#         if uri == element:
#             return sql_name

In [984]:
def get_sql_name(element, group, unique_names_by_group):
    """
    Returns the unique, SQL-safe name assigned to an ontology element.

    Fetches the resolved name for a given class, property, or individual
    from the central name registry produced by the Ontology Shredder.

    :param element (rdflib.URIRef): URI of the ontology element.
    :param group (str): Element group ("class", "property", or "individual").
    :param unique_names_by_group (dict): Mapping of element URIs to unique names.
    :return (str): SQL-compliant, unique name corresponding to the element.
    """
    if element == set():
        return None
    else:
        unique_names = unique_names_by_group.get("unique_names_by_group", unique_names_by_group)
        sql_name = unique_names.get(group).get(element)
        # for uri, sql_name in unique_names.get(group).items():
        #     if uri == element:
        return sql_name
        

### R3 - Schlüsselwahl


In [985]:
def get_primary_key(graph):
    """    
    Bestimmt die Art des Primärschlüssels für die spätere Tabellen.

    Strategie:
    - Wenn die Ontologie Individuen mit eindeutigen URIs enthält → URI als Primärschlüssel.
    - Wenn URIs mehrfach vorkommen → eindeutige ID generieren.
    - Wenn keine Individuen vorhanden sind → automatische Serien-ID verwenden.

    :param graph: rdflib.Graph - RDFLib-Graph der geladenen Ontologie
    :return: str - Primärschlüsselstrategie ("uri", "auto")
    """  

    g = graph

    # primary_key = 
    has_individuals = ontology_has_individuals(g)[0]
    
    if has_individuals:
        primary_key = "uri"
    else: 
        primary_key = "auto"
    
    return primary_key


def ontology_has_individuals(graph):
    """
    Checks whether the ontology contains individuals and if their URIs are unique.

    The function counts all RDF:type assertions that reference non-system namespaces
    and determines whether each class's individuals have unique URIs. 
    This information is crucial for deciding how to represent individuals 
    and primary keys in the relational schema.

    :param graph (rdflib.Graph): RDF graph containing the ontology.
    :return (tuple[bool, bool, dict]): 
        - **has_instances (bool)**: True if at least one individual exists.  
        - **all_uris_in_class_unique (bool)**: True if all individuals within each class have unique URIs.  
        - **individuals_not_unique (dict)**: Classes where duplicate individual URIs were found.
    """
    g = graph

    total_number_class_individuals = Counter()
    unique_class_individuals = defaultdict(set)
    individuals_not_unique = {}


    has_instances = False
    all_uris_in_class_unique = True
    individuals_not_unique = False

    EXCLUDED_NAMESPACES = {str(OWL), str(RDFS), str(XSD)}

    for s, p, o in g.triples((None, RDF.type, None)):
        if not any(str(o).startswith(ns) for ns in EXCLUDED_NAMESPACES):
            total_number_class_individuals[o] += 1
            unique_class_individuals[o].add(s)
            has_instances = True
    
    for cls in total_number_class_individuals:
        total_number = total_number_class_individuals[cls]
        unique_number = len(unique_class_individuals[cls])

        if not total_number == unique_number:
            all_uris_in_class_unique = False 
            individuals_not_unique[cls] = total_number != unique_number 

    if not all_uris_in_class_unique:
        return has_instances, all_uris_in_class_unique, individuals_not_unique
    
    return has_instances, all_uris_in_class_unique, individuals_not_unique

In [986]:
def get_primary_key_datatype(graph, prop, primary_key_type):
    """
    Determines the SQL datatype for the table's primary key column.

    The function estimates the appropriate VARCHAR length based on theprefix less 
    URI length of individuals. If URIs are too long, it defaults to VARCHAR.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param prop (rdflib.URIRef): A property.
    :param primary_key_type (dict): Contains the chosen primary key strategy ("uri" or "auto").
    :return (str): SQL datatype string for the primary key (e.g., "VARCHAR(100)").
    """
    g = graph
    
    pk_type = primary_key_type["primary_key_type"]
    
    if pk_type == "uri":
        # for individual in all_individuals:
        values = g.subjects(prop, None)
        
        max_len = max((len(remove_prefix(value)) for value in values), default=0)*1.2
        if max_len > 255:
            id_datatype = "VARCHAR"
        elif max_len <= 50:
            id_datatype = "VARCHAR(50)"
        elif max_len <= 100:
            id_datatype = "VARCHAR(100)"
        else:
            id_datatype = "VARCHAR(255)"
    else:
        id_datatype = "VARCHAR(100)"
    
    return id_datatype

### R5 - owl:equivalentClass (einfach)


Hilfsfunktionen \
Kombiniert mehrere gleiche Dictionaries.


In [987]:
def combine_annotation_values(annotation_dict):
    """
    Fügt mehrere Annotation-Dictionaries zu einem gemeinsamen Dictionary zusammen.

    Doppelte Keys werden zusammengeführt; bei mehreren Werten entsteht eine Liste.
    Auch verschachtelte "customAnnotations" werden kombiniert.

    :param annotation_dicts: List[dict] - Liste von Annotation-Dictionaries
    :return: dict - kombiniertes Annotation-Dictionary
    """
    combined = {}

    for anno_dict in annotation_dict:
        for cls, annotations in anno_dict.items():
            for prop, value in annotations.items():
                if value is None:
                    continue

                # Listen zusammenführen
                if isinstance(value, list):
                    combined.setdefault(prop, [])
                    for val in value:
                        if val not in combined[prop]:
                            combined[prop].append(f"{val} ({remove_prefix(cls)})")
                
                # Dictionaries zusammenführen
                elif isinstance(value, dict):
                    combined.setdefault(prop, {})  # Stelle sicher, dass ein dict existiert
                    for subkey, subval in value.items():
                        subval_str = ", ".join(subval) if isinstance(subval, list) else str(subval)
                        annotated_val = f"{subval_str} ({remove_prefix(cls)})"

                        combined[prop].setdefault(subkey, [])
                        if annotated_val not in combined[prop][subkey]:
                            combined[prop][subkey].append(annotated_val)

                # Bool: True gewinnt
                elif isinstance(value, bool):
                    combined[prop] = combined.get(prop, False) or value

                # Alle anderen Werte (None, string, URIRefs, etc.)
                else:
                    if prop not in combined:
                        combined[prop] = value
                    elif combined[prop] is None and value is not None:
                        combined[prop] = value
                    # sonst: existierender Wert bleibt, None wird ignoriert
    return combined

Mappt Äquivalentklassen/Äquivalenzproperties an einen Repräsentanten, an einander und kombiniert die Annotations.

In [988]:
def build_equivalent_map(graph, equivalent_type, unique_names, inverse_equivalent_props = {}, sub_equivalent_props = {}):
    """   
    Builds a mapping structure for equivalent classes or properties in the ontology.

    The function detects equivalence relationships (e.g., owl:equivalentClass, owl:equivalentProperty)
    and groups all connected elements into equivalence groups. For each group, one representative 
    element is selected based on lexical order.
    
    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param equivalent_type (rdflib.term.URIRef): Predicate defining the equivalence relation 
                                                 (e.g., OWL.equivalentClass or OWL.equivalentProperty).
    :param unique_names (dict): Central name registry for ontology elements, used to pick representatives.
    :param inverse_equivalent_props (dict): Optional mapping of inverse properties to integrate.
    :param sub_equivalent_props (dict): Optional mapping of subproperty equivalences to include.
    :return (tuple[dict, dict]): 
        - **equivalent_map (dict)**: Maps each element to its chosen representative.
        - **equivalent_group (dict)**: Maps each element to the full set of equivalent elements.
    """
    g = graph
    
    all_groups = [] # Liste aller gefundenen Äquivalenzklassen-Gruppen
    visited = set() # alle besuchten Klassen, verhindert Zyklen

    equivalent_group = {}
    equivalent_map = {}

    # 1. Äquivalenzklassen-Gruppen finden
    for elem in g.subjects(equivalent_type, None):
        if isinstance(elem, BNode): 
            continue 
        if (elem, RDF.type, OWL.AnnotationProperty) in g: 
            continue
        if elem in visited: 
            continue
        if is_deprecated(g, elem):
            continue
        
        group = set()
        stack = [elem]

        while stack:
            current = stack.pop()
            if current in group: continue
            group.add(current)
            visited.add(current)

            for neighbor in g.objects(current, equivalent_type):
                if isinstance(neighbor, URIRef) and (neighbor, RDF.type, OWL.AnnotationProperty) not in g and not is_deprecated(g, neighbor): 
                    stack.append(neighbor)
                if equivalent_type == OWL.equivalentClass and isinstance(neighbor, BNode):
                    group.add(neighbor)
            for neighbor in g.subjects(equivalent_type, current):
                if isinstance(neighbor, URIRef) and (neighbor, RDF.type, OWL.AnnotationProperty) not in g and not is_deprecated(g, neighbor):
                    stack.append(neighbor)
        if group and all(isinstance(x, URIRef) for x in group):
            if any(gr in inverse_equivalent_props for gr in group):
                group.update(v for k, vlist in inverse_equivalent_props.items() if k in group for v in vlist)
            
            if any(gr in sub_equivalent_props for gr in group):
                group.update(v for k, vlist in sub_equivalent_props.items() if k in group for v in vlist)
                
            for gr in group:
                equivalent_group[gr] = group # Jede Klasse der Gruppe zeigt auf ihre Gruppe
            all_groups.append(group)


    if equivalent_type == OWL.equivalentClass:
        name_group = "class" 
    else:
        name_group = "property"

    for group in all_groups:
        if not group:
            continue
        filtered = [x for x in group if is_internal(x)] # representative should be a internal element
        representative = sorted(
            filtered or group, 
            key=lambda x: get_sql_name(x, name_group, unique_names)
            )[0]
        for element in group:
            equivalent_map[element] = representative

    return equivalent_map, equivalent_group

In [989]:
def build_equivalent_class_map(graph, unique_names, sub_equivalent_group):
    """
    Builds the equivalence mapping for ontology classes.

    A wrapper around 'build_equivalent_map', specialized for class equivalences
    defined via owl:equivalentClass. The mapping merges equivalent classes and 
    ensures consistent representative selection for SQL schema generation.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param unique_names (dict): Central registry of unique element names.
    :param sub_equivalent_group (dict): Subclass equivalence relationships to consider.
    :return (tuple[dict, dict]): 
        Equivalent map and equivalence group for all ontology classes.
    """
    return build_equivalent_map(graph, OWL.equivalentClass, unique_names, {}, sub_equivalent_group)

### R4/R5 - inferred equivalents and sub classes

In [990]:
def compute_inferred_equivalents_and_sub_classes(graph, unique_names, all_classes, max_iterations=10):
    """
    Iteratively computes consistent equivalence and subclass groupings for ontology classes.

    The function refines both the equivalentClass and subClassOf hierarchies 
    until a stable fixpoint is reached. It alternates between computing 
    equivalences and subclass dependencies so that indirect equivalences 
    (e.g., via subclass relations) are correctly merged.

    :param graph (rdflib.Graph): RDF graph containing the ontology.
    :param unique_names (dict): Registry of element names used for representative selection.
    :param all_classes (set): All class URIs in the ontology.
    :param max_iterations (int): Maximum number of refinement iterations (default: 10).
    :return (dict): Final, stable subclass equivalence group mapping after all iterations.

    
    """
    g = graph
    # Initial: nur aus equivalences, inverses leer
    _, equivalent_class_group = build_equivalent_class_map(g, unique_names, {})
    _, _, sub_equivalent_group = build_sub_class_map(g, equivalent_class_group, all_classes)
    
    for i in range(max_iterations):
        # berechne neu mit gegenseitiger Abhängigkeit
        _, new_equiv_group = build_equivalent_class_map(g, unique_names, sub_equivalent_group)
        _, _, new_sub_equiv_group = build_sub_class_map(g, new_equiv_group, all_classes)
        
        if (new_equiv_group == equivalent_class_group 
            and  new_sub_equiv_group == sub_equivalent_group):
            break

        equivalent_class_group = new_equiv_group 
        sub_equivalent_group = new_sub_equiv_group 

    return sub_equivalent_group

## 2. Propertypaket

### Hilfsfunktionen - Nächste Sub-/Superklasse bzw. Sub-/Superproperty

<b> Hilfsfunktionen </b>: Build subclass hierarchy & compute levels

In [991]:
def build_sub_element_graph(graph, sub_type):
    """
    Build a parent and child mapping from all rdfs:subClassOf triples in the graph.

    :param graph: rdflib.Graph - RDFLib graph of the loaded ontology
    :param sub_type: rdflib.URIRef - RDFS.subClassOf or RDFS.subPropertyOf
    :returns: dict[rdflib.URIRef: set[rdflib.URIRef]] - dict{child -> set(parents)}
              dict[rdflib.URIRef: set[rdflib.URIRef]] - dict{parent -> set(children)}
    """
    g = graph

    parents = defaultdict(set)
    children = defaultdict(set)
    for s, o in g.subject_objects(sub_type):
        if isinstance(s, (URIRef, BNode)) and isinstance(o, (URIRef, BNode)):
            parents[s].add(o)
            children[o].add(s)

    return parents, children


In [992]:
def topological_levels(parents):
    """
    Compute a topological level (distance from root) for each element.
    - Roots (elements without parents) get level 0.
    - Children are one level deeper than their highest parent.
    
    :param parents: dict[rdflib.URIRef: set[rdflib.URIRef]] - 
    :returns: dict[rdflib.URIRef: int] - dict{class -> level}
    """
    level = {}
    all_nodes = set(parents.keys()) | {p for ps in parents.values() for p in ps}
    # root = any class without parents
    roots = [n for n in all_nodes if not parents.get(n)] # all classes without a superclass (parent)
    queue = deque(roots)
    for r in roots:
        level[r] = 0
    while queue:
        node = queue.popleft()
        for child in [v for v, ps in parents.items() if node in ps]:
            new_level = level[node] + 1
            if child not in level or new_level > level[child]:
                level[child] = new_level
                queue.append(child)

    return level

In [993]:
def superelement_inclusive(elem, element_group):
    """
    Compute union of a class and all its superclasses (transitively).

    :param elem: rdflib.URIRef - URI der angegebenen Klasse
    :param element_group: dict[rdflib.URIRef: set[rdflib.URIRef]] - 
    :return: set[rdflib.URIRef]

    """
    super_elements = {elem}
    stack = [elem]
    seen = {elem}
    while stack:
        x = stack.pop()
        for e in element_group.get(x, []):
            if e not in seen:
                seen.add(e)
                super_elements.add(e)
                stack.append(e)
    return super_elements

In [994]:
def nearest_common_super_sub_element(graph, elements, sub_type, mode = "super"):
    """
    Compute the nearest common super- or subelement of a given set of elements.
    
    elements contains the elements for which the next common super- or 
    subelement is to be determined.

    :param graph: rdflib.Graph - RDFLib-Graph der geladenen Ontologie
    :param elements: set[rdflib.URIRef] - 
    :param mode: str - either "super" or "sub"
    :return: rdflib.URIRef - 
    """
    g = graph
    
    elements = [e for e in elements]
    if not elements:
        return None
    parents, children = build_sub_element_graph(g, sub_type)
    levels = topological_levels(parents)
    # collect inclusive superclasses for each class
    if mode == "super":
        sup_sets = [superelement_inclusive(c, parents) for c in elements]
    elif mode == "sub":
        sup_sets = [superelement_inclusive(c, children) for c in elements]
    common = set.intersection(*sup_sets) if sup_sets else set() # *sub_set = sub_set[0], sub_set[1], ...
    if not common:
        return None

    start_level = max(levels.get(c, 0) for c in elements)
    return min(common, key=lambda c: abs(levels.get(c, 0) - start_level))

### R18 - owl:InverseFunctionalProperty

In [995]:
def get_inverse_functional_properties(graph, all_properties):
    """
    Determines whether a property is inverse functional and by which reasoning it was inferred.

    A property is considered **inverse functional** if it fulfills one or more of these criteria:
    - Empirically: each object is linked to at most one subject.
    - Declared as 'owl:InverseFunctionalProperty'.
    - Its inverse is declared as 'owl:FunctionalProperty'.
    - Any equivalent property is declared as 'owl:InverseFunctionalProperty'.
    - Any super-property is declared as 'owl:InverseFunctionalProperty'.

    :param graph (rdflib.Graph): RDF graph containing the ontology.
    :param all_properties (list[rdflib.URIRef]): All ontology properties to check.
    :return (tuple[dict, dict]): 
        - **inverse_functional_types**: Mapping of each property to a set of reasoning labels  
          (e.g., {"empirically inverse functional", "functional"}).  
        - **used_as_inverse_functional**: Boolean map indicating whether each property is effectively used as inverse functional.

    """

    g = graph
    inverse_functional_types= {}
    used_as_inverse_functional = {}

    for prop in all_properties:
        # alle Objekte von Triplen mit der Property als Prädikat
        object_with_prop = set(g.objects(None, prop))
        prop_inv_func_types = set()
        prop_used_as_inv_func = False
        
        # if the inverse has type owl:FunctionalProperty than the property is inverse functional
        for s, _, o in g.triples((None, OWL.inverseOf, None)):
            if o == prop and (s, RDF.type, OWL.FunctionalProperty) in g:
                prop_inv_func_types.add("inverse functional")
            elif s == prop and (o, RDF.type, OWL.FunctionalProperty) in g:
                prop_inv_func_types.add("inverse functional")  
        
        # if a equivalent has type owl:InverseFunctionalProperty than the property is inverse functional
        for s, o in g.subject_objects(OWL.equivalentProperty):
            if s == prop and (o, RDF.type, OWL.InverseFunctionalProperty) in g:
                prop_inv_func_types.add("inverse functional")
            if o == prop and (s, RDF.type, OWL.InverseFunctionalProperty) in g:
                prop_inv_func_types.add("inverse functional")

        # if a super Property has type owl:InverseFunctionalProperty than the property is inverse functional
        for o in g.objects(prop, RDFS.subPropertyOf):
            if (o, RDF.type, OWL.InverseFunctionalProperty) in g:
                prop_inv_func_types.add("inverse functional")
        
        # check owl:InverseFunctionalProperty
        if (prop, RDF.type, OWL.InverseFunctionalProperty) in g:
            prop_inv_func_types.add("inverse functional")
        
        # no triples given but functional in any way
        if len(object_with_prop) == 0 and prop_inv_func_types:
            prop_inv_func_types.add("inverse functional")
            prop_used_as_inv_func = True

        # checks if the property is empirically inverse functional
        elif object_with_prop or "inverse functional" in inverse_functional_types:
            is_empirically_inverse_functional = True
            for o in object_with_prop:
                values = list(g.subjects(prop, o))
                if len(values) != 1:
                    is_empirically_inverse_functional = False
                    break
            if is_empirically_inverse_functional:
                prop_inv_func_types.add("empirically inverse functional")
                prop_used_as_inv_func = True
        
        if prop_inv_func_types:    
            inverse_functional_types[prop] = prop_inv_func_types
        used_as_inverse_functional[prop] = prop_used_as_inv_func

    return inverse_functional_types, used_as_inverse_functional

### R17 - owl:inverseOf

In [996]:
def build_inverse_property_group(graph, equivalent_property_group, all_properties):
    """
       Iteratively computes consistent equivalence, subproperty, and inverse property groupings.

    This function refines the mutual dependencies between equivalent, inverse, and subproperty 
    relations until a stable fixpoint is reached. It ensures that indirect relationships 
    (e.g., inverses of equivalents or equivalents of subproperties) are fully propagated 
    through the ontology before relational transformation.

    :param graph (rdflib.Graph): RDF graph containing the ontology.
    :param unique_names (dict): SQL-safe name registry for representative selection.
    :param all_properties (set): Set of all ontology property URIs.
    :param max_iterations (int): Maximum number of refinement iterations (default: 10).
    :return (tuple[dict, dict, dict]): 
        - **inverse_property_group**: Mapping of each property to its inverse group.  
        - **inverse_equivalents_group**: New equivalence relationships derived from inverse links.  
        - **sub_equivalent_group**: Derived subproperty equivalence groupings.
    """
    g = graph
    inverse_group = {}
    inverse_map = {}
    new_equivalents_group = {}
    visited = set()
    
    
    
    candidates = set(g.subjects(OWL.inverseOf, None)) | set(g.objects(None, OWL.inverseOf))
    
    for prop in candidates:
        if prop in visited or prop not in all_properties:
            continue
        
        all_inverses = set()
        
        if isinstance(prop, BNode):
            all_inverses.add(g.value(OWL.equivalentProperty, prop))
            inverse_group[prop] = all_inverses
            continue
        
        equivalent_group_prop = equivalent_property_group.get(prop, {prop})
        visited |= equivalent_group_prop

        
        if any((p, RDF.type, OWL.SymmetricProperty) in g for p in equivalent_group_prop):
            for p in equivalent_group_prop:
                new_equivalents_group.setdefault(p, set()).update(equivalent_group_prop)
            continue
        
        # 1. add direct inverses
        for p in equivalent_group_prop:
            for inv in set(g.objects(p, OWL.inverseOf)).union(g.subjects(OWL.inverseOf, p)):
                if inv == p or isinstance(inv, BNode):
                    continue
                all_inverses.add(inv)
                
        
        # 2. add equivalents from the Inverses
        expanded_inverses = set()
        for inv in all_inverses:
            if (inv, RDF.type, OWL.inverseOf) in g:
                for p in equivalent_group_prop:
                    new_equivalents_group.setdefault(p, set()).add(inv)
                    new_equivalents_group.setdefault(inv, set()).add(p)
            else:
                expanded_inverses |= equivalent_property_group.get(inv, {inv})
        
        # 3. new equivalents over inverse of inverses
        new_equivalents = set()
        for inv in expanded_inverses:
            new_equivalents |= {x for x in g.objects(inv, OWL.inverseOf) if not isinstance(x, BNode)}
            new_equivalents |= {x for x in g.subjects(OWL.inverseOf, inv) if not isinstance(x, BNode)} 
        
        
        if expanded_inverses:
            for p in equivalent_group_prop:
                inverse_group[p] = expanded_inverses
                # inverse_map[p] = representative
        if new_equivalents:
            for p in equivalent_group_prop:
                new_equivalents_group.setdefault(p, set()).update(new_equivalents)
        
    return inverse_group, new_equivalents_group

In [997]:
def build_inverse_property_map(graph, context_dict):
    """
    Builds a representative mapping between inverse properties.

    For each property with an inverse, a tuple of both representatives 
    is created, ensuring consistent naming across the SQL schema. 
    Equivalent properties are unified using the global equivalence map.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param context_dict (dict): Shared ontology context containing:
        - "all_properties"
        - "unique_names_by_group"
        - "equivalent_property_map"
        - "inverse_property_group"
    :return (dict): Mapping of each property to a tuple of its two representative inverses.

    """
    g = graph
    inverse_property_map = {}
    
    all_properties = context_dict["all_properties"]
    unique_names = context_dict["unique_names_by_group"]
    
    equivalent_property_map = context_dict["equivalent_property_map"]
    inverse_property_group = context_dict["inverse_property_group"]
    
    
    for prop in all_properties:
        if not inverse_property_group.get(prop):
            continue        
        
        representative = equivalent_property_map.get(prop, prop)
        
        inverse_prop_group = inverse_property_group.get(prop)
        inverse = next(iter(inverse_prop_group))

        inv_representative = equivalent_property_map.get(inverse, inverse)
        
        prop_inv_rep = sorted(
            {representative, inv_representative}, 
            key=lambda x: (len(get_sql_name(x, "property", unique_names)),
                            get_sql_name(x, "property", unique_names))
        )
        
        inverse_property_map[prop] = tuple(prop_inv_rep)
        
    return inverse_property_map

In [998]:
def get_common_domain_range_class_for_inverse_properties(graph, context_dict, domain_range_map):
    """
    Determines the shared domain and range classes for pairs of inverse properties.

    For each inverse property group, all domain and range declarations are collected.
    If multiple exist, their least common superclasses are computed to harmonize the schema.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param context_dict (dict): Ontology context containing equivalence and inverse groups.
    :param domain_range_map (dict): Mapping of each property to its domain and range.
    :return (dict): Mapping of each property to its harmonized domain and range:
        {
            propertyURI: {"domain": URIRef | None, "range": URIRef | None}
        }
    """
    g = graph
    seen = set()
    common_domain_range = {}

    equivalent_property_group = context_dict["equivalent_property_group"]
    inverse_property_group = context_dict["inverse_property_group"]
    
    for prop, inverse in inverse_property_group.items():
        all_domains = set()
        all_ranges = set()
        for inv in inverse:
            if domain_range_map[inv]["range"]:
                all_domains.add(domain_range_map[inv]["range"])
            if domain_range_map[inv]["domain"]:
                all_ranges.add(domain_range_map[inv]["domain"])
        
        for gr in equivalent_property_group.get(prop, [prop]):
            if domain_range_map[gr]["domain"]:
                all_domains.add(domain_range_map[gr]["domain"])
            if domain_range_map[gr]["range"]:
                all_ranges.add(domain_range_map[gr]["range"])
            
        if len(all_domains) == 1:
            lcs_domain = all_domains.pop()
        else:
            lcs_domain = nearest_common_super_sub_element(g, all_domains, RDFS.subClassOf, "super") if all_domains else None     

        if len(all_ranges) == 1:
            lcs_range = all_ranges.pop()
        else:
            lcs_range  = nearest_common_super_sub_element(g, all_ranges, RDFS.subClassOf, "super")  if all_ranges  else None
        
        common_domain_range[prop] = {
            "domain": lcs_domain if lcs_domain else None,
            "range": lcs_range if lcs_range else None
        }
    
    # TODO INTERSECTION Table for lcs_domain or lcs_range if they are empty
    
    return common_domain_range

### R15 - rdfs:subPropertyOf

In [999]:
def build_sub_property_map(graph, equivalent_property_group, all_properties):
    """
    Builds the subproperty hierarchy map for ontology properties.

    A thin wrapper around 'build_sub_elemenet_map', specialized for 'rdfs:subPropertyOf' relations.
    It structures hierarchical dependencies between properties, supporting inheritance 
    and constraint propagation in the relational schema.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param equivalent_property_group (dict): Mapping of equivalent properties to their groups.
    :param all_properties (set): Set of all property URIs in the ontology.
    :return (tuple[dict, dict, dict]): 
        Superproperty distance, subproperty distance, and derived equivalence groups.
    """
    return build_sub_elemenet_map(graph, RDFS.subPropertyOf, equivalent_property_group, all_properties) 

### R16 - owl:equivalentProperty

Siehe Paket 1 R5/R16 - Äquivalentzklassen (einfach) / Äquivalentzproperties

In [1000]:
def build_equivalent_property_map(graph, unique_names, inverse_equivalent_props, sub_equivalent_props):
    """
    Builds the equivalence map for ontology properties.

    Wrapper around 'build_equivalent_map' specialized for 'owl:equivalentProperty'. 
    It merges direct equivalences, inverse links, and subproperty-derived equivalences 
    into unified equivalence clusters for property alignment in SQL.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param unique_names (dict): SQL-safe name registry.
    :param inverse_equivalent_props (dict): Merged inverses to include.
    :param sub_equivalent_props (dict): Merged subproperty equivalents to include.
    :return (tuple[dict, dict]): Equivalent map and equivalence groups for all properties.
    """
    return build_equivalent_map(graph, OWL.equivalentProperty, unique_names, inverse_equivalent_props, sub_equivalent_props)

Zusammenlegung der Domain/Range für equivalente Properties

In [1001]:
def get_common_domain_range_class_for_equivalent_properties(graph, context_dict, domain_range_map):
    """
    Determines unified domain and range classes for each equivalent property group.

    Each equivalence group is analyzed to check whether all members share the same 
    domain and range. If not, the least common superclass (LCS) of each set is computed. 
    This ensures consistent relational table structures for equivalent properties.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param context_dict (dict): Ontology context containing equivalence mappings.
    :param domain_range_map (dict): Property-to-domain/range mapping.
    :return (dict): 
        Mapping of each property to a dictionary with harmonized domain and range:
        {
            propertyURI: {"domain": URIRef | None, "range": URIRef | None}
        }
    
    """
    g = graph
    
    equivalent_property_map = context_dict["equivalent_property_map"]
    equivalent_property_group = context_dict["equivalent_property_group"]

    seen = set()
    common_domain_range = {}

    for prop, group in equivalent_property_group.items():
        if prop in seen:
            continue

        all_domains = set()
        all_ranges = set()

        for gr in group:
            seen.add(gr)
            if domain_range_map[gr]["domain"]:
                all_domains.add(domain_range_map[gr]["domain"])
            if domain_range_map[gr]["range"]:
                all_ranges.add(domain_range_map[gr]["range"])
        
        if len(all_domains) == 1:
            lcs_domain = all_domains.pop()
        else:
            lcs_domain = nearest_common_super_sub_element(g, all_domains, RDFS.subClassOf, "super") if all_domains else None     

        if len(all_ranges) == 1:
            lcs_range = all_ranges.pop()
        else:
            lcs_range  = nearest_common_super_sub_element(g, all_ranges, RDFS.subClassOf, "super")  if all_ranges  else None
        
        for gr in group:
            common_domain_range[gr] = {
                "domain": lcs_domain if lcs_domain else None,
                "range": lcs_range if lcs_range else None,
            }
    
    # TODO INTERSECTION Table for lcs_domain or lcs_range if they are empty
    
    return common_domain_range

### R15/R16/R17 - inferred equivalents, inverses and sub properties

In [1002]:
def compute_inferred_equivalents_sub_property_and_inverses(graph, unique_names, all_properties, max_iterations=10):
    """
    Iteratively computes consistent equivalence and subclass groupings for ontology classes.

    The function refines both the equivalentClass and subClassOf hierarchies 
    until a stable fixpoint is reached. It alternates between computing 
    equivalences and subclass dependencies so that indirect equivalences 
    (e.g., via subclass relations) are correctly merged.

    :param graph (rdflib.Graph): RDF graph containing the ontology.
    :param unique_names (dict): Registry of element names used for representative selection.
    :param all_properties (set): All property URIs in the ontology.
    :param max_iterations (int): Maximum number of refinement iterations (default: 10).
    :return (dict): Final, stable subclass equivalence group mapping after all iterations.
    """
    g = graph
    # Initial: nur aus equivalences, inverses leer
    equivalent_property_map, equivalent_property_group = build_equivalent_property_map(g, unique_names, {}, {})
    inverse_property_group, inverse_equivalents_group = build_inverse_property_group(g, equivalent_property_group, all_properties)
    _, _, sub_equivalent_group = build_sub_property_map(g, equivalent_property_group, all_properties)
    
    for i in range(max_iterations):
        # berechne neu mit gegenseitiger Abhängigkeit
        new_equiv_map, new_equiv_group = build_equivalent_property_map(g, unique_names, inverse_equivalents_group, sub_equivalent_group)
        new_inv_group, new_inv_equi_group = build_inverse_property_group(g, new_equiv_group, all_properties)
        _, _, new_sub_equiv_group = build_sub_property_map(g, new_equiv_group, all_properties)
        
        if (new_equiv_group == equivalent_property_group 
            and (new_inv_group == {} or new_inv_group == inverse_property_group)
            and  new_sub_equiv_group == sub_equivalent_group):
            break

        equivalent_property_group = new_equiv_group 
        inverse_property_group = new_inv_group
        inverse_equivalents_group = new_inv_equi_group
        sub_equivalent_group = new_sub_equiv_group 

    return inverse_property_group, inverse_equivalents_group, sub_equivalent_group

In [1003]:
def get_rdf_types_and_inferred_types(graph, context_dict):
    """
    Determines and harmonizes all explicit and inferred RDF types for ontology properties.

    The function integrates reasoning from multiple sources:
    - Declared rdf:type statements,
    - Equivalence and inverse relations,
    - Sub- and super-property inheritance,
    - Functional and inverse functional property propagation.

    The result provides an enriched view of each propertys classification 
    (e.g., FunctionalProperty, ObjectProperty, etc.), as well as whether it 
    is empirically used as functional or inverse functional.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param context_dict (dict): Shared ontology context containing equivalence, 
                                inverse, sub/super relations, and functional status.
    :return (tuple[dict, dict, dict, dict, dict]): 
        - **inferred_rdf_types**: All inferred RDF.type values for each property.  
        - **is_functional**: Boolean flag for functional properties.  
        - **is_inverse_functional**: Boolean flag for inverse functional properties.  
        - **functional_types**: Aggregated sources/reasons for functional inference.  
        - **inverse_functional_types**: Aggregated sources/reasons for inverse functional inference.
    """
    g = graph
    
    all_properties = context_dict["all_properties"]
    
    equivalent_property_group = context_dict["equivalent_property_group"]
    inverse_property_group = context_dict["inverse_property_group"]
    sub_property_map = context_dict["sub_property_of_dist"]
    super_property_map = context_dict["super_property_of_dist"]
    
    used_as_functional = context_dict["used_as_functional_solo"]
    functional_types_prop = context_dict["functional_types_solo"]
    used_as_inverse_functional = context_dict["used_as_inverse_functional_solo"]
    inverse_functional_types_prop = context_dict["inverse_functional_types_solo"]
    
    
    inferred_rdf_types = {}
    is_functional = {}
    functional_types = {}
    is_inverse_functional = {}
    inverse_functional_types = {}
    
    for prop in all_properties:
        prop_functional = used_as_functional.get(prop)
        prop_functional_type = functional_types_prop.get(prop, set())
        prop_inverse_functional = used_as_inverse_functional.get(prop)
        prop_inverse_functional_type = inverse_functional_types_prop.get(prop, set())
        equivalent_prop_group = equivalent_property_group.get(prop, [])
        inverse_prop_group = inverse_property_group.get(prop, [])
        sub_prop_of_map = sub_property_map.get(prop, [])
        super_property_of_map = super_property_map.get(prop, [])

        
        all_rdf_type = {t for t in g.objects(prop, RDF.type) if isinstance(t, URIRef)}
        if (prop, OWL.deprecated, Literal(True)) in g:
            all_rdf_type.add(OWL.deprecated)
        all_func_types = prop_functional_type
        all_inv_func_types = prop_inverse_functional_type
        
        for equi in equivalent_prop_group:
            types = {t for t in g.objects(equi, RDF.type) if isinstance(t, URIRef) and t != OWL.DeprecatedProperty}
            all_rdf_type |= types
            all_func_types |= functional_types_prop.get(equi, set())
            all_inv_func_types |= inverse_functional_types_prop.get(equi, set())
            
            if prop_functional and not used_as_functional.get(equi):
                prop_functional = False
                
            if prop_inverse_functional and not used_as_inverse_functional.get(equi):
                prop_inverse_functional = False
            
        for inv in inverse_prop_group:
            for t in g.objects(inv, RDF.type):
                if t == OWL.DeprecatedProperty:
                    continue
                
                if t == OWL.FunctionalProperty:
                    all_rdf_type.add(OWL.InverseFunctionalProperty)
                elif t == OWL.InverseFunctionalProperty:
                    all_rdf_type.add(OWL.FunctionalProperty)
                elif isinstance(t, URIRef):
                    all_rdf_type.add(t)

            
            if prop_functional and not used_as_inverse_functional.get(inv):
                prop_functional = False
            if prop_inverse_functional and not used_as_functional.get(inv):
                prop_inverse_functional = False

        
        for super_prop in sub_prop_of_map:
            for t in g.objects(super_prop, RDF.type):
                if t in [OWL.FunctionalProperty, OWL.InverseFunctionalProperty, OWL.AsymmetricProperty, OWL.IrreflexiveProperty]:
                    all_rdf_type.add(t)
                all_func_types |= functional_types_prop.get(super_prop, set())
                all_inv_func_types |= inverse_functional_types_prop.get(super_prop, set())
                
        # If a sub property is not (inverse) functional, the super property cannot be (inverse) functional either
        for sub_prop in super_property_of_map:
            if prop_functional and not used_as_functional.get(sub_prop):
                prop_functional = False
            if prop_inverse_functional and not used_as_inverse_functional.get(sub_prop):
                prop_inverse_functional = False
        
        # NOTE ist eigentlich nicht nötig, da das alles schon beim Reasoning abgefangen wird.
        #      Und meine Vorrausetzung ja ein erfolgreiches Reasoning ist.
        if OWL.SymmetricProperty in all_rdf_type and OWL.AsymmetricProperty in all_rdf_type:
            all_rdf_type = "Error"
        elif OWL.ReflexiveProperty in all_rdf_type and OWL.AsymmetricProperty in all_rdf_type:
            all_rdf_type = "Error"
        elif OWL.ReflexiveProperty in all_rdf_type and OWL.IrreflexiveProperty in all_rdf_type:
            all_rdf_type = "Error"
        elif OWL.DatatypeProperty in all_rdf_type and OWL.ObjectProperty in all_rdf_type:
            all_rdf_type = "Error"
        
        if isinstance(all_rdf_type, set):
            if not prop_functional:
                all_rdf_type.discard(OWL.FunctionalProperty)
            else:
                all_rdf_type.add(OWL.FunctionalProperty)
            if not prop_inverse_functional:
                all_rdf_type.discard(OWL.InverseFunctionalProperty)
            else:
                all_rdf_type.add(OWL.InverseFunctionalProperty)
        
        inferred_rdf_types[prop] = all_rdf_type
        is_functional[prop] = prop_functional
        is_inverse_functional[prop] = prop_inverse_functional
        functional_types[prop] = all_func_types
        inverse_functional_types[prop] = all_inv_func_types

    return inferred_rdf_types, is_functional, is_inverse_functional, functional_types, inverse_functional_types
        

### R29 - rdfs:domain/rdfs:range

In [1004]:
def infer_side(graph, prop, side, context_dict):
    """
    Infers the domain or range of a given property based on ontology structure and instance data.

    The inference follows a priority order:
    1. Explicitly declared 'rdfs:domain' or 'rdfs:range'.
    2. Inherited from subproperties or superproperties.
    3. Empirically derived from instance data (if available).

    :param graph (rdflib.Graph): RDF graph containing the ontology.
    :param prop (rdflib.URIRef): Property whose domain or range is to be inferred.
    :param side (str): Either "domain" or "range".
    :param context_dict (dict): Contains precomputed sub- and super-property maps.
    :return (tuple[rdflib.URIRef | None, str | None]): 
        - The inferred class URI (if found).  
        - A textual description of the inference source ("declared", "super properties", "empirical", etc.).

    """
    g = graph 
    
    sub_property_map = context_dict["sub_property_of_dist"]
    super_property_map = context_dict["super_property_of_dist"]
    
    candidates = set()
    
    for cls in g.objects(prop, getattr(RDFS, side)):
        candidates.add(cls)
    if candidates:
        if len(candidates) == 1:
            return candidates.pop(), "declared"
        elif len(candidates) > 1:
            super_class = nearest_common_super_sub_element(g, candidates, RDFS.subClassOf, "super")
            if super_class:
                return super_class, "declared"


    # 4. Subproperties
    if sub_property_map.get(prop):
        candidates = set()
        for sub, dist in sub_property_map[prop].items():
            if dist == 1:
                for cls in g.objects(sub, getattr(RDFS, side)):
                    candidates.add(cls)
        if len(candidates) == 1:
            return candidates.pop(), "sub properties"
        elif len(candidates) > 1:
            super_class = nearest_common_super_sub_element(g, candidates, RDFS.subClassOf, "super")
            if super_class:
                return super_class, "sub properties"
    # 5. Superproperty
    if super_property_map.get(prop):
        candidates = set()
        for sub, dist in super_property_map[prop].items():
            if dist == 1:
                for cls in g.objects(sub, getattr(RDFS, side)):
                    candidates.add(cls)
        if len(candidates) == 1:
            return candidates.pop(), "super properties"
        elif len(candidates) > 1:
            super_class = nearest_common_super_sub_element(g, candidates, RDFS.subClassOf, "super")
            if super_class:
                return super_class, "super properties"

    # 6. Empirical from data
    candidates = set()
    for ins in g.subjects(prop, None):
        for cls in g.objects(ins, RDF.type):
            candidates.add(cls)
    if len(candidates) == 1:
        return candidates.pop(), "empirical"
    elif len(candidates) > 1:
        superc = nearest_common_super_sub_element(g, candidates, RDFS.subClassOf, "super")
        if superc:
            return superc, "empirical"

    return None, None



In [1005]:
def build_domain_range_map(graph, context_dict, common_domain_range_equi, common_domain_range_inv):
    """
    Builds a unified domain/range map for all properties.

    The function integrates domain and range information from multiple inference sources
    in a fixed priority order:
      1. From equivalent property groups.
      2. From inverse property groups.
      3. From local declarations or structural inference (via 'infer_side').

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param context_dict (dict): Shared ontology context with property hierarchies.
    :param common_domain_range_equi (dict): Domain/range inferred from equivalent properties.
    :param common_domain_range_inv (dict): Domain/range inferred from inverse properties.
    :return (dict): 
        Mapping of each property to its harmonized domain and range structure:
        {
            propURI: {
                "domain": URIRef | None,
                "source_domain": str,
                "range": URIRef | None,
                "source_range": str
            }
        }
    """
    g = graph
    
    domain_range_map = {}
    all_properties = context_dict["all_properties"]
    
    for prop in all_properties:
        # Prio 1: Equivalent property
        if common_domain_range_equi.get(prop):
            domain_range_map[prop] = {
                "domain": common_domain_range_equi[prop]["domain"],
                "source_domain": "equivalent",
                "range": common_domain_range_equi[prop]["range"],
                "source_range": "equivalent",
            }
        
        # Prio 2: Inverse property
        elif common_domain_range_inv.get(prop):
            domain_range_map[prop] = {
                "domain": common_domain_range_inv[prop]["domain"],
                "source_domain": "inverse",
                "range": common_domain_range_inv[prop]["range"],
                "source_range": "inverse",
            }
        
        # Prio 3 - 6: 
        else:
            domain, src_domain = infer_side(g, prop, "domain", context_dict)
            range_, src_range = infer_side(g, prop, "range", context_dict)

            domain_range_map[prop] = {
                "domain": domain,
                "source_domain": src_domain,
                "range": range_,
                "source_range": src_range,
            }

    return domain_range_map

            

In [1006]:
def infer_domain_range_fixpoint(graph, context_dict, max_iterations=10):
    """
    Iteratively refines domain and range assignments until a stable fixpoint is reached.

    The function repeatedly combines three inference sources:
    1. Direct and structural inference per property ('build_domain_range_map')
    2. Common domain/range consolidation across equivalent properties
    3. Common domain/range consolidation across inverse properties

    Iteration stops once no new domain or range assignments are produced, 
    ensuring a semantically stable mapping basis for relational transformation.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param context_dict (dict): Global ontology context with equivalence, inverse, and subproperty mappings.
    :param max_iterations (int): Maximum number of iterations before termination (default: 10).
    :return (tuple[dict, dict, dict]): 
        - **domain_range_map**: Final per-property domain and range assignments.  
        - **common_domain_range_equivalent**: Shared domain/range per equivalence group.  
        - **common_domain_range_inverse**: Shared domain/range per inverse property group.
    """
    g = graph
    domain_range_map = {}
    common_domain_range_equivalent = {}
    common_domain_range_inverse = {}

    for i in range(max_iterations):

        # inferred pro Property
        domain_range_map_new = build_domain_range_map(
            g, context_dict, common_domain_range_equivalent, common_domain_range_inverse
        )

        # common pro Äquivalenzgruppe
        common_domain_range_equivalent_new = get_common_domain_range_class_for_equivalent_properties(
            g, context_dict, domain_range_map_new
        )

        # common pro Inversegruppe
        common_domain_range_inverse_new = get_common_domain_range_class_for_inverse_properties(
            g, context_dict, domain_range_map_new
        )
        # Fixpunktprüfung
        if (domain_range_map_new == domain_range_map 
            and common_domain_range_equivalent_new == common_domain_range_equivalent
            and common_domain_range_inverse_new == common_domain_range_inverse
            ):
            break

        domain_range_map = domain_range_map_new
        common_domain_range_equivalent = common_domain_range_equivalent_new
        common_domain_range_inverse = common_domain_range_inverse_new
        
    return domain_range_map, common_domain_range_equivalent, common_domain_range_inverse

### R12 - hasKey

- einzelne SV Properties: siehe <b> R6/R7 - Single Value Property </b>
- mehrer SV Properties: siehe <b> Klassen Info + Tabellen Definition & SQL-Erstellung </b>
- min eine MV Property:

### R13/R14 - Multi Value Properties

### R19 - owl:SymmetricProperty

### R20 - owl:AsymmetricProperty

### R21 - owl:TransitiveProperty

### R22 - owl:ReflexiveProperty

get_property_info(...):
- Test ob Property Reflexive ist (SV + MV)
- Definition für CHECK subjekt_id <> objekt_id (MV)

table_definition_class(...)
- Kommentar an Attribut (SV)
- Definition der GENERATED Column (SV)

table_definition_multi_value_properties(...)
- Kommentar an Tablle (MV)
- Definition der View (MV)

### R23 - owl:IrreflexiveProperty

get_property_info(...):
- Test ob Property Irreflexive (SV + MV)
- Definition für CHECK subjekt_id <> objekt_id (SV + MV)

table_definition_class(...)
- Kommentar an Attribut (SV)

table_definition_multi_value_properties(...)
- Kommentar an Tablle (MV)

### R25 - owl:allDisjointProperties

In [1007]:
def extend_element_disjoint_map(graph, element_type, disjoint_map, disjoint_pairs, all_disjoint, disjoint_classes):
    """
    Extends an existing disjoint property map with additional disjointness information.

    Processes both 'owl:propertyDisjointWith' and 'owl:AllDisjointProperties' constructs 
    to identify mutually exclusive property pairs or groups. Distinguishes between:
    - Properties that can share a table ("sameTable")
    - Properties requiring separate tables ("differentTable")
    based on domain/range compatibility and functional usage.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param disjoint_map (dict): Existing disjointness map to extend.
    :param disjoin_pairs (list[tuple]): Accumulated list of detected disjoint property pairs.
    :param used_as_functional (dict): Boolean map indicating if each property is used functionally.
    :return (tuple[dict, list[tuple]]): 
        - **disjoint_map**: Updated mapping of properties to their disjoint groups.  
        - **disjoin_pairs**: List of all discovered disjoint property pairs.
    """
    g = graph
    
    for s in set(g.subjects(RDF.type, all_disjoint)):
        if any(isinstance(list(Collection(g, members_bnode)), BNode) for members_bnode in set(g.objects(s, OWL.members))):
            continue
        for members_bnode in set(g.objects(s, OWL.members)):

            members = list(Collection(g, members_bnode))
            if len(members) == 2:
                s = members[0]
                o = members[1]
                domain_s = g.value(s, RDFS.domain)
                range_s = g.value(s, RDFS.range)
                domain_o = g.value(o, RDFS.domain)
                range_o = g.value(o, RDFS.range)
                
                domain_disjoint = False
                range_disjoint = False
                if any(set((domain_o, domain_s)).issubset(dc) for dc in disjoint_classes):
                    domain_disjoint = True
                if any(set((range_o, range_s)).issubset(dc) for dc in disjoint_classes):
                    range_disjoint = True
                
                # common_domain_subclass = nearest_common_super_sub_element(g, [domain_o, domain_s], RDFS.subClassOf, "sub")
                # common_range_subclass = nearest_common_super_sub_element(g, [range_o, range_s], RDFS.subClassOf, "sub")
                disjoint_map[s].add(o)
                if element_type == "class" or (not domain_disjoint and not range_disjoint):
                    disjoint_pairs.append((s,o))

            else:
                members_domain = set()
                members_range = set()
                for m in members:
                    members_domain.add(g.value(m, RDFS.domain))
                    members_range.add(g.value(m, RDFS.range))

                domain_disjoint = False
                range_disjoint = False
                if any(members_domain.issubset(dc) for dc in disjoint_classes):
                    domain_disjoint = True
                if any(members_range.issubset(dc) for dc in disjoint_classes):
                    range_disjoint = True
                
                # common_domain_subclass = nearest_common_super_sub_element(g, members_domain, RDFS.subClassOf, "sub")
                # common_range_subclass = nearest_common_super_sub_element(g, members_range, RDFS.subClassOf, "sub")
                for m in members:
                    disjoint_map[m].add(tuple(members))
                if element_type == "class" or (not domain_disjoint and not range_disjoint):
                    if len(members) <= 6:
                        disjoint_pairs.append(tuple(members))   
            
    return disjoint_map, disjoint_pairs

### R24 - owl:propertyDisjointWith

In [1008]:
def build_element_disjoint_map(graph, element_type, disjoint_with, all_disjoint, context_dict):
    """
    Builds a disjointness map for ontology elements (classes or properties).

    Integrates pairwise and grouped disjointness relations using both
    'owl:disjointWith' and 'owl:AllDisjoint...' constructs, while also checking
    for domain/range compatibility for properties.

    :param graph (rdflib.Graph): RDF graph containing the ontology.
    :param element_type (str): "class" or "property".
    :param disjoint_with (rdflib.term.URIRef): Predicate used for pairwise disjointness (e.g. OWL.disjointWith).
    :param all_disjoint (rdflib.term.URIRef): Predicate for grouped disjointness (e.g. OWL.AllDisjointClasses).
    :param context_dict (dict): Shared ontology context containing domain/range mappings.
    :return (tuple[dict, list[tuple]]): 
        - **disjoint_map**: Mapping of each element to disjoint elements/groups.  
        - **disjoint_pairs**: List of disjoint element pairs (for constraint generation).
    """
    g = graph
    disjoint_map = defaultdict(set)
    disjoint_pairs = []

    domain_range_map = context_dict["domain_range_map"]
    if element_type == "property":
        disjoint_classes = context_dict["classes_disjoint_pairs"]
    else:
        disjoint_classes = []
    
    for s, o in g.subject_objects(disjoint_with):
        if isinstance(o, BNode) or isinstance(s, BNode):
            continue
        domain_s = domain_range_map.get(s, {}).get("domain")
        range_s = domain_range_map.get(s, {}).get("range")
        domain_o = domain_range_map.get(o, {}).get("domain")
        range_o = domain_range_map.get(o, {}).get("range")
        
        domain_disjoint = False
        range_disjoint = False
        if any(set((domain_o, domain_s)).issubset(dc) for dc in disjoint_classes):
            domain_disjoint = True
        if any(set((range_o, range_s)).issubset(dc) for dc in disjoint_classes):
            range_disjoint = True
            
        # common_domain_subclass = nearest_common_super_sub_element(g, [domain_o, domain_s], RDFS.subClassOf, "sub")
        # common_range_subclass = nearest_common_super_sub_element(g, [range_o, range_s], RDFS.subClassOf, "sub")
        disjoint_map[s].add(o)
        disjoint_map[o].add(s)
        if element_type == "class" or (not domain_disjoint and not range_disjoint):
            # disjoint_map[s].setdefault("differentTable", set()).add(o)
            disjoint_pairs.append((s,o))

            
    disjoint_map, disjoint_pairs = extend_element_disjoint_map(g, element_type, disjoint_map, disjoint_pairs, all_disjoint, disjoint_classes)
    
    return dict(disjoint_map), disjoint_pairs

In [1009]:
def build_property_disjoint_map(graph, context_dict):
    """
    Builds the disjointness map specifically for ontology properties.

    Wrapper around 'build_element_disjoint_map' with 'OWL.propertyDisjointWith' 
    and 'OWL.AllDisjointProperties'. The resulting map is used to create 
    relational-level integrity checks preventing co-occurrence of disjoint properties.

    :param graph (rdflib.Graph): RDF graph containing the ontology.
    :param context_dict (dict): Shared ontology context (incl. domain/range map).
    :return (tuple[dict, list[tuple]]): 
        Disjoint map and list of disjoint property pairs.
    """
    g = graph
    return build_element_disjoint_map(g, "property", OWL.propertyDisjointWith, OWL.AllDisjointProperties, context_dict)

In [1010]:
def build_class_disjoint_map(graph, context_dict):
    """
    Builds the disjointness map for ontology classes.

    Wrapper around 'build_element_disjoint_map' with 'OWL.disjointWith' 
    and 'OWL.AllDisjointClasses'. The result is used to enforce mutual 
    exclusivity between class instances on the relational level.

    :param graph (rdflib.Graph): RDF graph containing the ontology.
    :param context_dict (dict): Shared ontology context.
    :return (tuple[dict, list[tuple]]): 
        Disjoint map and list of disjoint class pairs/groups.
    """
    g = graph
    return build_element_disjoint_map(g, "class", OWL.disjointWith, OWL.AllDisjointClasses, context_dict)

### R26 - owl:propertyChainAxiom

In [1011]:
def get_property_chains_map(graph):
    """
    Extracts all owl:propertyChainAxiom definitions from the RDF graph.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :return (dict): list of property chains for each property.
    """
    g = graph
    
    property_chain_map = defaultdict(list)

    for prop, chain_props in g.subject_objects(OWL.propertyChainAxiom):
        chain_list = Collection(g, chain_props)
        
        property_chain_map[prop].append([chain_prop for chain_prop in chain_list])
        
    return property_chain_map
        

### R27 - owl:NegativePropertyAssertion

In [1012]:
def get_negative_property_assertion(graph, all_properties, equivalent_property_map):
    """
    Extracts all forbidden subject-object pairs defined by 'owl:NegativePropertyAssertion'.

    Negative property assertions specify instance pairs that **must not** 
    be connected by a given property. For symmetric properties, the 
    inverse direction is also included automatically.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param all_properties (set[rdflib.URIRef]): All property URIs to consider.
    :param equivalent_property_map (dict): Mapping of equivalent properties to their representatives.
    :return (dict[rdflib.URIRef, set[tuple]]): 
        Mapping of each property to the set of disallowed (subject, object) pairs.
    """
    g = graph
    assertions = defaultdict(set)
    
    symmetric_props = {
        prop for prop in all_properties
        if (prop, RDF.type, OWL.SymmetricProperty) in g
    }

    for npa in g.subjects(RDF.type, OWL.NegativePropertyAssertion):
        source = g.value(npa, OWL.sourceIndividual)
        asserted_prop = g.value(npa, OWL.assertionProperty)
        target = g.value(npa, OWL.targetIndividual)
        
        if asserted_prop in all_properties:
            prop_name = (equivalent_property_map.get(asserted_prop) or asserted_prop)
            assertions[prop_name].add((source, target))

            if asserted_prop in symmetric_props:
                assertions[prop_name].add((target, source))

    return dict(assertions)

## 3. Komplexe Klassen

### R39 - owl:oneOf

In [1013]:
def build_one_of_map(graph, all_classes):
    """
    Extracts all 'owl:oneOf' enumerations for classes.

    Detects explicitly enumerated class definitions such as:
        - Directly attached 'owl:oneOf' collections.
        - Nested enumerations within 'rdfs:subClassOf' or 'owl:equivalentClass'.

    The resulting structure groups classes by relation type and lists their member individuals.

    :param graph (rdflib.Graph): RDF graph containing the ontology.
    :param all_classes (set[rdflib.URIRef]): Set of all ontology classes.
    :return (dict[rdflib.URIRef, dict]): 
        Mapping of each class to a dictionary of enumeration lists per relation:
        {
            ex:ColorClass: {
                owl:equivalentClass: [[ex:Red, ex:Blue, ex:Green]],
                "direct": [[ex:A, ex:B]]
            }
        }
    """
    g = graph
    one_of_map = {}
    
    for cls in all_classes:
        one_of_map_cls = defaultdict(list)
        
        # owl:oneOf in owl:equivalentClass or rdfs:subClassOf
        for rel in [RDFS.subClassOf, OWL.equivalentClass]:
            for bnode in g.objects(cls, rel):
                if (bnode, OWL.oneOf, None) in g:
                    for collection in g.objects(bnode, OWL.oneOf):
                        members = list(g.items(collection))
                        if members:
                            one_of_map_cls[rel].append(members)
        
        # directly at the class
        for collection in g.objects(cls, OWL.oneOf):
            members = list(g.items(collection))
            if members:
                one_of_map_cls["direct"].append(members)
        
        if one_of_map_cls:
            one_of_map[cls] = dict(one_of_map_cls)
    
    return dict(one_of_map)

### R30/R31/R32 - komplexe Klassen

In [1014]:
def get_complex_class_map(graph, all_classes):
    """
    Extracts complex class expressions such as restrictions, intersections, unions, complements, and embedded enumerations.

    Parses logical OWL constructs connected to each class through:
        - 'rdfs:subClassOf'
        - 'owl:equivalentClass'
        - 'owl:disjointWith'

    Identifies restriction patterns (e.g. 'someValuesFrom', 'allValuesFrom', cardinalities),
    logical combinations ('unionOf', 'intersectionOf', 'complementOf'),
    and embedded 'oneOf' enumerations. 

    :param graph (rdflib.Graph): RDF graph containing the ontology.
    :param all_classes (set[rdflib.URIRef]): Set of ontology classes to analyze.
    :return (dict[rdflib.URIRef, dict]): 
        Detailed structure for each class describing its logical composition:
        {
            ex:Person: {
                owl:equivalentClass: {
                    "predicate": owl:intersectionOf,
                    "classes": {ex:Human, ex:Agent},
                    "restrictions": [...],
                    "oneOf": [...]
                },
                rdfs:subClassOf: {...}
            }
        }
    """
    g = graph

    all_restrictions = {}
    
    def parse_restriction(g, node, containing_elements, containing_restriction_classes):
        """
        Extrahiert OWL-Restriction-Details aus einem BNode.
        """
        restriction = {
            "onProperty": g.value(node, OWL.onProperty),
            "onClass": g.value(node, OWL.onClass),
            "hasValue": g.value(node, OWL.hasValue),
            "hasSelf": g.value(node, OWL.hasSelf),
            "someValuesFrom": g.value(node, OWL.someValuesFrom),
            "allValuesFrom": g.value(node, OWL.allValuesFrom),
            "cardinality": g.value(node, OWL.cardinality),
            "minCardinality": g.value(node, OWL.minCardinality),
            "maxCardinality": g.value(node, OWL.maxCardinality),
            "qualifiedCardinality": g.value(node, OWL.qualifiedCardinality),
            "minQualifiedCardinality": g.value(node, OWL.minQualifiedCardinality),
            "maxQualifiedCardinality": g.value(node, OWL.maxQualifiedCardinality),
        }
        restriction = {k: v for k, v in restriction.items() if v}
        contains_bnode = any(isinstance(v, BNode) for v in restriction.values())
        containing_elements.update({v for k, v in restriction.items()
                if isinstance(v, URIRef) and k in ["onProperty"]
            })
        containing_restriction_classes.update({v for k, v in restriction.items()
                if isinstance(v, URIRef) and k in ["onClass", "someValuesFrom", "allValuesFrom"]
            })
        return restriction, contains_bnode, containing_elements, containing_restriction_classes
                
    for cls in all_classes:
        cls_restrictions = {}
        
        for relation in [RDFS.subClassOf, OWL.equivalentClass, OWL.disjointWith]:
            containing_elements = set()
            containing_restriction_classes = set()
            related_nodes = set(g.objects(cls, relation))
            if not related_nodes:
                continue
            
            if len(related_nodes) == 1:
                rest = next(iter(related_nodes))
                
                if isinstance(rest, URIRef) or (rest, OWL.oneOf, None) in g:
                    continue
                
                if (rest, RDF.type, OWL.Restriction) in g:
                    restriction, col_contains_bnode, containing_elements, containing_restriction_classes = parse_restriction(g, rest, containing_elements, containing_restriction_classes)
                    cls_restrictions[relation] = {
                        "no_bnodes": not col_contains_bnode,
                        "number_of_conditions": 1,
                        "predicate": "single",
                        "containing_elements": containing_elements,
                        "classes_in_restrictions": containing_restriction_classes,
                        "restrictions": restriction
                    }
                    continue
                
                for list_pred in [OWL.unionOf, OWL.intersectionOf, OWL.complementOf]:
                    if (rest, list_pred, None) not in g:
                        continue
                    
                    contains_bnode = False
                    classes = set()
                    owl_restriction = []
                    one_ofs = []
                    
                    for collection in g.objects(rest, list_pred):
                        for node in g.items(collection):
                            if isinstance(node, URIRef):
                                classes.add(node)
                            elif (node, RDF.type, OWL.Restriction) in g:
                                restriction, col_contains_bnode, containing_elements, containing_restriction_classes = parse_restriction(g, node, containing_elements, containing_restriction_classes)
                                owl_restriction.append(restriction)
                                if not contains_bnode and col_contains_bnode:
                                    contains_bnode = True
                            elif (node, OWL.oneOf, None) in g:
                                for col in g.objects(node, OWL.oneOf):
                                    one_ofs.extend(list(g.items(col)))
                    
                    cls_restrictions[relation] = {
                        "no_bnodes": not contains_bnode,
                        "number_of_conditions": len(classes) + len(owl_restriction) + (1 if one_ofs else 0),
                        "containing_elements": containing_elements.union(classes),
                        "predicate": list_pred,
                    }
                    if classes:
                        cls_restrictions[relation]["classes"] = classes
                    if owl_restriction:
                        cls_restrictions[relation]["restrictions"] = owl_restriction
                        cls_restrictions[relation]["classes_in_restrictions"] = containing_restriction_classes,
                    if one_ofs:
                        cls_restrictions[relation]["oneOf"] = one_ofs
                        
                                
            elif len(related_nodes) > 1:
                if all(isinstance(rest, URIRef) for rest in related_nodes):
                    continue
                if all((rest, OWL.oneOf, None) in g for rest in related_nodes):
                    continue
                
                contains_bnode = False
                classes = set()
                owl_restriction = []
                one_ofs = []
                bnodes = 0
                
                for node in related_nodes:
                    if isinstance(node, URIRef):
                        classes.add(node)
                    elif (node, OWL.oneOf, None) in g:
                        for col in g.objects(node, OWL.oneOf):
                            one_ofs.extend(list(g.items(col)))
                    elif (node, RDF.type, OWL.Restriction) in g:
                        restriction, col_contains_bnode, containing_elements, containing_restriction_classes = parse_restriction(g, node, containing_elements, containing_restriction_classes)
                        owl_restriction.append(restriction)
                        if not contains_bnode and col_contains_bnode:
                            contains_bnode = True
                    elif (isinstance(node, BNode)):
                        contains_bnode = True
                        bnodes += 1
                    
                            
                cls_restrictions[relation] = {
                    "no_bnodes": not contains_bnode,
                    "number_of_conditions": len(classes) + len(owl_restriction) + (1 if one_ofs else 0) + bnodes,
                    "containing_elements": containing_elements.union(classes),
                    "predicate": "implicit intersection",
                }
                if classes:
                    cls_restrictions[relation]["classes"] = classes
                if owl_restriction:
                    cls_restrictions[relation]["restrictions"] = owl_restriction
                    cls_restrictions[relation]["classes_in_restrictions"] = containing_restriction_classes,
                if one_ofs:
                    cls_restrictions[relation]["oneOf"] = one_ofs
                
        if cls_restrictions:
            all_restrictions[cls] = cls_restrictions
             
    return all_restrictions

## 4. Optimierung Tabellen Anzahl

### R48/R49 - Reduzeirung Tabellen Anzahl 

In [1015]:
def get_class_usage_map(graph, context_dict):
    """_summary_

    :param graph: _description_
    :type graph: _type_
    :param context_dict: _description_
    :type context_dict: _type_
    """
    g = graph
    class_usage_map = {}
    
    all_classes = context_dict["all_classes"]
    all_properties = context_dict["all_properties"]
    domain_range_map = context_dict["domain_range_map"]
    super_class_of = context_dict["super_class_of_dist"]
    sub_class_of = context_dict["sub_class_of_dist"]
    equivalent_map = context_dict["equivalent_class_map"]
    equivalent_group = context_dict["equivalent_class_group"]
    
    def get_cls_usage(cls):
        domain_props = set()
        range_props = set()
        for prop in all_properties:
            domain_range_map_prop = domain_range_map.get(prop)
            prop_domain = equivalent_map.get(domain_range_map_prop.get("domain")) or domain_range_map_prop.get("domain", None)
            prop_range = equivalent_map.get(domain_range_map_prop.get("range")) or domain_range_map_prop.get("range", None) 
            if prop_domain and cls == prop_domain:
                domain_props.add(prop)
            if prop_range and cls == prop_range:
                range_props.add(prop)
        number_domain = len(domain_props)
        number_range = len(range_props)
        
        number_instances = len(set(g.subjects(RDF.type, cls)))
        
        super_class_of_cls = super_class_of.get(cls, {}).keys()
        sub_class_of_cls = sub_class_of.get(cls, {}).keys()
        number_sub_classes = len(super_class_of_cls)
        number_super_classes = len(sub_class_of_cls)
        
        cls_usage_map = {
            "domain": number_domain if number_domain else False,
            "range": number_range if number_range else False,
            "instances": number_instances if number_instances else False,
            "sub_classes": number_sub_classes if number_sub_classes else False,
            "super_classes": number_super_classes if number_super_classes else False,
        }
        domain_and_range = bool(number_domain or number_range)
        
        return cls_usage_map, domain_and_range
    
    for cls in all_classes:
        sub_classes_used = any(
            get_cls_usage(sub_class)[1]
            for sub_class in super_class_of.get(cls, set())
        )
        super_classes_used = any(
            get_cls_usage(super_class)[1]
            for super_class in sub_class_of.get(cls, set())
        )
        equivalent_classes_used = any(
            get_cls_usage(equivalent_cls)[1]
            for equivalent_cls in equivalent_group.get(cls, set())
        ) 
                
        cls_usage_map, domain_and_range = get_cls_usage(cls)
        cls_usage_map["sub_classes_in_use"] = sub_classes_used
        cls_usage_map["super_classes_in_use"] = super_classes_used
        cls_usage_map["equivalent_classes_in_use"] = equivalent_classes_used
        class_usage_map[cls] = cls_usage_map
        
    return class_usage_map

In [1016]:
def get_property_usage_map(graph, context_dict):
    """_summary_

    :param graph: _description_
    :type graph: _type_
    :param context_dict: _description_
    :type context_dict: _type_
    """
    g = graph
    property_usage_map = {}
    
    all_properties = context_dict["all_properties"]
    domain_range_map = context_dict["domain_range_map"]
    super_property_of = context_dict["super_property_of_dist"]
    sub_property_of = context_dict["sub_property_of_dist"]
    equivalent_map = context_dict["equivalent_property_map"]
    equivalen_group = context_dict["equivalent_property_group"]
    
    def get_prop_usage(prop):
        domain_range_map_prop = domain_range_map.get(prop)
        prop_domain = equivalent_map.get(domain_range_map_prop.get("domain")) or domain_range_map_prop.get("domain", None)
        prop_range = equivalent_map.get(domain_range_map_prop.get("range")) or domain_range_map_prop.get("range", None) 
        
        number_triple = len(set(g.subject_objects(prop)))
        
        super_property_of_cls = super_property_of.get(prop, {}).keys()
        sub_property_of_cls = sub_property_of.get(prop, {}).keys()
        number_sub_properties = len(super_property_of_cls)
        number_super_properties = len(sub_property_of_cls)
        
        prop_usage_map = {
            "triples": number_triple if number_triple else False,
            "sub_properties": number_sub_properties if number_sub_properties else False,
            "super_properties": number_super_properties if number_super_properties else False,
        }
        domain_and_range = bool(prop_domain or prop_range)
        
        return prop_usage_map, domain_and_range
    
    for prop in all_properties:
        sub_properties_used = any(
            get_prop_usage(sub_property)[1]
            for sub_property in super_property_of.get(prop, set())
        )
        super_properties_used = any(
            get_prop_usage(super_property)[1]
            for super_property  in sub_property_of.get(prop, set())
        )
        
        equi_properties_used = any(
            get_prop_usage(equi_property)[1]
            for equi_property in equivalen_group.get(prop, set())
        )
        
        prop_usage_map, domain_and_range = get_prop_usage(prop)
        prop_usage_map["sub_properties_in_use"] = sub_properties_used
        prop_usage_map["super_properties_in_use"] = super_properties_used
        prop_usage_map["equivalent_properties_in_use"] = equi_properties_used
        property_usage_map[prop] = prop_usage_map
        
    return property_usage_map

## Context Dict

In [1017]:
def build_context_dict(graph, preferred_keywords, schema):
    """
    Builds the central ontology context dictionary used across the Shredder pipeline.

    This function orchestrates the extraction and inference of all structural, 
    semantic, and logical components from the ontology — including classes, 
    properties, individuals, equivalences, domains/ranges, disjointness, and annotations.

    The resulting 'context_dict' serves as the unified knowledge base for 
    all subsequent transformation and SQL generation modules.

    :param graph (rdflib.Graph): RDF graph containing the ontology.
    :return (dict): A comprehensive ontology metadata structure, including:
        {
            "all_classes": [...],
            "equivalent_class_group": {...},
            "inverse_property_group": {...},
            "domain_range_map": {...},
            "functional_types": {...},
            "classes_disjoint_with": {...},
            ...
        }
    """
    g = graph
    context_dict = {}
    
    all_classes = get_all_classes(g)
    all_properties, all_annotation_properties = get_all_properties(g)
    all_individuals = get_all_individuals(g, all_classes, all_properties)
    elements_by_group = get_elements_by_group(g, all_classes, all_properties, all_individuals)
    
    functional_types_solo, used_as_functional_solo = get_functional_properties(g, all_properties)
    inverse_functional_types_solo, used_as_inverse_functional_solo = get_inverse_functional_properties(g, all_properties)
    
    custom_datatypes = extract_custom_datatypes(g)
    
    unique_names_by_group = generate_unique_names(g, elements_by_group, preferred_keywords, schema)
    primary_key_type = get_primary_key(g)
    
    sub_class_equivalent_group = compute_inferred_equivalents_and_sub_classes(g, unique_names_by_group, all_classes)
    equivalent_class_map, equivalent_class_group = build_equivalent_class_map(g, unique_names_by_group, sub_class_equivalent_group)
    super_class_of_dist, sub_class_of_dist, _  = build_sub_class_map(g, equivalent_class_group, all_classes)

    inverse_property_group, inverse_prop_equivalents_group, sub_prop_equivalent_group = compute_inferred_equivalents_sub_property_and_inverses(g, unique_names_by_group, all_properties)
    equivalent_property_map, equivalent_property_group = build_equivalent_property_map(g, unique_names_by_group, inverse_prop_equivalents_group, sub_prop_equivalent_group)
    super_property_of_dist, sub_property_of_dist, _  = build_sub_property_map(g, equivalent_property_group, all_properties)
    
    negative_property_assertion = get_negative_property_assertion(g, all_properties, equivalent_property_map)
    
    one_of_class_map = build_one_of_map(g, all_classes)
    complex_class_map = get_complex_class_map(g, all_classes)
    
    property_chain_map = get_property_chains_map(g)
    
    context_dict = {
        # general
        "all_classes": all_classes,
        "all_properties": all_properties,
        "all_annotation_properties": all_annotation_properties,
        "all_individuals": all_individuals,
        "elements_by_group": elements_by_group,
        "custom_datatypes": custom_datatypes,
        "unique_names_by_group": unique_names_by_group,
        "primary_key_type": primary_key_type,
        # class
        "equivalent_class_map": equivalent_class_map,
        "equivalent_class_group": equivalent_class_group,
        "super_class_of_dist": super_class_of_dist,
        "sub_class_of_dist": sub_class_of_dist,
        "one_of_class_map": one_of_class_map,
        "complex_class_map": complex_class_map,
        # property
        "inverse_property_group": inverse_property_group,
        "inverse_equivalents_group": inverse_prop_equivalents_group,
        "equivalent_property_map": equivalent_property_map,
        "equivalent_property_group": equivalent_property_group,
        "super_property_of_dist": super_property_of_dist,
        "sub_property_of_dist": sub_property_of_dist,
        "property_chain_axiom": property_chain_map,
        # other
        "negative_property_assertion": negative_property_assertion,
        # (inverse) functional
        "functional_types_solo": functional_types_solo,
        "used_as_functional_solo": used_as_functional_solo,
        "inverse_functional_types_solo": inverse_functional_types_solo,
        "used_as_inverse_functional_solo": used_as_inverse_functional_solo,
    }

    prop_rdfs_types, prop_functional, prop_inverse_functional, functional_types, inverse_functional_type = get_rdf_types_and_inferred_types(g, context_dict)
    context_dict["rdf_types"] = prop_rdfs_types
    context_dict["used_as_functional"] = prop_functional
    context_dict["used_as_inverse_functional"] = prop_inverse_functional
    context_dict["functional_types"] = functional_types
    context_dict["inverse_functional_types"] = inverse_functional_type

    inverse_representative_map = build_inverse_property_map(g, context_dict)
    context_dict["inverse_property_map"] = inverse_representative_map

    domain_range_map, common_domain_range_equi, common_domain_range_inv = infer_domain_range_fixpoint(g, context_dict)
    context_dict["domain_range_map"] = domain_range_map
    context_dict["common_domain_range_equivalents"] = common_domain_range_equi
    context_dict["common_domain_range_inverses"] = common_domain_range_inv
    
    all_annotations = get_annotation_map(g, context_dict)
    context_dict["annotation_map"] = all_annotations
       
    class_disjoint_with, class_disjoint_pairs = build_class_disjoint_map(g, context_dict)
    context_dict["classes_disjoint_with"] = class_disjoint_with
    context_dict["classes_disjoint_pairs"] = class_disjoint_pairs
    
    prop_disjoint_with, prop_disjoint_pairs = build_property_disjoint_map(g, context_dict)
    context_dict["properties_disjoint_with"] = prop_disjoint_with
    context_dict["properties_disjoint_pairs"] = prop_disjoint_pairs
    
    class_usage_map = get_class_usage_map(g, context_dict)
    context_dict["class_usage_map"] = class_usage_map
    
    property_usage_map = get_property_usage_map(g, context_dict)
    context_dict["property_usage_map"] = property_usage_map
    
    return context_dict

## Property Infos

In [1018]:
def get_property_info(graph, prop, context_dict):
    """
    Collects all semantic and structural metadata for a given property.

    Extracts information from the RDF graph and the Shredder context, including:
    RDF type, domain, range, equivalences, inverses, SQL-safe name, 
    and functional/inverse functional status.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param prop (rdflib.URIRef): URI of the property to analyze.
    :param context_dict (dict): Shared ontology context with precomputed mappings.
    :return (dict): Property metadata record:
        {
            "rdf:type": URIRef | None,
            "domain": URIRef | None,
            "range": URIRef | None,
            "sqlName": str,
            "functional": bool,
            "inverseFunctional": bool,
            ...
        }
    """
    g = graph

    unique_names = context_dict["unique_names_by_group"]
    if prop not in PROPERTY_INFO_MAP: # Lazy Loading mit Lazy Dictionary

        used_in_class_expression = extract_property_restrictions(g)

        # used_in_intersection = set()
        # for _, _, intersection_node in g.triples((None, OWL.intersectionOf, None)):
        #     if (intersection_node, RDF.first, None) in g:
        #         members = Collection(g, intersection_node)
        #         for member in members:
        #             if (member, RDF.type, OWL.Restriction) in g:
        #                 property = g.value(member, OWL.onProperty)
        #                 if property:
        #                     used_in_intersection.add(property)


    #Grundlegende Infos
        domain_range_map = context_dict["domain_range_map"].get(prop)
        domain = context_dict["equivalent_class_map"].get(domain_range_map.get("domain")) or domain_range_map.get("domain", None)
        range_ = context_dict["equivalent_class_map"].get(domain_range_map.get("range")) or domain_range_map.get("range", None) 

        
        sql_name = get_sql_name(prop, "property", unique_names)
        property_key = get_sql_name(prop, "propertyKey", unique_names)

        annotations = context_dict["annotation_map"]["property"][prop]
        
        type_ = None
        all_types = context_dict["rdf_types"].get(prop)
        if isinstance(all_types, str):
            type_ = "Error"
        elif OWL.DatatypeProperty in all_types:
            type_ = OWL.DatatypeProperty
        elif OWL.ObjectProperty in all_types:
            type_ = OWL.ObjectProperty
        
        if OWL.SymmetricProperty in all_types:
            if domain != range_:
                domain = nearest_common_super_sub_element(g, [domain, range_], RDFS.subClassOf, "super")
                range_ = domain
            
        # zählt wie oft die Property in einer Klasse auftritt
        usage_count_by_class = {}  
        for s, _, _ in g.triples((None, prop, None)):
            for cls in set(g.objects(s, RDF.type)):
                if cls == OWL.NamedIndividual:
                    continue
                elif cls not in usage_count_by_class:
                    usage_count_by_class[cls] = 1
                else:
                    usage_count_by_class[cls] += 1

        distinct_subjects = set(s for s, _, _ in g.triples((None, prop, None)))
            # gibt alle unterschiedlichen Subjekte der Property zurück  
        distinct_objects = set(o for _, _, o in g.triples((None, prop, None)))
            # gibt alle unterschiedlichen Objekte der Propertie zurück 
        
        has_datatypes = extract_property_datatypes(g, prop, used_in_class_expression, context_dict)
        subject_datatype = get_primary_key_datatype(g, prop, context_dict)
        recommended_datatypes = recommended_datatype(g, prop, context_dict)
        
        functional_types = context_dict["functional_types"].get(prop)
        functional = context_dict["used_as_functional"].get(prop)
        
        inverse_functional_types = context_dict["inverse_functional_types"].get(prop)
        inverse_functional = context_dict["used_as_inverse_functional"].get(prop)
        
        
    # Struktur & Hierarchie
        prop_chain_map = context_dict["property_chain_axiom"].get(prop, set())
        
    # Eigenschaften
        one_of = g.value(prop, OWL.oneOf)
        
        has_keys = []
        for s, _, o in g.triples((None, OWL.hasKey, None)):
            keys = list(Collection(g, o))
            if prop in keys:
                has_keys.append({
                "inKeyForClass": s,
                "key": keys
                })

        symmetric_property = (True if OWL.SymmetricProperty in all_types else False)
        asymmetric_property = (True if OWL.AsymmetricProperty in all_types else False)
        reflexive_property = (True if OWL.ReflexiveProperty in all_types else False)
        irreflexive_property = (True if OWL.IrreflexiveProperty in all_types else False)
        transitive_property = (True if OWL.TransitiveProperty in all_types else False)
        deprecated_property = (True if any(x in all_types for x in (OWL.DeprecatedProperty, OWL.deprecated)) else False)
        
    # Restrictions
        has_predicates = set()
        if prop in used_in_class_expression:
            for predicates in used_in_class_expression[prop]:
                has_predicates.add(predicates["predicate"])
        has_container = set()
        if prop in used_in_class_expression:
            for container in used_in_class_expression[prop]:
                has_container.add(container["logicalContainer"])
        has_location = set()
        if prop in used_in_class_expression:
            for location in used_in_class_expression[prop]:
                has_location.add(location["restrictionLocation"])

    # äquivalent Properties
        equivalent_map = context_dict["equivalent_property_map"].get(prop, None)
        equivalent_group = context_dict["equivalent_property_group"].get(prop, None)
        equivalent_group_name = get_sql_name(equivalent_map, "property", unique_names)
    
    # inverse Properties
        inverse_group = context_dict["inverse_property_group"].get(prop, {})
        inverse_map = context_dict["inverse_property_map"].get(prop, {})
    
    # disjointe Properties    
        disjoint_with = context_dict["properties_disjoint_with"]
        disjoint_pairs = context_dict["properties_disjoint_pairs"]
        
        disjoint_with_groups = []
        if disjoint_with.get(prop):
            disjoint_with_groups.extend([list(dp) for dp in disjoint_pairs if prop in dp])
                    
    #CHECK Constraints
        check_contsraints = {}
        
        forbidden_pairs = context_dict["negative_property_assertion"]
        if equivalent_group_name:
            forbidden_pairs_prop = forbidden_pairs.get(equivalent_group_name)
        else:
            forbidden_pairs_prop = forbidden_pairs.get(prop)    
        if forbidden_pairs_prop:
            check_contsraints["negativePropertyAssertion"] = forbidden_pairs_prop
        
        if not functional and OWL.ReflexiveProperty in all_types:
            check_contsraints["reflexiveProperty"] = True
        
        if irreflexive_property:
            check_contsraints["irreflexiveProperty"] = True
        
        if asymmetric_property:
            check_contsraints["asymmetricProperty"] = True
    
    
    # implemented as
        property_usage = context_dict["property_usage_map"][prop]
        number_triples = property_usage["triples"]
        number_sub_properties = property_usage["sub_properties"]
        number_super_properties = property_usage["super_properties"]
        sub_properties_in_use = property_usage["sub_properties_in_use"]
        super_properties_in_use = property_usage["super_properties_in_use"]
        equi_property_in_use = property_usage["equivalent_properties_in_use"]
    
        implemented_as = None
        if type_ == "Error":
            implemented_as = "not at all: error"
        elif deprecated_property:
            implemented_as = "not at all: deprecated"
        # elif equi_property_in_use and prop != equivalent_map:
        #     implemented_as = "multivalue relation (equivalent)"
        # elif not domain and not range_ and (number_sub_properties == 0 or not sub_properties_in_use):
        #     if all(x == 0 for x in (number_triples, number_super_properties)):
        #         implemented_as = "not at all: not in use"
        #     elif number_super_properties and super_properties_in_use:
        #         implemented_as = "in super property"
        #     elif not super_properties_in_use:
        #         implemented_as = "not at all: not in use at and no superproperties in use"
        if not implemented_as:
            if inverse_map:
                if inverse_map[0] == prop:
                    implemented_as = "inverse relation (representative)"
                elif inverse_map[0] == equivalent_map:
                    implemented_as = "inverse relation (equivalent)"
                elif inverse_map[1] == prop:
                    implemented_as = "inverse relation (inverse side)"
                else:
                    implemented_as = "inverse relation (equivalent inverse side)"
            elif functional:
                if equivalent_map:
                    if equivalent_map == prop:
                        implemented_as = "functional relation"
                    else:
                        implemented_as = "functional relation (equivalent)"
                else:
                    implemented_as = "functional relation"
            else:
                if equivalent_map:
                    if equivalent_map == prop:
                        implemented_as = "multivalue relation"
                    else:
                        implemented_as = "multivalue relation (equivalent)"
                else:
                    implemented_as = "multivalue relation"


        PROPERTY_INFO_MAP[prop] = {
        # Grundlegende Infos
            "rdf:type": type_ if type_ else None,
            "rdfs:domain": domain if domain else None,
            "rdfs:range": range_ if range_ else None,          
            "sqlName": sql_name,
            "propertyKey": property_key,
            "uri": prop,
            "distinctSubjectCount": len(distinct_subjects),
            "distinctObjectCount": len(distinct_objects), 
            "possibleDataTypes": has_datatypes[prop], 
            "recommendedDataType": recommended_datatypes ,
            "isSingleValued": functional,
            "singleValueTypes": functional_types,
            "isMultiValued": not functional,
            "isInverseFunctional": inverse_functional,
            "inversefunctionalTypes": inverse_functional_types,
            "subject_datatype": subject_datatype,        
            "usedByClasses": usage_count_by_class if usage_count_by_class else None,  
        # Struktur & Hierarchie
            "subPropertyOf": context_dict["sub_property_of_dist"].get(prop, {}).keys() or None,
            "superPropertyOf": context_dict["super_property_of_dist"].get(prop, {}).keys() or None,
            "disjointWithGroups": disjoint_with_groups if disjoint_with_groups else None,
            "propertyChainAxiom": prop_chain_map if prop_chain_map else None,
        # Eigenschaften
            "inverseGroup": inverse_group if inverse_group else False,
            "inverseMap": inverse_map if inverse_map else None,
            "symmetricProperty": symmetric_property,
            "asymmetricProperty": asymmetric_property,
            "reflexiveProperty": reflexive_property,
            "irreflexiveProperty": irreflexive_property,
            "transitiveProperty": transitive_property,
            "deprecated": deprecated_property,
            "owl:oneOf": list(Collection(g, one_of)) if one_of else None,
            "owl:hasKey": has_keys if has_keys else None,
        # Restrictions
            "usedInClassExpression": used_in_class_expression[prop] if prop in used_in_class_expression else None, 
            "usedInClassExpressionCount": len(used_in_class_expression[prop]) if prop in  used_in_class_expression else 0,
            "usedInEqivalentClassDef": OWL.equivalentClass in has_location,
            "usedInSubClassOfDef": RDFS.subClassOf in has_location,
            "owl:hasValue": OWL.hasValue in has_predicates,
            "owl:someValuesFrom": OWL.someValuesFrom in has_predicates,
            "owl:allValuesFrom": OWL.allValuesFrom in has_predicates,
            "owl:hasSelf": OWL.hasSelf in has_predicates,
            "owl:minCardinality": OWL.minCardinality in has_predicates,
            "owl:maxCardinality": OWL.maxCardinality in has_predicates,
            "owl:cardinality": OWL.cardinality in has_predicates,
            "owl:minQualifiedCardinality": OWL.minQualifiedCardinality in has_predicates,
            "owl:maxQualifiedCardinality": OWL.maxQualifiedCardinality in has_predicates,
            "owl:qualifiedCardinality": OWL.qualifiedCardinality in has_predicates,
            "owl:intersectionOf": OWL.intersectionOf in has_container,
            "owl:unionOf": OWL.unionOf in has_container,
            "owl:complementOf": OWL.complementOf in has_container,
            "checkConstraints": check_contsraints if check_contsraints else None,
        # Annotations
            "annotations": annotations,
        # Äquivalentz Properties
            "equivalentPropertyRepresentative": equivalent_group_name if equivalent_group_name else None,
            "equivalentGroup": equivalent_group if equivalent_group else None,
            
            "implementedAs": implemented_as
        }   

    return PROPERTY_INFO_MAP[prop]

In [1019]:
def get_class_property_info(g, cls, context_dict):
    """
    Returns all properties that use a given class as their domain or range.

    This is useful for analyzing class-property relationships, 
    especially for generating table schemas or verifying mapping consistency.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param cls (rdflib.URIRef): URI of the class to inspect.
    :param context_dict (dict): Shared ontology context with domain/range information.
    :return (dict): 
        {
            classURI: {
                "asDomainOf": [propertyURI, ...],
                "asRangeOf": [propertyURI, ...]
            }
        }
    """    
    all_properties = context_dict["all_properties"]

    if cls not in CLASS_PROPERTY_MAP:
        domain_props = set()
        range_props = set()
        for prop in all_properties:
            
            prop_info = get_property_info(g, prop, context_dict)
            prop_domain = prop_info["rdfs:domain"]
            prop_range = prop_info["rdfs:range"]
            
            if prop_domain and cls == prop_domain:
                domain_props.add(prop)
            if prop_range and cls == prop_range:
                range_props.add(prop)
        
        
        CLASS_PROPERTY_MAP[cls] = {
            "asDomainOf": domain_props,
            "asRangeOf": range_props
        }
    return CLASS_PROPERTY_MAP[cls]

## Klassen Infos

Bestimmt alle Klassen und zugehörige Infos

In [1020]:
def get_class_info(graph, cls, context_dict):
    """
    Collects all semantic and structural metadata for a given class.

    Includes RDF type, SQL name, superclass/subclass relationships, 
    disjointness, and complexity (e.g. restriction-based or enumerated).

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param cls (rdflib.URIRef): URI of the class to analyze.
    :param context_dict (dict): Shared ontology context.
    :return (dict): 
        {
            "rdf:type": URIRef | None,
            "sqlName": str,
            "simpleClass": bool,
            "superClasses": [...],
            "restrictions": {...},
            ...
        }
    """
    g = graph
    unique_names = context_dict["unique_names_by_group"]

    if cls not in CLASS_INFO_MAP:

        # used_in_class_axioms = collect_class_usages(g)

        reference_props = [OWL.equivalentClass, RDFS.subClassOf, OWL.complementOf, OWL.disjointUnionOf]

    # Grundlegende Infos
        sql_name = get_sql_name(cls, "class", context_dict)
        class_key = get_sql_name(cls, "classKey", context_dict)

        type2 = set(g.objects(cls, RDF.type))
        if not type2 and any(g.value(cls, p) for p in reference_props):
            type2 = set(g.value(g.value(cls, p), RDF.type) for p in reference_props if g.value(cls, p))     
        type_ = (next(iter(type2)) if len(type2) == 1 else type2) 
        if (cls, OWL.deprecated, Literal(True)) in g:
            type2.add(OWL.deprecated)
        
        
        primary_key = context_dict["primary_key_type"]
        
        reference_count = sum(    
            1 for s, p, o in g.triples((None, None, cls))
            if isinstance(cls, URIRef)
            )
          
        # complement_of_subject, complement_restriction = extract_class_axioms(g, cls, OWL.complementOf)
        # complement_of_object = set(g.subjects(OWL.complementOf, cls))
        # complement_of = complement_of_subject.union(complement_of_object)
    # Struktur
        has_key = g.objects(cls, OWL.hasKey)
        has_key_group = [list(Collection(g, key_props)) for key_props in has_key]
    # als Domain oder Range
        as_domain_of = get_class_property_info(g, cls, context_dict)["asDomainOf"]
        as_range_of = get_class_property_info(g, cls, context_dict)["asRangeOf"]
    # Annotationen
        annotations = context_dict["annotation_map"]["class"][cls]
    # äquivalentz Klassen
        equivalent_map = context_dict["equivalent_class_map"].get(cls, set())
        equivalent_group = context_dict["equivalent_class_group"].get(cls, set())
        equivalent_group_name = get_sql_name(equivalent_map, "class", context_dict)
        
        sub_class_of = context_dict["sub_class_of_dist"].get(cls, {})
        super_class_of = context_dict["super_class_of_dist"].get(cls, {}).keys()
        
        initial_sub_refcount = None
        if super_class_of:
            initial_sub_refcount = {
                instance: sum(1 for t in g.objects(instance, RDF.type) if t in super_class_of)
                for instance in g.subjects(RDF.type, cls)
            }


        disjoint_with = context_dict["classes_disjoint_with"]
        disjoint_pairs = context_dict["classes_disjoint_pairs"]
        disjoint_with_groups = []
        if disjoint_with.get(cls):
            disjoint_with_groups.extend([list(dp) for dp in disjoint_pairs if cls in dp])
            
            
        one_of_map = context_dict["one_of_class_map"].get(cls, set())
        
        complex_class = context_dict["complex_class_map"].get(cls, set())
        
        class_usage = context_dict["class_usage_map"][cls]
        number_domain = class_usage["domain"]
        number_range = class_usage["range"]
        number_instances = class_usage["instances"]
        number_sub_classes = class_usage["sub_classes"]
        number_super_classes = class_usage["super_classes"]
        sub_classes_in_use = class_usage["sub_classes_in_use"]
        super_classes_in_use = class_usage["super_classes_in_use"]
        equi_classes_in_use = class_usage["equivalent_classes_in_use"]
        
        deprecated = any(x in type2 for x in (OWL.DeprecatedClass, OWL.deprecated))
        
        implemented_as = None
        if deprecated:
            implemented_as = "not at all: deprecated"
        # elif equi_classes_in_use and cls != equivalent_map:
        #     implemented_as = "table (equivalent)"
        # elif all(x == 0 for x in (number_domain, number_range)):
        #     if number_sub_classes == 0 or not sub_classes_in_use:
        #         if all(x == 0 for x in (number_instances, number_super_classes)):
        #             implemented_as = "not at all: not in use"
        #         elif number_super_classes and super_classes_in_use:
        #             implemented_as = "in super table"
        #         elif not super_classes_in_use:
        #             implemented_as = "not at all: not in use and no superclasses in use"
        #         else:
        #             if equivalent_map and cls != equivalent_map:
        #                 implemented_as = "table (equivalent)"
        #             else:
        #                 implemented_as = "table"
        #     else:
        #         implemented_as = "table"
        elif equivalent_map and not implemented_as:
            if cls != equivalent_map:
                implemented_as = "table (equivalent)"
            else:
                implemented_as = "table"
        else: 
            implemented_as = "table"
        
        # CHECK Constraints
        check_constraints = {}
        if one_of_map:
            check_constraints["oneOf"] = one_of_map
        
        
        CLASS_INFO_MAP[cls] = {
        # Grundlegende Infos
            "rdf:type": type_ if type_ else None,
            "sqlName": sql_name,
            "classKey": class_key, 
            "simpleClass": True if not complex_class else False,
            "deprecated": True if any(x in type2 for x in (OWL.DeprecatedClass, OWL.deprecated)) else False,
            "complexClass": complex_class,
            "primaryKeyType": primary_key,
            "owl:DeprecatedClass": OWL.DeprecatedClass in set(g.objects(cls, RDF.type)),
            "usedAsType": (None, RDF.type, cls) in g,
            "totalReferenceCount": sum(1 for _ in g.triples((None, None, cls))), 
            "axiomCount": sum(1 for p in reference_props if (cls, p, None) in g),
            "sub_refcount": initial_sub_refcount if initial_sub_refcount else None,
        # Hierarchie 
            "rdfs:subClassOf": sub_class_of or None,
            "superClassOf": super_class_of or None,
        # Struktur
            "disjointWithGroups": disjoint_with_groups if disjoint_with_groups else None,
            # "owl:disjointUnionOf": disjoint_union if disjoint_union else None,
            # "owl:complementOf": [c for c in complement_of] if complement_of else None,
            "owl:hasKey": has_key_group if has_key_group else None,
            "owl:oneOf": one_of_map if one_of_map else None,
            "deprecated": deprecated,
        # Definiert durch in Restrictions
            # "complementOfRestriction": complement_restriction if complement_restriction else None,
            "checkConstraints": check_constraints if check_constraints else None,
        # als Domain oder Range
            "asDomainOf": [c for c in as_domain_of] if as_domain_of else None,
            "asRangeOf": [c for c in as_range_of] if as_range_of else None, 
        # Annotationen
            "annotations": annotations,
        # Äquivalenzklassen:
            "equivalentClassRepresentative": equivalent_group_name if equivalent_group_name else None,
            "equivalentGroup": equivalent_group if equivalent_group else None,
        # optimization
            "implementedAs": implemented_as
        }

    return CLASS_INFO_MAP[cls]

## SQL Generierung

### Hilfsfunktionen

In [1021]:
def annotation_sql(info_dict, annotation_key, element, comment_list):
    """
    Bereitet ein Annotation-Dictionary für die Übersetzung in SQL-Kommentare vor.

    Die Funktion extrahiert Annotationen_Dictionaries aus dem Info-Dictionary und 
    ergänzt die Kommentar-Liste um formatierte Strings.

    :param info_dict: dict - Informations-Dictionary einer Klasse oder Property
    :param annotation_key: str - Schlüssel zum Annotation-Teil (z. B. "equivalentClassAnnotations")
    :param element: rdflib.URIRef - URI der betroffenen Klasse oder Property
    :param comment_list: List[str] - bereits vorhandene Kommentarzeilen
    :return: List[str] - erweiterte Kommentarzeilen inkl. Annotationen
    """
    equivalent_annotation_keys = ["equivalentClassAnnotations", "equivalentPropertyAnnotations"]
    annotations = (
        info_dict[annotation_key] if annotation_key in equivalent_annotation_keys
        else info_dict[annotation_key][element]
    )
    
    for key, values in annotations.items():
        if values in [True, False]:
            continue
        elif key != "customAnnotations" and values is not None:
            for val in values:
                comment_list.append(f"> {key}: {val}")
        elif key == "customAnnotations" and values is not None:
            for costum_anno, val in values.items():
                for v in val:
                    comment_list.append(f"> {remove_prefix(costum_anno)}: {v}")
        
    return comment_list


### Trigger

#### Symmetric Proeprty

In [1022]:
def symmetric_property_trigger_functions(schema = "public", sys_schema = "public_sys"):
    """
    Generates PL/pgSQL trigger functions to enforce symmetric properties in a PostgreSQL database.
    """
    
    insert_function_definition = f"""
    CREATE OR REPLACE FUNCTION {schema}.symmetric_property_insert_trigger_function()
    RETURNS TRIGGER AS $$
    DECLARE
        subject_col TEXT;
        object_col  TEXT;
        new_subject_value TEXT;
        new_object_value  TEXT;
    BEGIN
        -- Schutz gegen Endlos-Trigger
        IF current_setting('{schema}.symmetric_running', true) = 'on' THEN
            RETURN NEW;
        END IF;
        PERFORM set_config('{schema}.symmetric_running', 'on', true);

        SELECT subject_column, object_column
            INTO subject_col, object_col
        FROM {sys_schema}.all_properties_sys
        WHERE sql_name = TG_TABLE_NAME;
        
        new_subject_value := to_jsonb(NEW)->>subject_col;
        new_object_value  := to_jsonb(NEW)->>object_col;


        EXECUTE format(
            'INSERT INTO {schema}.%I (%I, %I)
            VALUES ($1, $2)
            ON CONFLICT DO NOTHING',
            TG_TABLE_NAME, subject_col, object_col
        ) USING new_object_value, new_subject_value;
    
        RAISE NOTICE 'The entry (%s, %s) has also been added as a result of symmetry.',
            new_object_value, new_subject_value;

        -- Schutzflag zurücksetzen
        PERFORM set_config('{schema}.symmetric_running', 'off', true);

        RETURN NEW;
    END;
    $$ LANGUAGE plpgsql;
    """
    
    delete_function_definition = f"""
    CREATE OR REPLACE FUNCTION {schema}.symmetric_property_delete_trigger_function()
    RETURNS TRIGGER AS $$
    DECLARE
        subject_col TEXT;
        object_col  TEXT;

        old_subject_value TEXT;
        old_object_value TEXT;
    BEGIN
        
        IF current_setting('{schema}.symmetric_running', true) = 'on' THEN
            RETURN OLD;
        END IF;
        PERFORM set_config('{schema}.symmetric_running', 'on', true);
        
        SELECT subject_column, object_column
            INTO subject_col, object_col
        FROM {sys_schema}.all_properties_sys
        WHERE sql_name = TG_TABLE_NAME;

        old_subject_value := to_jsonb(OLD)->>subject_col;
        old_object_value := to_jsonb(OLD)->>object_col;
        

        EXECUTE format(
            'DELETE FROM {schema}.%I
            WHERE %I = $1 AND %I = $2',
            TG_TABLE_NAME, subject_col, object_col
        ) USING old_object_value, old_subject_value;

        RAISE NOTICE 'The entry (%s, %s) has also been deleted as a result of symmetry.',
                old_object_value, old_subject_value;


        PERFORM set_config('{schema}.symmetric_running', 'off', true);
        RETURN OLD;
    END;
    $$ LANGUAGE plpgsql;
    """
    
    update_function_definition = f"""
    CREATE OR REPLACE FUNCTION {schema}.symmetric_property_update_trigger_function()
    RETURNS TRIGGER AS $$
    DECLARE
        subject_col TEXT;
        object_col  TEXT;

        old_subject_value TEXT;
        old_object_value TEXT;

        new_subject_value TEXT;
        new_object_value TEXT;

        row_exists BOOLEAN;
    BEGIN

        IF current_setting('{schema}.symmetric_running', true) = 'on' THEN
            RETURN NEW;
        END IF;
        PERFORM set_config('{schema}.symmetric_running', 'on', true);

        SELECT subject_column, object_column
            INTO subject_col, object_col
        FROM {sys_schema}.all_properties_sys
        WHERE sql_name = TG_TABLE_NAME;

        old_subject_value := to_jsonb(OLD)->>subject_col;
        old_object_value := to_jsonb(OLD)->>object_col;

        new_subject_value := to_jsonb(NEW)->>subject_col;
        new_object_value := to_jsonb(NEW)->>object_col;
        
        
        IF new_subject_value IS NOT DISTINCT FROM old_subject_value
            AND new_object_value IS NOT DISTINCT FROM old_object_value 
        THEN
            PERFORM set_config('{schema}.symmetric_running', 'off', true);
            RETURN NEW;
        END IF;

        EXECUTE format(
            'SELECT EXISTS (
                SELECT 1 FROM {schema}.%I
                WHERE %I = $1 AND %I = $2
            )', TG_TABLE_NAME, subject_col, object_col
        ) INTO row_exists
        USING new_subject_value, new_object_value;

        IF NOT row_exists THEN
            EXECUTE format(
            'DELETE FROM {schema}.%I 
                WHERE %I = $1 AND %I = $2',
                TG_TABLE_NAME, subject_col, object_col
            ) USING old_object_value, old_subject_value;
            
            RAISE NOTICE 'The entry (%s, %s) has also been deleted as a result of symmetry.',
                    old_object_value, old_subject_value;
                
            EXECUTE format(
                'INSERT INTO {schema}.%I (%I, %I)
                VALUES ($1, $2)',
                TG_TABLE_NAME, subject_col, object_col
            ) USING new_object_value, new_subject_value;
            
            RAISE NOTICE 'The entry (%s, %s) has also been added as a result of symmetry.',
                new_object_value, new_subject_value;
        ELSE
            RAISE EXCEPTION 'The entry can not be updated because (%s, %s) alrady exists.',
                new_subject_value, new_object_value ;
        END IF;

        PERFORM set_config('{schema}.symmetric_running', 'off', true);
        RETURN NEW;
    END;
    $$ LANGUAGE plpgsql;
    """
    
    
    
    
    return insert_function_definition + "\n" + delete_function_definition + "\n" + update_function_definition

In [1023]:
def trigger_definition_symmetric_property(graph, prop_mapping, context_dict):
    """
    Creates trigger definitions for symmetric properties.

    For each property declared as 'owl:SymmetricProperty', a trigger is generated
    to automatically insert the inverse subject-object pair whenever a new row
    is added to the corresponding property table. This ensures that for every
    (A, B) pair, the inverse (B, A) pair also exists.

    :param graph (rdflib.Graph): RDFLib graph of the ontology.
    :param prop_mapping (dict): Mapping of property URIs to SQL table names.
    :param context_dict (dict): Ontology context containing property metadata.
    :return (list[dict]): List of SQL trigger definitions enforcing symmetry.
    """
    g = graph
    trigger_definition = []
    
    for prop in prop_mapping:
        prop_info = get_property_info(g, prop, context_dict)
        mapping = prop_mapping[prop]
        if not prop_info["symmetricProperty"]:
            continue
        if not mapping["mapped"]:
            continue
        
        
        prop_name = prop_info["sqlName"]
        table_name = mapping.get("name")
        
        trigger_definition.append({
            "triggerName": f"trg_insert_symmetric_property_{prop_name}",
            "pointInTime": "AFTER",
            "event": "INSERT", 
            "target": table_name,
            "validity": "ROW",
            "function": "symmetric_property_insert_trigger_function()"
        })
        
        trigger_definition.append({
            "triggerName": f"trg_delete_symmetric_property_{prop_name}",
            "pointInTime": "BEFORE",
            "event": "DELETE", 
            "target": table_name,
            "validity": "ROW",
            "function": "symmetric_property_delete_trigger_function()"
        })
        
        trigger_definition.append({
            "triggerName": f"trg_update_symmetric_property_{prop_name}",
            "pointInTime": "BEFORE",
            "event": "UPDATE", 
            "target": table_name,
            "validity": "ROW",
            "function": "symmetric_property_update_trigger_function()"
        })

              
    return trigger_definition

#### Asymmetric Property

In [1024]:
def asymmetric_property_trigger_functions(schema = "public", sys_schema = "public_sys"):
    """
    Generates PL/pgSQL trigger functions to enforce asymmetric properties in a PostgreSQL database.
    """
    
    insert_function = f"""
    CREATE OR REPLACE FUNCTION {schema}.asymmetric_property_insert_trigger_function()
    RETURNS TRIGGER AS $$
    DECLARE
        subject_col TEXT;
        object_col  TEXT;
        new_subject_value TEXT;
        new_object_value  TEXT;

        row_exists BOOLEAN;
    BEGIN
        -- Schutz gegen Endlos-Trigger
        IF current_setting('ontology.asymmetric_running', true) = 'on' THEN
            RETURN NEW;
        END IF;
        PERFORM set_config('ontology.asymmetric_running', 'on', true);
        
        SELECT subject_column, object_column
            INTO subject_col, object_col
        FROM {sys_schema}.all_properties_sys
        WHERE sql_name = TG_TABLE_NAME;
        
        new_subject_value := to_jsonb(NEW)->>subject_col;
        new_object_value  := to_jsonb(NEW)->>object_col;

        -- Prüfen ob die Gegenrichtung existiert
        EXECUTE format(
            'SELECT EXISTS (
                SELECT 1 FROM {schema}.%I
                WHERE %I = $1 AND %I = $2
            )', TG_TABLE_NAME, subject_col, object_col
        ) INTO row_exists
        USING new_object_value, new_subject_value;  

        IF NOT row_exists THEN
            RETURN NEW;
        END IF;

        -- Schutzflag zurücksetzen
        PERFORM set_config('ontology.asymmetric_running', 'off', true);
        
        RAISE NOTICE 'The entry (%, %) cannot be added. The inverse entry (%, %) already exists in the table, and property % is declared asymmetric.',
                new_subject_value, new_object_value, new_object_value, new_subject_value, TG_TABLE_NAME;

        RETURN NEW;
    END;
    $$ LANGUAGE plpgsql;
    """
    
    update_function = f"""
    CREATE OR REPLACE FUNCTION {schema}.asymmetric_property_update_trigger_function()
    RETURNS TRIGGER AS $$
    DECLARE
        subject_col TEXT;
        object_col  TEXT;

        old_subject_value TEXT;
        old_object_value TEXT;

        new_subject_value TEXT;
        new_object_value TEXT;

        row_exists BOOLEAN;
    BEGIN
    
        IF current_setting('ontology.asymmetric_running', true) = 'on' THEN
            RETURN OLD;
        END IF;
        PERFORM set_config('ontology.asymmetric_running', 'on', true);
        
        SELECT subject_column, object_column
            INTO subject_col, object_col
        FROM {sys_schema}.all_properties_sys
        WHERE sql_name = TG_TABLE_NAME;

        old_subject_value := to_jsonb(OLD)->>subject_col;
        old_object_value := to_jsonb(OLD)->>object_col;

        new_subject_value := to_jsonb(NEW)->>subject_col;
        new_object_value := to_jsonb(NEW)->>object_col;
        
        IF new_subject_value IS NOT DISTINCT FROM old_subject_value
            AND new_object_value IS NOT DISTINCT FROM old_object_value 
        THEN
            PERFORM set_config('{schema}.symmetric_running', 'off', true);
            RETURN NEW;
        END IF;
        
        
        EXECUTE format(
            'SELECT EXISTS (
                SELECT 1 FROM {schema}.%I
                WHERE %I = $1 AND %I = $2
            )', TG_TABLE_NAME, subject_col, object_col
        ) INTO row_exists
        USING new_object_value, new_subject_value;    
            
        IF NOT row_exists 
        THEN RETURN NEW;
        END IF;

        RAISE NOTICE 'The update cannot be applied: the pair (%, %) already exists in the table. 
            And % is a asymetric property.',
            new_object_value, new_subject_value, TG_TABLE_NAME;

        PERFORM set_config('ontology.asymmetric_running', 'off', true);
        
        RETURN OLD;
    END;
    $$ LANGUAGE plpgsql;
    """
    
    return insert_function + "\n" + update_function

In [1025]:
def trigger_definition_asymmetric_property(graph, prop_mapping, context_dict):
    """
    Creates SQL trigger definitions for asymmetric properties.

    For each property declared as 'owl:AsymmetricProperty', a trigger is generated
    to prevent insertion of inverse subject-object pairs.  
    It ensures that if (A, B) exists, the inverse (B, A) is forbidden,
    thus maintaining the asymmetry constraint in the relational model.

    :param graph (rdflib.Graph): RDFLib graph of the ontology.
    :param prop_mapping (dict): Mapping of property URIs to SQL table names.
    :param context_dict (dict): Ontology context containing property metadata.
    :return (list[dict]): List of SQL trigger definitions enforcing asymmetry.
    """
    g = graph
    trigger_definition = []
    
    for prop in prop_mapping:
        mapping = prop_mapping[prop]
        prop_info = get_property_info(g, prop, context_dict)
        if not prop_info["asymmetricProperty"]:
            continue
        if not mapping["mapped"]:
            continue
                
        prop_name = prop_info["sqlName"]
        table_name = mapping.get("name")
        
        trigger_definition.append({
            "triggerName": f"trg_insert_asymmetric_property_{prop_name}",
            "pointInTime": "BEFORE",
            "event": "INSERT", 
            "target": table_name,
            "validity": "ROW",
            "function": "asymmetric_property_insert_trigger_function()"
        })
        
        trigger_definition.append({
            "triggerName": f"trg_update_asymmetric_property_{prop_name}",
            "pointInTime": "BEFORE",
            "event": "UPDATE", 
            "target": table_name,
            "validity": "ROW",
            "function": "asymmetric_property_update_trigger_function()"
        })
   
    return trigger_definition

#### SubClassOf

SubClassOf - Trigger für Variante 4 von R4 

In [1026]:
def trigger_definition_sub_classes(graph, sub_class_name, super_classes, context_dict):
    """
    Defines SQL trigger specifications for handling subclass relationships (Rule R4.4).

    For each subclass, two triggers are created to ensure consistent handling of
    inserts and deletes on the subclass table.  
    For each corresponding superclass, a delete trigger is defined to prevent
    deletion conflicts and maintain referential completeness along the class hierarchy.

    :param sub_class_name (str): SQL table name of the subclass.
    :param super_classes (set[rdflib.URIRef]): All superclasses of the subclass.
    :param context_dict (dict): Shared ontology context containing the equivalent class map.
    :return (list[dict]): Trigger definition objects in the format:
        [
            {
                "triggerName": "trg_insert_into_sub_class_movie",
                "pointInTime": "BEFORE",
                "event": "INSERT",
                "target": "movie",
                "validity": "ROW",
                "function": "insert_into_sub_class()"
            },
            {
                "triggerName": "trg_delete_from_sub_class_movie",
                "pointInTime": "AFTER",
                "event": "DELETE",
                "target": "movie",
                "validity": "ROW",
                "function": "delete_from_sub_class()"
            },
            ...
        ]
    """
    g = graph
    
    equivalent_class_map = context_dict["equivalent_class_map"]

    trigger_definition = []


    trigger_definition_sub_insert = {
        "triggerName": f"trg_insert_into_sub_class_{sub_class_name}",
        "pointInTime": "AFTER",
        "event": "INSERT", 
        "target": sub_class_name,
        "validity": "ROW",
        "function": f"insert_into_sub_class()"
    }
    trigger_definition.append(trigger_definition_sub_insert)

    trigger_definition_sub_delete = {
        "triggerName": f"trg_delete_from_sub_class_{sub_class_name}",
        "pointInTime": "AFTER",
        "event": "DELETE", 
        "target": sub_class_name,
        "validity": "ROW",
        "function": f"delete_from_sub_class()"
    }
    trigger_definition.append(trigger_definition_sub_delete)
    
    trigger_definition_sub_update = {
        "triggerName": f"trg_update_sub_class_{sub_class_name}",
        "pointInTime": "AFTER",
        "event": "UPDATE", 
        "target": sub_class_name,
        "validity": "ROW",
        "function": f"update_sub_class()"
    }
    trigger_definition.append(trigger_definition_sub_update)

    seen_super_class = []

    for super_class in super_classes:
        equi_super_class = equivalent_class_map.get(super_class, {super_class})
        
        if equi_super_class not in seen_super_class:
            seen_super_class.append(equi_super_class)
            
            if super_class in equivalent_class_map:
                    super_class_name = get_class_info(g, equi_super_class, context_dict)["sqlName"]
            else:    
                super_class_name = get_class_info(g, super_class, context_dict)["sqlName"]
        
            trigger_definition_super_delete = {
                "triggerName": f"trg_delete_from_super_class_{super_class_name}",
                "pointInTime": "BEFORE",
                "event": ["DELETE", "UPDATE"], 
                "target": super_class_name,
                "validity": "ROW",
                "function": "delete_from_super_class()"
            }
            trigger_definition.append(trigger_definition_super_delete)

    return trigger_definition

In [1027]:
def sub_class_trigger_functions(sys_schema = "puplic_sys", schema = "public"):
    """
    Generates PL/pgSQL trigger functions to enforce subClassOf in a PostgreSQL database.
    """

    delete_from_super_class = f"""
    CREATE OR REPLACE FUNCTION {schema}.delete_from_super_class() 
    RETURNS trigger AS $$
    DECLARE
        new_table_id TEXT; 
        old_table_id TEXT;
        colname TEXT;
    BEGIN
        colname := TG_TABLE_NAME || '_id';
        new_table_id := to_jsonb(NEW)->>colname;
        old_table_id := to_jsonb(OLD)->>colname;

    -- Bedingung prüfen
        IF OLD.sub_refcount_sys = 0 THEN
            IF TG_OP = 'DELETE' THEN
                RETURN OLD;
            ELSE
                RETURN NEW;
            END IF;
        
        ELSIF OLD.sub_refcount_sys <> 0 
            THEN
            IF TG_OP = 'UPDATE' AND new_table_id IS NOT DISTINCT FROM old_table_id
                THEN RETURN NEW;

            ELSIF OLD.source_sys = 'derived' THEN
                RAISE EXCEPTION 'Deleting/Updating is not possible because the ID is only derived and is still used by % subtables.', 
                OLD.sub_refcount_sys ;
            
            ELSE
                RAISE EXCEPTION 'Deleting/Updating is not possible even though the ID is explicet, because there are % subtables still using this ID.',
                OLD.sub_refcount_sys;
            END IF;
        END IF;
    END;
    $$ LANGUAGE plpgsql;
    """
    
    
    # insert_into_super_class = f"""
    # CREATE OR REPLACE FUNCTION {schema}.insert_into_super_class() 
    # RETURNS trigger AS $$
    # DECLARE
    #     exists_derived boolean;
    #     id_col_name TEXT;
    #     id_value TEXT;
    # BEGIN
    #     id_col_name := TG_TABLE_NAME || '_id';
    #     id_value := to_jsonb(NEW)->>id_col_name;

    #     -- Dynamische EXISTS-Prüfung
    #     EXECUTE format(
    #         'SELECT EXISTS (
    #             SELECT 1 
    #             FROM {schema}.%I 
    #             WHERE %I = $1 AND source = 'derived'
    #         )', TG_TABLE_NAME, id_col_name
    #     ) INTO exists_derived USING id_value;

    #     IF NOT exists_derived THEN
    #         RETURN NEW;
    #     END IF;
        
    #     EXECUTE format(
    #         'UPDATE {schema}.%I 
    #         SET source = 'explicit'
    #         WHERE %I = $1',
    #     TG_TABLE_NAME, id_col_name
    #     ) USING id_value;

    #     RETURN  NULL;
    # END;
    # $$ LANGUAGE plpgsql;
    # """


    insert_into_sub_class = f"""
    CREATE OR REPLACE FUNCTION {schema}.insert_into_sub_class()
    RETURNS TRIGGER AS $$
    DECLARE 
        super TEXT;
        colname TEXT;
        new_table_id TEXT; 
        super_name TEXT;

    BEGIN
        colname := TG_TABLE_NAME || '_id';
        new_table_id := to_jsonb(NEW)->>colname; 
        
        FOR super IN 
            SELECT super_class 
            FROM {sys_schema}.sub_class_relationship_table_sys
            WHERE sub_class = TG_TABLE_NAME::TEXT 
        LOOP
             -- Metadata für  Superklasse laden
            SELECT sql_name into super_name
            FROM {sys_schema}.all_classes_sys
            WHERE class_id = super;
            
            -- fehlenden Super-Eintrag als 'derived' anlegen
            EXECUTE format(
                'INSERT INTO %I.%I (%I, source_sys) 
                VALUES ($1, $2)
                ON CONFLICT (%I) DO NOTHING', 
                '{schema}', super_name, super_name || '_id', super_name || '_id'
            ) USING new_table_id, 'derived';

            -- sub_refcount_sys +1
            EXECUTE format(
                'UPDATE %I.%I 
                SET sub_refcount_sys = sub_refcount_sys + 1 
                WHERE %I = $1', 
                '{schema}', super_name, super_name || '_id'
            ) USING new_table_id;

            IF super = super_name THEN
                RAISE NOTICE 'For table % (super table from %) a new entry with % % was added. You may should look over it.', 
                        super, TG_TABLE_NAME, super || '_id', new_table_id;
            ELSE 
                RAISE NOTICE 'For table % (equivalent to % and super table from %) a new entry with % % was added. You may should look over it.', 
                        super_name, super, TG_TABLE_NAME, super_name || '_id', new_table_id;
            END IF;
        END LOOP;

        RETURN NEW;
    END;
    $$ LANGUAGE plpgsql;
    """

    delete_from_sub_class = f"""
    CREATE OR REPLACE FUNCTION {schema}.delete_from_sub_class()
    RETURNS TRIGGER AS $$
    DECLARE 
        super TEXT; 
        v_source TEXT; 
        v_cnt int;
        colname TEXT;
        old_table_id TEXT;
        super_name TEXT;
        source TEXT;
        sub_refcount INTEGER;
    BEGIN
        colname := TG_TABLE_NAME || '_id';
        old_table_id := to_jsonb(OLD)->>colname;  -- Wert als TEXT aus NEW holen
    
        FOR super IN 
            SELECT super_class FROM {sys_schema}.sub_class_relationship_table_sys
            WHERE sub_class = TG_TABLE_NAME::TEXT 
        LOOP
            -- Metadata für  Superklasse laden
            SELECT sql_name into super_name
            FROM {sys_schema}.all_classes_sys
            WHERE class_id = super;
            
            EXECUTE format(
                'UPDATE %I.%I
                SET sub_refcount_sys = GREATEST(sub_refcount_sys - 1, 0)
                WHERE %I = $1
                RETURNING source_sys, sub_refcount_sys',
                '{schema}', super_name, super_name || '_id'
            ) INTO source, sub_refcount USING old_table_id;
            
            IF source = 'derived' AND sub_refcount = 0 THEN
               EXECUTE format(
                    'DELETE FROM %I.%I WHERE %I = $1', 
                    '{schema}', super_name, super_name || '_id'
                ) USING old_table_id;
            END IF;
            
        END LOOP;

        RETURN NULL;
    END;
    $$ LANGUAGE plpgsql;
    """
    
    update_sub_class = f"""
    
    CREATE OR REPLACE FUNCTION {schema}.insert_into_sub_class_internal(
        table_name TEXT, table_id TEXT, table_label TEXT
    )
    RETURNS VOID AS $$
    DECLARE
        super TEXT;
        super_name TEXT;
    BEGIN
        FOR super IN
            SELECT super_class
            FROM {sys_schema}.sub_class_relationship_table_sys
            WHERE sub_class = table_name
        LOOP
             -- Metadata für  Superklasse laden
            SELECT sql_name into super_name
            FROM {sys_schema}.all_classes_sys
            WHERE class_id = super;

            EXECUTE format(
                'INSERT INTO %I.%I (%I, source_sys)
                VALUES ($1, $2)
                ON CONFLICT (%I) DO NOTHING',
                '{schema}', super_name, super_name || '_id', super_name || '_id'
            ) USING table_id, 'derived';

            EXECUTE format(
                'UPDATE %I.%I
                SET sub_refcount_sys = sub_refcount_sys + 1
                WHERE %I = $1',
                '{schema}', super_name, super_name || '_id'
            ) USING table_id;
            
            IF super = super_name THEN
                RAISE NOTICE 'For table % (super table from %) a new entry with % % was added. You may should look over it.', 
                        super, TG_TABLE_NAME, super || '_id', new_table_id;
            ELSE 
                RAISE NOTICE 'For table % (equivalent to % and super table from %) a new entry with % % was added. You may should look over it.', 
                        super_name, super, TG_TABLE_NAME, super_name || '_id', new_table_id;
            END IF;
        
        END LOOP;
    END;
    $$ LANGUAGE plpgsql;
    
    CREATE OR REPLACE FUNCTION {schema}.delete_from_sub_class_internal(
        table_name TEXT, table_id TEXT
    )
    RETURNS VOID AS $$
    DECLARE
        super TEXT;
        super_name TEXT;
        source TEXT;
        sub_refcount INTEGER;
    BEGIN
        FOR super IN
            SELECT super_class
            FROM {sys_schema}.sub_class_relationship_table_sys
            WHERE sub_class = table_name
        LOOP
            -- Metadata für  Superklasse laden
            SELECT sql_name into super_name
            FROM {sys_schema}.all_classes_sys
            WHERE class_id = super;
            
            EXECUTE format(
                'UPDATE %I.%I
                SET sub_refcount_sys = GREATEST(sub_refcount_sys - 1, 0)
                WHERE %I = $1
                RETURNING source_sys, sub_refcount_sys',
                '{schema}', super_name, super_name || '_id'
            ) INTO source, sub_refcount USING table_id;
            
            IF source = 'derived' AND sub_refcount = 0 THEN
               EXECUTE format(
                    'DELETE FROM %I.%I WHERE %I = $1', 
                    '{schema}', super_name, super_name || '_id'
                ) USING old_table_id;
            END IF;
        
        END LOOP;
    END;
    $$ LANGUAGE plpgsql;
    
    CREATE OR REPLACE FUNCTION {schema}.update_sub_class()
    RETURNS TRIGGER AS $$
    DECLARE 
        new_table_id TEXT; 
        old_table_id TEXT;
        colname TEXT;
        super_name TEXT;
    BEGIN
        colname := TG_TABLE_NAME || '_id';
        new_table_id := to_jsonb(NEW)->>colname;
        old_table_id := to_jsonb(OLD)->>colname;

        IF new_table_id IS NOT DISTINCT FROM old_table_id
        THEN
            RETURN NEW;
        END IF;

        PERFORM {schema}.delete_from_sub_class_internal(
            TG_TABLE_NAME,
            old_table_id
        );

        PERFORM {schema}.insert_into_sub_class_internal(
            TG_TABLE_NAME,
            new_table_id,
            NEW.label
        );
       
        RETURN NEW;
    END;
    $$ LANGUAGE plpgsql;
    """

    return delete_from_super_class + "\n" + delete_from_sub_class + "\n" + insert_into_sub_class + "\n" + update_sub_class

Fügt die Spalten source und sub_refcount in alle neuen Tabellen ein

In [1028]:
def trigger_definition_insert_source_sub_refcount(schema = "public"):
    """
    Trigger definition and function to add source_sys and sub_refcount_sys columns to new tables.
    """
    
    event_trigger = f"""
    DROP EVENT TRIGGER IF EXISTS trg_add_source_sub_refcount;
    CREATE EVENT TRIGGER trg_add_source_sub_refcount
    ON ddl_command_end
    WHEN TAG IN ('CREATE TABLE')
    EXECUTE FUNCTION {schema}.add_source_sub_refcount();
    """
    
    new_attributes_function = f"""
    CREATE OR REPLACE FUNCTION {schema}.add_source_sub_refcount()
    RETURNS event_trigger AS $$
    DECLARE
        obj record;
    BEGIN
        FOR obj IN SELECT * FROM pg_event_trigger_ddl_commands() LOOP
            IF obj.command_tag = 'CREATE TABLE'
                AND obj.object_identity LIKE '{schema}.%' THEN

                RAISE NOTICE 'Füge Spalten zu neuer Tabelle % hinzu', obj.object_identity;

                -- Spalten hinzufügen
                EXECUTE format('ALTER TABLE %s
                                ADD COLUMN IF NOT EXISTS source_sys VARCHAR(8) DEFAULT %L CHECK (source_sys IN (%L, %L)),
                                ADD COLUMN IF NOT EXISTS sub_refcount_sys INTEGER DEFAULT 0;',
                            obj.object_identity, 'explicit', 'explicit', 'derived');

                EXECUTE format(
                    'COMMENT ON COLUMN %s.source_sys IS %L;',
                    obj.object_identity,
                    'Says if the entry is either explicit or derived from the subtables of this table.'
                );

                EXECUTE format(
                    'COMMENT ON COLUMN %s.sub_refcount_sys IS %L;',
                    obj.object_identity,
                    'Says how many subtables also contain this entry.'
                );
            END IF;
        END LOOP;
    END;
    $$ LANGUAGE plpgsql;
    """
    return new_attributes_function + "\n" + event_trigger
    

#### SubPropertyOf

In [1029]:
def trigger_definition_sub_property(graph, prop_mapping, context_dict):
    """
    Defines SQL trigger specifications for handling subproperty relationships (Rule R4.4 analogue for properties).

    For each property table, insert and delete triggers are created to propagate
    changes along the subproperty hierarchy.  
    For each corresponding superproperty, a delete trigger ensures that
    removal consistency is maintained across the property inheritance chain.

    :param prop_mapping (dict): Mapping of properties to their SQL metadata 
        (table name, subject/object columns, etc.).
    :param context_dict (dict): Shared ontology context containing subproperty relationships.
    :return (list[dict]): Trigger definition objects in the format:
        [
            {
                "triggerName": "trg_insert_into_sub_property_directed_movie",
                "pointInTime": "BEFORE",
                "event": "INSERT",
                "target": "directed_movie",
                "validity": "ROW",
                "function": "insert_into_sub_property(subject_col, object_col, prop_name)"
            },
            {
                "triggerName": "trg_delete_from_super_property_movie_relation",
                "pointInTime": "BEFORE",
                "event": "DELETE",
                "target": "movie_relation",
                "validity": "ROW",
                "function": "delete_from_super_property()"
            },
            ...
        ]
    """
    g = graph
    super_props = context_dict["sub_property_of_dist"]
    
    trigger_definition = []
    seen_super_prop = []
    seen_props = []
    
    for prop in prop_mapping:
        
        super_properties = {x for x, dist in super_props.get(prop, {}).items() if dist == 1}
        prop_table = prop_mapping.get(prop, {}).get("table")
        
        if prop_table in seen_props or not prop_mapping.get(prop, {}).get("mapped"):
            continue
        if not "_inv" in prop_table:
            seen_props.append(prop_table)
        
        if not super_properties:
            continue
        
        prop_subject_column = prop_mapping.get(prop, {}).get("subject_column")
        prop_object_column = prop_mapping.get(prop, {}).get("object_column")
        prop_name = get_property_info(g, prop, context_dict)["sqlName"]
        prop_domain = prop_mapping.get(prop, {}).get("domain")
        if "_inv" in prop_table:
            trg_name = prop_name
        else:
            trg_name = prop_table

        trigger_definition_sub_insert = {
            "triggerName": f"trg_insert_into_sub_property_{prop_name}",
            "pointInTime": "BEFORE",
            "event": "INSERT", 
            "target": prop_table,
            "validity": "ROW",
            "function": f"insert_into_sub_property({prop_subject_column}, {prop_object_column}, {prop_name})"
        }
        trigger_definition.append(trigger_definition_sub_insert)

        trigger_definition_sub_delete = {
            "triggerName": f"trg_delete_from_sub_property_{prop_name}",
            "pointInTime": "AFTER",
            "event": "DELETE", 
            "target": prop_table,
            "validity": "ROW",
            "function": f"delete_from_sub_property({prop_subject_column}, {prop_object_column}, {prop_name})"  
        }
        trigger_definition.append(trigger_definition_sub_delete)
        
        trigger_definition_sub_update = {
            "triggerName": f"trg_update_sub_property_{prop_name}",
            "pointInTime": "AFTER",
            "event": "UPDATE", 
            "target": prop_table,
            "validity": "ROW",
            "function": f"update_sub_property({prop_subject_column}, {prop_object_column}, {prop_name})"  
        }
        trigger_definition.append(trigger_definition_sub_update)

        for super_prop in super_properties:
            if super_prop in seen_super_prop:
                continue
            
            seen_super_prop.append(super_prop)
            
            super_prop_table = prop_mapping.get(super_prop, {}).get("name")
            
            super_prop_subject_column = prop_mapping.get(super_prop, {}).get("subject_column")
            super_prop_object_column = prop_mapping.get(super_prop, {}).get("object_column")
            super_prop_name = get_property_info(g, super_prop, context_dict)["sqlName"]
        
            trigger_definition_super = {
                "triggerName": f"trg_delete_from_super_property_{super_prop_name}",
                "pointInTime": "BEFORE",
                "event": ["DELETE", "UPDATE"], 
                "target": super_prop_table,
                "validity": "ROW",
                "function": f"delete_from_super_property({super_prop_subject_column}, {super_prop_object_column}, {super_prop_name})"
            }
            trigger_definition.append(trigger_definition_super)


    return trigger_definition


In [1030]:
def sub_property_trigger_functions(sys_schema = "puplic_sys", schema = "public"):
    """
    SQL-Code der Trigger-Funktipnen gemäß Regel R4.4
    """

    delete_from_super_property = f"""
    CREATE OR REPLACE FUNCTION {schema}.delete_from_super_property() 
    RETURNS trigger AS $$
    BEGIN
    -- Bedingung prüfen
        IF OLD.sub_refcount_sys = 0 THEN
            IF TG_OP = 'DELETE' THEN
                RETURN OLD;
            ELSE
                RETURN NEW;
            END IF;

        ELSIF OLD.sub_refcount_sys <> 0 AND OLD.source_sys = 'derived' THEN
            RAISE EXCEPTION 'Deleting/Updating is not possible because the ID is only derived and is still used by % subtables.', 
            OLD.sub_refcount_sys ;

        ELSE
            RAISE EXCEPTION 'Deleting/Updating is not possible even though the ID is explicet, because there are % subtables still using this ID.',
            OLD.sub_refcount_sys;
        
        END IF;
    END;
    $$ LANGUAGE plpgsql;
    """

    insert_into_sub_property = f"""
    CREATE OR REPLACE FUNCTION {schema}.insert_into_sub_property()
    RETURNS TRIGGER AS $$
    DECLARE 
        super TEXT;
        sub_subject_col TEXT := TG_ARGV[0];
        sub_object_col TEXT := TG_ARGV[1];
        sub_property_id TEXT := TG_ARGV[2];
        new_sub_col TEXT;
        new_obj_col TEXT;
        new_sub_property_id TEXT;
        super_subject_col TEXT;
        super_object_col TEXT;
        super_table TEXT;
        obj_col_type TEXT;
        
    BEGIN
        new_sub_col := to_jsonb(NEW)->>sub_subject_col;
        new_obj_col := to_jsonb(NEW)->>sub_object_col;
        
        FOR super IN 
            SELECT super_property 
            FROM {sys_schema}.sub_property_relationship_table_sys
            WHERE sub_property = sub_property_id 
        LOOP
            -- get super property subject and object column and name of the table
            SELECT subject_column, object_column, sql_name, object_datatype
            INTO super_subject_col, super_object_col, super_table, obj_col_type
            FROM {sys_schema}.all_properties_sys
            WHERE property_id = super ;
            
            -- fehlenden Super-Eintrag als 'derived' anlegen
            EXECUTE format(
                'INSERT INTO %I.%I (%I, %I, source_sys) VALUES ($1, $2::%s, $3) ON CONFLICT (%I, %I) DO NOTHING', 
                '{schema}', super_table, super_subject_col, super_object_col, obj_col_type, super_subject_col, super_object_col
            ) USING new_sub_col, new_obj_col, 'derived'::provenance;

            -- sub_refcount_sys +1
            EXECUTE format(
                'UPDATE %I.%I SET sub_refcount_sys = sub_refcount_sys + 1 WHERE %I = $1 AND %I = $2::%s', 
                '{schema}', super_table, super_subject_col, super_object_col, obj_col_type
            ) USING new_sub_col, new_obj_col;

            RAISE NOTICE 'For table % (super table from %) a new entry with % % and % % was added. You may should look over it.',
                super_table, TG_TABLE_NAME, super_subject_col, new_sub_col, super_object_col, new_obj_col;
        END LOOP;

        RETURN NEW;
    END;
    $$ LANGUAGE plpgsql;
    """

    delete_from_sub_property = f"""
    CREATE OR REPLACE FUNCTION {schema}.delete_from_sub_property()
    RETURNS TRIGGER AS $$
    DECLARE 
        super TEXT; 
        v_source TEXT; 
        v_cnt int;
        query TEXT;
        sub_subject_col TEXT := TG_ARGV[0];
        sub_object_col TEXT := TG_ARGV[1];
        sub_property_id TEXT := TG_ARGV[2];
        old_sub_col TEXT;
        old_obj_col TEXT;
        old_sub_property_id TEXT;
        super_subject_col TEXT;
        super_object_col TEXT;
        super_table TEXT;
        
    BEGIN
        old_sub_col := to_jsonb(OLD)->>sub_subject_col;
        old_obj_col := to_jsonb(OLD)->>sub_object_col;
        FOR super IN 
            SELECT super_property FROM {sys_schema}.sub_property_relationship_table_sys
            WHERE sub_property = sub_property_id
        LOOP
            -- get super property subject and object column and name of the table
            SELECT subject_column, object_column, sql_name
            INTO super_subject_col, super_object_col, super_table
            FROM {sys_schema}.all_properties_sys
            WHERE property_id = super ;
            
            EXECUTE format(
                'UPDATE %I.%I SET sub_refcount_sys = GREATEST(sub_refcount_sys - 1, 0) WHERE %I = $1 AND %I = $2',
                '{schema}', super_table, super_subject_col, super_object_col
            ) USING old_sub_col, old_obj_col;
        END LOOP;

        RETURN NULL;
    END;
    $$ LANGUAGE plpgsql;
    """
    
    update_sub_property = f"""
    # CREATE OR REPLACE FUNCTION {schema}.update_sub_property()
    # RETURNS TRIGGER AS $$
    # BEGIN
    #     -- DELETE-Teil (für OLD)
    #     PERFORM {schema}.delete_from_sub_property();

    #     -- INSERT-Teil (für NEW)
    #     PERFORM {schema}.insert_into_sub_property();

    #     RETURN NEW;
    # END;
    # $$ LANGUAGE plpgsql;
    
    CREATE OR REPLACE FUNCTION {schema}.update_sub_property()
    RETURNS TRIGGER AS $$
    DECLARE
        sub_subject_col TEXT := TG_ARGV[0];
        sub_object_col  TEXT := TG_ARGV[1];
        sub_property_id TEXT := TG_ARGV[2];

        old_sub_col TEXT;
        old_obj_col TEXT;

        new_sub_col TEXT;
        new_obj_col TEXT;

    BEGIN
        -- Werte extrahieren
        old_sub_col := to_jsonb(OLD)->>sub_subject_col;
        old_obj_col := to_jsonb(OLD)->>sub_object_col;

        new_sub_col := to_jsonb(NEW)->>sub_subject_col;
        new_obj_col := to_jsonb(NEW)->>sub_object_col;


        IF new_sub_col IS NOT DISTINCT FROM old_sub_col
            AND new_obj_col IS NOT DISTINCT FROM old_obj_col 
        THEN
            RETURN NEW;
        END IF;

        -- DELETE-Propagation (für alte Werte)
        PERFORM {schema}.delete_from_sub_property(
            old_sub_col,
            old_obj_col,
            sub_property_id
        );

        -- INSERT-Propagation (für neue Werte)
        PERFORM {schema}.insert_into_sub_property(
            new_sub_col,
            new_obj_col,
            sub_property_id
        );

        RETURN NEW;
    END;
    $$ LANGUAGE plpgsql;

    """




    return delete_from_super_property +  "\n" + delete_from_sub_property + "\n" + insert_into_sub_property  + "\n" + update_sub_property

In [1031]:
def sub_property_trigger_functions(sys_schema = "puplic_sys", schema = "public"):
    """
    TGenerates PL/pgSQL trigger functions to enforce subPropertyOf in a PostgreSQL database.
    """
  
    delete_from_super_property = f"""
    CREATE OR REPLACE FUNCTION {schema}.delete_from_super_property() 
    RETURNS trigger AS $$
    DECLARE
        super_subject_col TEXT := TG_ARGV[0];
        super_object_col  TEXT := TG_ARGV[1];
        super_property_id TEXT := TG_ARGV[2];

        old_sub TEXT;
        old_obj TEXT;

        new_sub TEXT;
        new_obj TEXT;
    BEGIN
        old_sub := to_jsonb(OLD)->>super_subject_col;
        old_obj := to_jsonb(OLD)->>super_object_col;

        new_sub := to_jsonb(NEW)->>super_subject_col;
        new_obj := to_jsonb(NEW)->>super_object_col;

        IF OLD.sub_refcount_sys = 0 THEN
            IF TG_OP = 'DELETE' THEN
                RETURN OLD;
            ELSE
                RETURN NEW;
            END IF;

        ELSIF TG_OP = 'UPDATE' 
            AND old_sub IS NOT DISTINCT FROM new_sub
            AND old_obj IS NOT DISTINCT FROM new_obj
            THEN RETURN NEW;            
        
        ELSIF OLD.sub_refcount_sys <> 0 AND OLD.source_sys = 'derived' THEN
            RAISE EXCEPTION 'Deleting/Updating table % is not possible because the subject-object-pair is only derived and is still used by % subtables.', 
            TG_TABLE_NAME, OLD.sub_refcount_sys ;

        ELSE
            RAISE EXCEPTION 'Deleting/Updating in table % is not possible even though the subject-object-pair is explicet, because there are % subtables still using this subject-object-pair.',
            TG_TABLE_NAME, OLD.sub_refcount_sys;
        
        END IF;
    END;
    $$ LANGUAGE plpgsql;
    """
    
    
    propagate_sub_property_internal = f"""
    CREATE OR REPLACE FUNCTION {schema}.propagate_sub_property_internal(
        is_insert BOOLEAN,
        sub_column TEXT,
        obj_column TEXT,
        sub_property_id TEXT
    )
    RETURNS VOID AS $$
    DECLARE
        super TEXT;
        super_subject_col TEXT;
        super_object_col TEXT;
        super_table TEXT;
        subj_col_type TEXT;
        obj_col_type TEXT;
        source TEXT;
        sub_refcount INTEGER;
    BEGIN
        FOR super IN
            SELECT super_property
            FROM {sys_schema}.sub_property_relationship_table_sys
            WHERE sub_property = sub_property_id
        LOOP
            -- Metadaten der Superproperty laden
            SELECT subject_column, object_column, sql_name, object_datatype
                INTO super_subject_col, super_object_col, super_table, obj_col_type
            FROM {sys_schema}.all_properties_sys
            WHERE property_id = super;

            IF is_insert THEN
                -- Fehlenen Super-Entry erzeugen
                EXECUTE format(
                    'INSERT INTO %I.%I (%I, %I, source_sys)
                    VALUES ($1, $2::%s, $3)
                    ON CONFLICT (%I, %I) DO NOTHING',
                    '{schema}', super_table, super_subject_col, super_object_col, obj_col_type, super_subject_col, super_object_col
                ) USING sub_column, obj_column, 'derived';

                -- sub_refcount erhöhen
                EXECUTE format(
                    'UPDATE %I.%I
                    SET sub_refcount_sys = sub_refcount_sys + 1
                    WHERE %I = $1 AND %I = $2::%s',
                    '{schema}', super_table, super_subject_col, super_object_col, obj_col_type
                ) USING sub_column, obj_column;
                
                IF super = super_table THEN
                    RAISE NOTICE 'For table % (super table from %) a new entry with % % and % % was added. You may should look over it.', 
                            super_table, 'TG_TABLE_NAME', super_subject_col, sub_column, super_object_col, obj_column;
                ELSE 
                    RAISE NOTICE 'For table % (equivalent to % and super table from %) a new entry with % % and % % was added. You may should look over it.', 
                            super_table, super, 'TG_TABLE_NAME', super_subject_col, sub_column, super_object_col, obj_column;
                END IF;

            ELSE
                -- DELETE-Propagation
                EXECUTE format(
                    'UPDATE %I.%I
                    SET sub_refcount_sys = GREATEST(sub_refcount_sys - 1, 0)
                    WHERE %I = $1 AND %I = $2::%s
                    RETURNING source_sys, sub_refcount_sys',
                    '{schema}', super_table, super_subject_col, super_object_col, obj_col_type
                ) INTO source, sub_refcount USING sub_column, obj_column;
                
                IF source = 'derived' AND sub_refcount = 0 THEN
                    EXECUTE format(
                        'DELETE FROM %I.%I 
                        WHERE %I = $1 AND %I = $2::%s',
                        '{schema}', super_table, super_subject_col, super_object_col, obj_col_type
                    ) USING sub_column, obj_column;
                END IF;
                
            END IF;

        END LOOP;
    END;
    $$ LANGUAGE plpgsql;
    """
    
    insert_into_sub_property = f"""
    CREATE OR REPLACE FUNCTION {schema}.insert_into_sub_property()
    RETURNS TRIGGER AS $$
    DECLARE
        sub_subject_col TEXT := TG_ARGV[0];
        sub_object_col  TEXT := TG_ARGV[1];
        sub_property_id TEXT := TG_ARGV[2];

        new_sub TEXT;
        new_obj TEXT;
    BEGIN
        new_sub := to_jsonb(NEW)->>sub_subject_col;
        new_obj := to_jsonb(NEW)->>sub_object_col;

        PERFORM {schema}.propagate_sub_property_internal(
            TRUE,               -- is_insert
            new_sub,
            new_obj,
            sub_property_id
        );

        RETURN NEW;
    END;
    $$ LANGUAGE plpgsql;
    """

    delete_from_sub_property = f"""
    CREATE OR REPLACE FUNCTION {schema}.delete_from_sub_property()
    RETURNS TRIGGER AS $$
    DECLARE
        sub_subject_col TEXT := TG_ARGV[0];
        sub_object_col  TEXT := TG_ARGV[1];
        sub_property_id TEXT := TG_ARGV[2];

        old_sub TEXT;
        old_obj TEXT;
    BEGIN
        old_sub := to_jsonb(OLD)->>sub_subject_col;
        old_obj := to_jsonb(OLD)->>sub_object_col;

        PERFORM {schema}.propagate_sub_property_internal(
            FALSE,              -- is_insert
            old_sub,
            old_obj,
            sub_property_id
        );

        RETURN OLD;
    END;
    $$ LANGUAGE plpgsql;
    """
    
    update_sub_property = f"""
    CREATE OR REPLACE FUNCTION {schema}.update_sub_property()
    RETURNS TRIGGER AS $$
    DECLARE
        sub_subject_col TEXT := TG_ARGV[0];
        sub_object_col  TEXT := TG_ARGV[1];
        sub_property_id TEXT := TG_ARGV[2];

        old_sub TEXT;
        old_obj TEXT;

        new_sub TEXT;
        new_obj TEXT;
    BEGIN
        old_sub := to_jsonb(OLD)->>sub_subject_col;
        old_obj := to_jsonb(OLD)->>sub_object_col;

        new_sub := to_jsonb(NEW)->>sub_subject_col;
        new_obj := to_jsonb(NEW)->>sub_object_col;

        -- Wenn sich nichts geändert hat: fertig
        IF old_sub IS NOT DISTINCT FROM new_sub
            AND old_obj IS NOT DISTINCT FROM new_obj 
        THEN
            RETURN NEW;
        END IF;

        -- DELETE
        PERFORM {schema}.propagate_sub_property_internal(
            FALSE,
            old_sub,
            old_obj,
            sub_property_id
        );

        -- INSERT
        PERFORM {schema}.propagate_sub_property_internal(
            TRUE,
            new_sub,
            new_obj,
            sub_property_id
        );

        RETURN NEW;
    END;
    $$ LANGUAGE plpgsql;
    """

    return delete_from_super_property +  "\n" + propagate_sub_property_internal + "\n" + delete_from_sub_property + "\n" + insert_into_sub_property  + "\n" + update_sub_property

#### Nachhaltige Anlage von neuen Triggern

SubClassOf

In [1032]:
def install_new_subclassof_trigger_function(sys_schema = "puplic_sys", schema = "public"):
    """
    Trigger function to create and drop triggers on subtables when subclass relationships are created or removed.
    """

    create_subtable_triggers = f"""
    CREATE OR REPLACE FUNCTION {sys_schema}.create_subtable_triggers_sys(
        target_table TEXT,
        target_table_type TEXT, 
        element_type TEXT, 
        new_table_id TEXT, 
        subject_col TEXT, 
        object_col TEXT)
    RETURNS VOID AS $$
    DECLARE
        insert_name TEXT := format('trg_insert_into_sub_%s_%s', element_type, new_table_id);
        delete_name TEXT := format('trg_delete_from_%s_%s_%s', target_table_type, element_type, new_table_id);
        update_name TEXT := format('trg_update_%s_%s_%s', target_table_type, element_type, new_table_id);
        arglist TEXT := '';
    BEGIN
        -- prevents two sessions from creating/deleting the same trigger at the same time
        PERFORM pg_advisory_xact_lock(hashtext('create_equiv_triggers:'||target_table));

        IF element_type = 'property' THEN
            arglist := format('%s, %s, %s', subject_col, object_col, new_table_id);
        else 
            arglist := '';
        END IF;

        IF target_table_type = 'sub' THEN
            EXECUTE format(
                'DROP TRIGGER IF EXISTS %I ON %I.%I', 
                insert_name, '{schema}', target_table
            );
            
            EXECUTE format(
                'CREATE TRIGGER %I 
                AFTER INSERT ON %s.%I 
                FOR EACH ROW
                EXECUTE FUNCTION %s.insert_into_sub_%I(%s)',
                insert_name, '{schema}', target_table, '{schema}', element_type, arglist
            );

            EXECUTE format(
                'DROP TRIGGER IF EXISTS %I ON %s.%I', 
                update_name, '{schema}', target_table
            );
            
            EXECUTE format(
                'CREATE TRIGGER %I 
                AFTER UPDATE ON %s.%I 
                FOR EACH ROW
                EXECUTE FUNCTION %s.update_sub_%I(%s)',
                update_name, '{schema}', target_table, '{schema}', element_type, arglist
            );
        END IF;

        EXECUTE format(
            'DROP TRIGGER IF EXISTS %I ON %s.%I', 
            delete_name, '{schema}', target_table
        );
        
        IF target_table_type = 'sub' THEN
            EXECUTE format(
                'CREATE TRIGGER %I 
                AFTER DELETE ON %s.%I 
                FOR EACH ROW
                EXECUTE FUNCTION %s.delete_from_sub_%I(%s)',
                delete_name, '{schema}', target_table, '{schema}', element_type, arglist
            );
            
        ELSIF target_table_type = 'super' THEN
            EXECUTE format(
                'CREATE TRIGGER %I 
                BEFORE DELETE ON %s.%I 
                FOR EACH ROW
                EXECUTE FUNCTION %s.delete_from_super_%I(%s)',
                delete_name, '{schema}', target_table, '{schema}', element_type, arglist
            );
        END IF;
        

    END;
    $$ LANGUAGE plpgsql;
    """


    drop_subtable_trigger = f"""
    CREATE OR REPLACE FUNCTION {sys_schema}.drop_subtable_triggers_sys(
        target_table TEXT,
        old_table_id TEXT, 
        target_table_type TEXT, 
        element_type TEXT
    )
    RETURNS VOID AS $$
    DECLARE
        insert_name TEXT := format('trg_insert_into_sub_%s_%s', element_type, old_table_id);
        delete_name TEXT := format('trg_delete_from_%s_%s_%s', target_table_type, element_type, old_table_id);
        update_name TEXT := format('trg_update_sub_%s_%s', element_type, old_table_id);
    BEGIN
        -- prevents two sessions from creating/deleting the same trigger at the same time
        PERFORM pg_advisory_xact_lock(hashtext('create_equiv_triggers:'||target_table));
        
    
        IF target_table_type = 'sub' THEN
            -- Drop INSERT trigger,
            EXECUTE format(
                'DROP TRIGGER IF EXISTS %I ON {schema}.%I', 
                insert_name, target_table
            );
            
            -- Drop UPDATE trigger
            EXECUTE format(
                'DROP TRIGGER IF EXISTS %I ON {schema}.%I', 
                update_name, target_table
            );
        END IF;

        -- Drop DELETE trigger
        EXECUTE format(
            'DROP TRIGGER IF EXISTS %I ON {schema}.%I', 
            delete_name, target_table
        );
        
    END;
    $$ LANGUAGE plpgsql;
    """

    insert_new_instances = f"""
    CREATE OR REPLACE FUNCTION {sys_schema}.sync_new_sub_table_instances_sys(
        sub_table TEXT,
        super_table TEXT,
        sub_cols TEXT[],     
        super_cols TEXT[]   
    )
    RETURNS VOID AS $$
    DECLARE
        v1 TEXT;
        v2 TEXT;

        select_sql TEXT;
        insert_sql TEXT;
    BEGIN

        -- NULL-Schutz
        IF sub_table IS NULL OR super_table IS NULL THEN
            RAISE NOTICE 'sync_new_sub_table_instances_sys() skipped: table NULL';
            RETURN;
        END IF;

        -- subClassOf
        IF array_length(sub_cols,1) = 1 THEN
            select_sql := format(
                'SELECT %I FROM {schema}.%I',
                sub_cols[1], sub_table
            );

            insert_sql := format(
                'INSERT INTO {schema}.%I (%I, source_sys, sub_refcount_sys)
                VALUES ($1, ''derived'', 1)
                ON CONFLICT (%I)
                DO UPDATE SET sub_refcount_sys = {schema}.%I.sub_refcount_sys + 1',
                super_table, super_cols[1], super_cols[1], super_table
            );

            FOR v1 IN EXECUTE select_sql LOOP
                EXECUTE insert_sql USING v1;
            END LOOP;

            RETURN;
        END IF;

        -- subPropertyOf
        IF array_length(sub_cols,1) = 2 THEN
            select_sql := format(
                'SELECT %I, %I FROM {schema}.%I',
                sub_cols[1], sub_cols[2], sub_table
            );

            insert_sql := format(
                'INSERT INTO {schema}.%I (%I, %I, source_sys, sub_refcount_sys)
                VALUES ($1, $2, ''derived'', 1)
                ON CONFLICT (%I, %I)
                DO UPDATE SET sub_refcount_sys = {schema}.%I.sub_refcount_sys + 1',
                super_table, super_cols[1], super_cols[2], super_cols[1], super_cols[2], super_table
            );

            FOR v1, v2 IN EXECUTE select_sql LOOP
                EXECUTE insert_sql USING v1, v2;
            END LOOP;

            RETURN;
        END IF;
    END;
    $$ LANGUAGE plpgsql;

    """
    
    delete_old_instances = f"""
    CREATE OR REPLACE FUNCTION {sys_schema}.sync_remove_sub_table_instances_sys(
        sub_table TEXT, 
        super_table TEXT, 
        sub_cols TEXT[], 
        super_cols TEXT[]
    )
    RETURNS VOID AS $$
    DECLARE
        rec RECORD;
        update_sql TEXT;
        delete_sql TEXT;
        where_clause TEXT;
        i INT;
        v_source TEXT;
        v_cnt INT;
        v1 TEXT;
        v2 TEXT;
    BEGIN
        IF sub_table IS NULL OR super_table IS NULL THEN
            RAISE NOTICE 'sync_remove_sub_table_instances_sys() skipped: table NULL';
            RETURN;
        END IF;
        
        -- WHERE clause bauen
        where_clause := '';
        FOR i IN 1 .. array_length(sub_cols, 1) LOOP
            where_clause := where_clause || format('%I = $%s AND ', super_cols[i], i);
        END LOOP;
        where_clause := left(where_clause, length(where_clause) - 5);

        update_sql := format(
            'UPDATE {schema}.%I
            SET sub_refcount_sys = GREATEST(sub_refcount_sys - 1, 0)
            WHERE %s
            RETURNING source_sys, sub_refcount_sys',
            super_table, where_clause
        );

        delete_sql := format(
            'DELETE FROM {schema}.%I WHERE %s',
            super_table, where_clause
        );
        
        IF array_length(sub_cols, 1) = 2 THEN
            FOR v1, v2 IN
            EXECUTE format(
                'SELECT %I, %I FROM {schema}.%I',
                sub_cols[1], sub_cols[2], sub_table
            )
            LOOP
                EXECUTE update_sql USING v1, v2;
                
                IF v_cnt = 0 AND v_source = 'derived' THEN
                    EXECUTE delete_sql USING v1, v2;
                END IF;
            END LOOP;
            
        ELSE
            FOR v1 IN
            EXECUTE format(
                'SELECT %I FROM {schema}.%I',
                sub_cols[1], sub_table
            )
            LOOP
                EXECUTE update_sql USING v1;

                IF v_cnt = 0 AND v_source = 'derived' THEN
                    EXECUTE delete_sql USING v1;
                END IF;
            END LOOP;
        END IF;
        
    END;
    $$ LANGUAGE plpgsql;
    
    """

    help_function_sql_name_props = f""" 
    CREATE OR REPLACE FUNCTION {sys_schema}.get_sql_name_for_property_sys(
        prop_id TEXT,
        sys_schema_name TEXT,
        elements_table TEXT
    )
    RETURNS TEXT AS $$
    DECLARE
        result TEXT;
    BEGIN
    
        EXECUTE format(
            'SELECT sql_name FROM {sys_schema}.%I WHERE property_id = $1',
            elements_table
        )
        INTO result
        USING prop_id;

        RETURN result;
    END;
    $$ LANGUAGE plpgsql;
    """
    
    help_function_sync_instances = f""" 
    CREATE OR REPLACE FUNCTION {sys_schema}.call_sync_functions_sys(
        element_type TEXT,
        sync_type TEXT,
        sub_table TEXT,
        super_table TEXT,
        sub_subject_col TEXT,
        sub_object_col TEXT,
        super_subject_col TEXT,
        super_object_col TEXT
    )
    RETURNS TEXT AS $$
    DECLARE
        result TEXT;
    BEGIN
    
        IF sync_type = 'new' THEN
        
            IF element_type = 'property' THEN
                    PERFORM {sys_schema}.sync_new_sub_table_instances_sys(
                        sub_table,
                        super_table,
                        ARRAY[sub_subject_col, sub_object_col],
                        ARRAY[super_subject_col, super_object_col]
                    );
            ELSE
                PERFORM {sys_schema}.sync_new_sub_table_instances_sys(
                    sub_table,
                    super_table,
                    ARRAY[sub_table||'_id'],
                    ARRAY[super_table||'_id']
                );
            END IF;
            RETURN '';
            
        ELSE
            IF element_type = 'property' THEN
                    PERFORM {sys_schema}.sync_remove_sub_table_instances_sys(
                        sub_table,
                        super_table,
                        ARRAY[sub_subject_col, sub_object_col],
                        ARRAY[super_subject_col, super_object_col]
                    );
            ELSE
                PERFORM {sys_schema}.sync_remove_sub_table_instances_sys(
                    sub_table,
                    super_table,
                    ARRAY[sub_table||'_id'],
                    ARRAY[super_table||'_id']
                );
            END IF;
            RETURN '';
        END IF; 
    
    END;
    $$ LANGUAGE plpgsql;
    """    
    
    
    install_new_subtable_triggers = f"""
    CREATE OR REPLACE FUNCTION {sys_schema}.install_new_subtable_trigger_sys()
    RETURNS TRIGGER AS $$
    DECLARE 
        has_any boolean;
        element_type TEXT := TG_ARGV[0];
        sub_table_col TEXT := TG_ARGV[1];
        super_table_col TEXT := TG_ARGV[2];
        sys_table TEXT := TG_ARGV[3];
        
        new_sub_table TEXT;
        new_super_table TEXT;
        new_sub_target_table TEXT;
        new_super_target_table TEXT;
        
        old_sub_table TEXT;
        old_super_table TEXT;
        old_sub_target_table TEXT;
        old_super_target_table TEXT;
        
        all_elements_sys TEXT;
        
        old_sub_subject_col TEXT := '';
        old_sub_object_col TEXT := '';
        old_sub_property_id TEXT := '';
        old_super_subject_col TEXT := '';
        old_super_object_col TEXT := '';
        old_super_property_id TEXT := '';
        
        sub_subject_col TEXT := '';
        sub_object_col TEXT := '';
        sub_property_id TEXT := '';
        super_subject_col TEXT := '';
        super_object_col TEXT := '';
        super_property_id TEXT := '';
    BEGIN
        new_sub_table := to_jsonb(NEW) ->> sub_table_col;
        new_super_table := to_jsonb(NEW) ->> super_table_col;
        old_sub_table := to_jsonb(OLD) ->> sub_table_col;
        old_super_table := to_jsonb(OLD) ->> super_table_col;
        
        IF element_type = 'class' THEN all_elements_sys := 'all_classes_sys';
        ELSE
            all_elements_sys := 'all_properties_sys';
            IF TG_OP = 'INSERT' OR TG_OP 'UPDATE' THEN
                SELECT subject_column, object_column
                    INTO sub_subject_col, sub_object_col
                FROM {sys_schema}.all_properties_sys
                WHERE property_id = new_sub_table;
                
                SELECT subject_column, object_column
                    INTO super_subject_col, super_object_col
                FROM {sys_schema}.all_properties_sys
                WHERE property_id = new_super_table;
            END IF;
            
            IF TG_OP = 'DELETE' OR TG_OP 'UPDATE' THEN
                SELECT subject_column, object_column
                    INTO old_sub_subject_col, old_sub_object_col
                FROM {sys_schema}.all_properties_sys
                WHERE property_id = old_sub_table;
                
                SELECT subject_column, object_column
                    INTO old_super_subject_col, old_super_object_col
                FROM {sys_schema}.all_properties_sys
                WHERE property_id = old_super_table;
            END IF;
        END IF;
        
        
        -- get super property subject and object column and name of the table
        IF element_type = 'property' THEN
            new_sub_target_table  := {sys_schema}.get_sql_name_for_property_sys(new_sub_table,  '{sys_schema}', all_elements_sys);
            new_super_target_table := {sys_schema}.get_sql_name_for_property_sys(new_super_table, '{sys_schema}', all_elements_sys);
            old_sub_target_table  := {sys_schema}.get_sql_name_for_property_sys(old_sub_table,  '{sys_schema}', all_elements_sys);
            old_super_target_table := {sys_schema}.get_sql_name_for_property_sys(old_super_table, '{sys_schema}', all_elements_sys);
        ELSE
            new_sub_target_table := new_sub_table;
            new_super_target_table := new_super_table;
            old_sub_target_table := old_sub_table;
            old_super_target_table := old_super_table;
        END IF;
        
        
        IF TG_OP = 'INSERT' THEN
            PERFORM {sys_schema}.call_sync_functions_sys(
                element_type, 'new',
                new_sub_target_table, new_super_target_table,
                sub_subject_col, sub_object_col,
                super_subject_col, super_object_col
            );   
            
            PERFORM {sys_schema}.create_subtable_triggers_sys(new_sub_target_table, 'sub', element_type, new_sub_table, sub_subject_col, sub_object_col);
            PERFORM {sys_schema}.create_subtable_triggers_sys(new_super_target_table, 'super', element_type, new_super_table, super_subject_col, super_object_col);
            RETURN NEW;
        END IF;


        IF TG_OP = 'UPDATE' THEN

            IF new_sub_table IS DISTINCT FROM old_sub_table OR new_super_table IS DISTINCT FROM old_super_table THEN
                PERFORM {sys_schema}.call_sync_functions_sys(
                    element_type, 'remove', 
                    old_sub_target_table, old_super_target_table,
                    old_sub_subject_col, old_sub_object_col,
                    old_super_subject_col, old_super_object_col
                );
                PERFORM {sys_schema}.call_sync_functions_sys(
                    element_type, 'new', 
                    new_sub_target_table, new_super_target_table,
                    sub_subject_col, sub_object_col,
                    super_subject_col, super_object_col
                );
            END IF;
            
            IF old_sub_table IS DISTINCT FROM new_sub_table THEN
                EXECUTE format(
                    'SELECT EXISTS (
                        SELECT 1 FROM {sys_schema}.%I
                        WHERE %I = $1
                    )', sys_table, sub_table_col
                ) INTO has_any USING old_sub_table;
            
                IF NOT has_any THEN
                    PERFORM {sys_schema}.drop_subtable_triggers_sys(old_sub_target_table, old_sub_table, 'sub', element_type);
                END IF;

                PERFORM {sys_schema}.create_subtable_triggers_sys(new_sub_target_table, 'sub', element_type, new_sub_table, sub_subject_col, sub_object_col);

            ELSIF old_super_table IS DISTINCT FROM new_super_table THEN

                EXECUTE format(
                    'SELECT EXISTS (
                        SELECT 1 FROM {sys_schema}.%I
                        WHERE %I = $1
                    )', sys_table, sub_table_col
                ) INTO has_any USING old_super_table;
                
                IF NOT has_any THEN
                    PERFORM {sys_schema}.drop_subtable_triggers_sys(old_super_target_table, old_super_table, 'super', element_type);
                END IF;
                
                PERFORM {sys_schema}.create_subtable_triggers_sys(new_super_target_table, 'super', element_type, new_super_table, super_subject_col, super_object_col);                
            ELSE
                RETURN NEW; 
            END IF;
            
            RETURN NEW;
        END IF;


        IF TG_OP = 'DELETE' THEN
            PERFORM {sys_schema}.call_sync_functions_sys(
                element_type, 'remove',
                old_sub_target_table, old_super_target_table,
                old_sub_subject_col, old_sub_object_col,
                old_super_subject_col, old_super_object_col
            );
        
            EXECUTE format(
                'SELECT EXISTS (
                    SELECT 1 FROM {sys_schema}.%I
                    WHERE %I = $1
                )', sys_table, sub_table_col
            ) INTO has_any USING old_sub_table;
            
            IF NOT has_any THEN
                PERFORM {sys_schema}.drop_subtable_triggers_sys(old_sub_target_table, old_sub_table, 'sub', element_type);
            END IF;

            EXECUTE format(
                'SELECT EXISTS (
                    SELECT 1 FROM {sys_schema}.%I
                    WHERE %I = $1
                )', sys_table, super_table_col
            ) INTO has_any USING old_super_table;
            
            IF NOT has_any THEN
                PERFORM {sys_schema}.drop_subtable_triggers_sys(old_super_target_table, old_super_table, 'super', element_type);
            END IF;
            
            RETURN OLD;
        END IF;

        RETURN COALESCE(NEW, OLD);
    END;
    $$ LANGUAGE plpgsql;
    """

    return (create_subtable_triggers + "\n" + insert_new_instances + "\n" + delete_old_instances + "\n" + drop_subtable_trigger + "\n" + help_function_sql_name_props + "\n" + help_function_sync_instances + "\n" + install_new_subtable_triggers )

#### komplexe Klassen

In [1033]:
def complex_class_trigger_functions(function_name, class_discription, complex_cls, cls_mapping, 
                                    prop_mapping, context_dict, relation, 
                                    schema = "public"):
    """
    Generates a PL/pgSQL trigger function enforcing complex OWL class conditions at the SQL level.

    This function translates logical OWL constructs (restrictions, unions, intersections,
    complements, and enumerations) into executable SQL checks. The resulting function ensures 
    that each INSERT or UPDATE into a class table complies with the semantic constraints 
    defined by the ontology.

    Supported OWL patterns include:
      - owl:someValuesFrom / owl:allValuesFrom / owl:hasValue
      - owl:cardinality / minCardinality / maxCardinality (and their qualified variants)
      - owl:unionOf, owl:intersectionOf, owl:complementOf
      - owl:oneOf enumerations
      - owl:equivalentClass and owl:disjointWith relations

    The function dynamically builds EXISTS and COUNT-based SQL checks for each condition
    and combines them according to the class predicate.

    :param function_name (str): Name of the generated PL/pgSQL trigger function.
    :param class_discription (dict): Parsed logical structure of the complex class
        (from 'context_dict["complex_class_map"]').
    :param com_cls (rdflib.URIRef): Target class (complex OWL class).
    :param cls_mapping (dict): Mapping of classes to their SQL table representations.
    :param prop_mapping (dict): Mapping of properties to their SQL table representations.
    :param context_dict (dict): Shared ontology context with identifiers, names, and prefixes.
    :param relation (rdflib.URIRef): OWL relation type (e.g. owl:equivalentClass, owl:disjointWith).
    :param schema (str, optional): Target SQL schema name. Defaults to "public".
    :return (str): Full SQL definition of the generated PL/pgSQL trigger function.
    """

    def get_prop_info(rest):
        prop = rest["onProperty"]
        info = prop_mapping.get(prop, {})
        return prop, info.get("table"), info.get("subject_column"), info.get("object_column")

    def sql_exists(query): 
        sql_exists = (
            f"SELECT EXISTS (\n"
            f"{query}\n"
            ")\n"
        )
        return sql_exists


    summary = []
    classes = []
    one_ofs = []
    all_restrictions = []
    
    final_sql = []
    
    mapping = cls_mapping.get(complex_cls)
    complex_cls_table = mapping.get("name")
    complex_cls_id = f"{complex_cls_table}_id"
    not_ = (" NOT " if relation == OWL.disjointWith else "")
    equi_class = (True if relation == OWL.equivalentClass and "insert_into" in function_name else False)
    equi_cls_table = True if relation == OWL.equivalentClass and "_rel"not  in function_name else False


    if equi_class:
        final_sql.append(
            f"EXECUTE 'SELECT EXISTS (\n"
            f"SELECT 1 FROM {schema}.{complex_cls_table} t\n"
            f"WHERE t.{complex_cls_id}=$1\n"
            f")' INTO is_valid USING new_id_col;\n"
            f"IF is_valid THEN RETURN NULL;\n"
            f"END IF;\n"
        )   


    for cls in class_discription.get("classes", set()):
        cls_table = cls_mapping.get(cls, {}).get("name")
        cls_id = f"{cls_table}_id"
        classes.append(sql_exists(
            f"SELECT 1 FROM {schema}.{cls_table} t WHERE t.{cls_id}=$1"
            ))


    if class_discription.get("oneOf", set()):
        oneofs = {get_sql_name(x, "individual", context_dict) or remove_prefix(x) for x in class_discription["oneOf"]}
        oneofs = f"''{"'', ''".join(oneofs)}''"
        one_ofs.append(f"SELECT $1 IN ({oneofs})")


    RESTRICTION_TEMPLATES = {
        "hasValue": ("SELECT 1 FROM {schema}.{table} t \n"
                      "WHERE t.{sub}=$1 \n"
                      "AND t.{obj}=''{val}''"
                      ),
        "someValuesFrom": ("SELECT 1 FROM {schema}.{table} t \n"
                           "JOIN {schema}.{cls_table} d ON d.{cls_id}=t.{obj} \n"
                           "WHERE t.{sub}=$1"
                           ),
        "allValuesFrom": ("SELECT 1 FROM {schema}.{table} t \n"
                          "WHERE t.{sub}=$1 \n"
                          "AND NOT EXISTS ( \n"
                          "SELECT 1 FROM {schema}.{cls_table} d \n"
                          "WHERE d.{cls_id}=t.{obj})"
                        ),     
        "hasSelf": ("SELECT 1 FROM {schema}.{table} t \n"
                    "WHERE t.{sub}=$1 \n"
                    "AND t.{obj}=$1"
                    )        
    }

    restrictions = class_discription.get("restrictions", {})
    if isinstance(restrictions, dict): 
        restrictions = [restrictions]

    for rest in restrictions:
        for key, template in RESTRICTION_TEMPLATES.items():
            if key in rest:
                prop, table, sub, obj = get_prop_info(rest)
                val, cls_table, cls_id = "", "", ""
                if key == "hasValue":
                    val = get_sql_name(rest[key], "individual", context_dict) or remove_prefix(rest[key])
                if "ValuesFrom" in key:
                    cls = rest[key]
                    cls_table = cls_mapping.get(cls, {}).get("name")
                    cls_id = f"{cls_table}_id"
                query = template.format(schema=schema, table=table, sub=sub, obj=obj, val=val, cls_table=cls_table, cls_id=cls_id)
                all_restrictions.append(sql_exists(query))
    

        CARDINALITY_RULES = {
            "cardinality": ("=", "exactly"),
            "minCardinality": (">=", "at least"),
            "maxCardinality": ("<=", "at most"),
            "qualifiedCardinality": ("=", "exactly"),
            "minQualifiedCardinality": (">=", "at least"),
            "maxQualifiedCardinality": ("<=", "at most")
        }
        for key, (op, desc) in CARDINALITY_RULES.items():
            if key in rest:
                prop, table, sub, obj = get_prop_info(rest)
                card_val = rest[key].toPython()
                join_clause = ""
                if "Qualified" in key:
                    cls = rest["onClass"]
                    cls_table = cls_mapping.get(cls, {}).get("name")
                    cls_id = f"{cls_table}_id"
                    join_clause = f" JOIN {schema}.{cls_table} d ON d.{cls_id}=t.{obj}"
                query = (
                    f"SELECT COUNT({sub}) {op} {card_val}\n" 
                    f"FROM {schema}.{table} t {join_clause}\n" 
                    f"WHERE t.{sub}=$1\n"
                )
                all_restrictions.append(query)

    summary = classes + one_ofs + all_restrictions
    predicate = class_discription.get("predicate")


    if predicate == "single":
        final_sql.append(f"EXECUTE '{"".join(summary)}' INTO is_valid USING new_id_col;\n")
        if equi_class:
            final_sql.append(
                "IF NOT is_valid THEN RETURN NULL;\n"
                "END IF;\n"
            )
        else:
            final_sql.append(
                f"IF{not_} is_valid THEN RETURN NEW;\n"
                "END IF;\n"
                f"RAISE EXCEPTION 'Condition violated for complex class {complex_cls_table}: not satisfied.';\n"
            )

    elif predicate in ("implicit intersection", OWL.intersectionOf):
        for i, part in enumerate(summary, 1):
            final_sql.append(
                f"\n-- Condition {i}\n"
                f"EXECUTE '{part.strip()}' INTO is_valid USING new_id_col;\n")
            check = "NOT is_valid" if equi_class else f"{not_}NOT is_valid"
            msg = f"'Intersection violated for complex class {complex_cls_table}: condition {i} not satisfied.'"
            if not equi_class:
                final_sql.append(
                    f"\nIF {check} THEN\n"
                    f"RAISE EXCEPTION {msg};\n"
                    "END IF;\n"
                )
            else:
                final_sql.append(
                    f"IF {check} THEN RETURN NULL;\n"
                    "END IF;\n"
                )

    elif predicate == OWL.unionOf:
        for i, part in enumerate(summary, 1):
            final_sql.append(
                f"\n-- Condition {i}\n"
                f"EXECUTE '{part.strip()}' INTO is_valid USING new_id_col;\n"
            )
            check = "NOT is_valid" if equi_class else f"{not_} is_valid"
            final_sql.append(
                f"IF {check} THEN RETURN {'NULL' if equi_class else 'NEW'};\n"
                "END IF;\n"
            )
        if not equi_class:
            final_sql.append(
                f"RAISE EXCEPTION 'Union violated for complex class {complex_cls_table}: no condition satisfied.';\n"
                "RETURN NULL;\n"
            )
        else: 
            final_sql.append(
                f"IF {check} THEN RETURN NULL;\n"
                "END IF;\n"
            )

    elif predicate == OWL.complementOf:
        for i, part in enumerate(summary, 1):
            final_sql.append(
                f"-- Condition {i}\n"
                f"EXECUTE '{part.strip()}' INTO is_valid USING new_id_col;\n")
            check = "is_valid" if equi_class else f"{not_} is_valid"
            if not equi_class:
                final_sql.append(
                    f"IF {check} THEN \n"
                    f"RAISE EXCEPTION 'Complement violated for complex class {complex_cls_table}: condition {i} satisfied.';\n"
                    "END IF;\n"
                )
            else:
                final_sql.append(
                    f"IF {check} THEN RETURN NULL;\n"
                    "END IF;\n"
                )

    if equi_class:
        final_sql.append(
            f"\nEXECUTE 'INSERT INTO {schema}.{complex_cls_table} ({complex_cls_id}) VALUES ($1)' \nUSING new_id_col ;\n"
            f"RAISE NOTICE 'For the complex equivalent class {complex_cls_table} a new entry was added. You May schould look over it.'; \n"
            "RETURN NEW;\n"
        )

    final_sql = "".join(final_sql)


    trigger_function = (
f"""
CREATE OR REPLACE FUNCTION {schema}.{function_name}()
RETURNS trigger AS $$
DECLARE
    is_valid BOOLEAN;
    id_col TEXT:=TG_ARGV[0];
    new_id_col TEXT;
BEGIN
    new_id_col:=to_jsonb(NEW)->>id_col;
    
{final_sql}
RETURN NEW;
END;
$$ LANGUAGE plpgsql;
"""
    )
    

    return trigger_function


In [1034]:
def trigger_definition_complex_classes(graph, cls_mapping, prop_mapping, context_dict, schema = "public"):
    """
    Defines and generates trigger functions for enforcing complex OWL class semantics.

    This function iterates through all complex class definitions detected in the ontology
    and creates corresponding trigger definitions and PL/pgSQL functions via
    'complex_class_trigger_functions()'.

    For each non-simple class, the following triggers are generated:
      - A BEFORE INSERT trigger on the class table to validate the complex condition.
      - For equivalentClass constructs: additional AFTER INSERT triggers on all
        involved component tables (classes or properties), ensuring automatic propagation
        of individuals satisfying the condition into the equivalent class table.

    :param graph (rdflib.Graph): RDF graph containing the ontology.
    :param cls_mapping (dict): Mapping of classes to SQL tables.
    :param prop_mapping (dict): Mapping of properties to SQL tables.
    :param context_dict (dict): Shared ontology context containing complex class structures.
    :param schema (str, optional): SQL schema name. Defaults to "public".
    :return (tuple[list[dict], str]): 
        - **trigger_definition**: List of all generated SQL trigger definitions in dict format.
        - **trigger_function**: Combined SQL string of all PL/pgSQL trigger functions.
    """
    g = graph
    trigger_definition = []
    trigger_function = []
    
    for cls in cls_mapping:
        if get_class_info(g, cls, context_dict)["simpleClass"]:
            continue
        
        class_discription = context_dict["complex_class_map"].get(cls)
        
        table_name = cls_mapping[cls]["name"]
        
        for relation in class_discription:
            class_dis = class_discription[relation]
                
            if not class_dis["no_bnodes"]:
                continue
            
            function_name_check = f"complex_class_{table_name}_check"
            trigger_definition.append({
                "triggerName": f"trg_complex_class_{table_name}",
                "pointInTime": "BEFORE",
                "event": "INSERT", 
                "target": table_name,
                "validity": "ROW",
                "function": f"{function_name_check}({table_name}_id)"
            })
            trigger_function.append(complex_class_trigger_functions(function_name_check, class_dis, cls, cls_mapping, prop_mapping, context_dict, relation, schema))
            
            if relation == OWL.equivalentClass:
                for element in class_dis["containing_elements"]:
                    mapping = cls_mapping.get(element, {}) or prop_mapping.get(element, {})
                    
                    element_id = mapping.get("subject_column", set()) or f"{mapping.get("name")}_id"
                    function_name_insert = f"complex_class_{table_name}_insert_into"
                    
                    trigger_definition.append({
                        "triggerName": f"trg_complex_class_{table_name}_{mapping.get("name")}",
                        "pointInTime": "AFTER",
                        "event": "INSERT", 
                        "target": mapping.get("name"),
                        "validity": "ROW",
                        "function": f"{function_name_insert}({element_id})"
                    })
                trigger_function.append(complex_class_trigger_functions(function_name_insert, class_dis, cls, cls_mapping, prop_mapping, context_dict, relation, schema))
    trigger_function = "\n".join(trigger_function)

    return trigger_definition, trigger_function

### Views

In [1035]:
def view_definition_reflexive_property(graph, property_name, table_name, domain_name):
    """
    Generates SQL view definitions for reflexive OWL properties.

    Reflexive properties imply that every instance of the domain class is related to itself.
    This function creates a view that merges:
      - All explicitly stored property assertions from the base table.
      - All *implicit* self-relations derived from the reflexivity axiom.

    The result ensures that both explicit and logically implied relations are visible,
    providing a complete representation of the reflexive property at the SQL level.

    :param graph (rdflib.Graph): RDF graph containing the ontology.
    :param property_name (str | list[str]): Property name(s) for which the reflexive view is generated.
    :param table_name (str): SQL table name that stores the explicit property data.
    :param domain_name (str | list[str]): Corresponding domain class name(s).
    :return (list[dict]): A list of view definitions, each in the format:
        [
            {
                "viewName": "hasSibling_ref_view",
                "select": [
                    "d.person_id, d.person_id AS hasSibling, 'derived'::text AS source_sys",
                    "t.person_id, t.hasSibling, 'explicit'::text AS source_sys"
                ],
                "from": ["person d", "hasSibling t"],
                "union": "UNION ALL",
                "viewComments": "Describes the reflexive nature of the property..."
            }
        ]
    """
    g = graph
    reflexive_view_definition = []
    
    if not isinstance(property_name, list):
        property_name = [property_name]
        domain_name = [domain_name]
    

    i = 0
    for prop_name in property_name:
        
        reflexive_property_view_definition = {}
        select = []
        from_ = []
        where = []
        union_selects = "UNION ALL"
        
        if len(property_name) == 2:
            id_column = next((x for x in domain_name if prop_name not in x))
            domain_id_column = f"{domain_name[i]}_id" if domain_name[i] else f"none_id"
            current_domain_name = domain_name[i]
        else:
            id_column = f"{domain_name[0]}_id" if domain_name[0] else "none_id"
            domain_id_column = f"{domain_name[0]}_id"
            current_domain_name = domain_name[0]
        
        if current_domain_name:
            view_name = f"{current_domain_name}_{prop_name}_rel_reflexive"
            
            select.append(f"d.{domain_id_column}, d.{domain_id_column} AS {prop_name}, 'derived'::text AS source_sys, 0 AS sub_refcount_sys")
            select.append(f"t.{id_column}, t.{prop_name}, t.source_sys::text , t.sub_refcount_sys")
            
            from_.append(f"{current_domain_name} d")
            from_.append(f"{table_name} t")
            where.append(
                f"NOT EXISTS (\n"
                f"SELECT 1 \n"
                f"FROM {table_name} \n"
                f"WHERE {id_column} = d.{domain_id_column} AND {prop_name} = d.{domain_id_column} \n"
                ")"
            )
        else:
            view_name = f"none_{prop_name}_rel_reflexive"
            
            select.append(f"d.{id_column}, d.{id_column} AS {prop_name}, 'derived'::text AS source_sys, 0 AS sub_refcount_sys")
            select.append(f"t.{id_column}, t.{prop_name}, t.source_sys::text , t.sub_refcount_sys")
            
            from_.append(f"{table_name} d")
            from_.append(f"{table_name} t")
            where.append(
                f"NOT EXISTS (\n"
                f"SELECT 1 \n"
                f"FROM {table_name} \n"
                f"WHERE {id_column} = d.{id_column} AND {prop_name} = d.{id_column} )"
            )
        
        view_comments = (
            f"{prop_name} is defined as a reflexive property. \n"
            f"All instances of {current_domain_name} are implicitly related to themselves via this property. \n"
            f"These self-references are not stored in the base table {table_name} but are represented here for completeness."
        )
        
        i += 1
        
        reflexive_property_view_definition = {
            "viewName": view_name,
            "select": select,
            "from": from_,
            "union": union_selects,
            "viewComments": view_comments,
            "where": where
        }
        
        reflexive_view_definition.append(reflexive_property_view_definition)
        
    return reflexive_view_definition

In [1036]:
def view_definition_element_disjoint_with(graph, mapping, context_dict, element_type, schema = "public"):
    """
    Generates SQL view definitions to identify violations of 'owl:disjointWith' or 
    'owl:propertyDisjointWith' constraints.

    For disjoint classes or properties, this view collects all instances or 
    subject-object pairs that appear in more than one disjoint group.  
    Each group corresponds to one 'AllDisjointClasses' or 'AllDisjointProperties' construct.

    The view counts how often the same entity or pair occurs across disjoint elements
    and aggregates the conflicting tables, allowing semantic inconsistencies to be 
    detected directly at the SQL layer.

    :param graph (rdflib.Graph): RDF graph containing the ontology.
    :param mapping (dict): Mapping of classes or properties to their SQL table and column metadata.
    :param context_dict (dict): Ontology context containing precomputed disjoint element pairs.
    :param element_type (str): Either '"class"' or '"properties"', depending on the view target.
    :param schema (str, optional): Target SQL schema name. Defaults to "public".
    :return (dict | None): A single view definition if disjoint pairs exist, otherwise 'None'.
    """
    g = graph
    disjoint_pairs = context_dict[f"{element_type}_disjoint_pairs"]
    
    
    if disjoint_pairs:

        if element_type == "property":
            view_comments = (
                "Lists all subject-object pairs that violate owl:propertyDisjointWith and owl:AllDisjointPropeties constraints defined in the ontology. "
                "Each row represents a case where the same subject is associated with the same object via two or more properties "
                "that are declared to be disjoint. This view helps identify semantic inconsistencies that arise when "
                "disjoint properties are mistakenly used with overlapping values."
            )
        else:
            view_comments = (
                "Lists all individuals that violate owl:disjointWith and owl:AllDisjointClasses constraints defined in the ontology. "
                "Each row represents a case where the same individual is part of two or more classes "
                "that are declared to be disjoint. This view helps identify semantic inconsistencies that arise when "
                "disjoint classes are mistakenly used with overlapping values."
            )
        
        disjoint_property_view_definition = {
            "viewName": f"disjoint_{element_type}_violations",
            "viewComments": view_comments
        }
        select = []
        from_ = []
        
        i = 1
        
        select = (
            f"group_id, {'subject' if element_type == 'properties' else 'instance'}, "
            f"{'object, ' if element_type == 'properties' else ''} "
            f"COUNT(DISTINCT source) AS number_of_occurrences, ARRAY_AGG(DISTINCT source) AS tables"
        )

        for pair in disjoint_pairs:

            from_select = []
            for p in pair:
                table_name = mapping.get(p, {}).get("name")
                subject_column = mapping.get(p, {}).get("subject_column")
                object_column = mapping.get(p, {}).get("object_column")
                id_column = mapping.get(p, {}).get("id_column")

                from_select.append(
                    f"SELECT 'G{i}' AS group_id, '{table_name}' AS source, "
                    f"{subject_column + ' AS subject, ' if element_type == 'properties' else id_column + ' AS instance'}"
                    f"{object_column + ' AS object ' if element_type == 'properties' else ''} "
                    f"FROM {schema}.{table_name}"
                )
            from_.append("\nUNION ALL\n".join(from_select))
            i += 1

        
        disjoint_property_view_definition.update({
            "select": select,
            "from": f"({'\nUNION ALL\n'.join(from_)}) AS all_pairs",
            "group_by": f"group_id, {'subject, object' if element_type == 'properties' else 'instance'}",
            "having": "COUNT(DISTINCT source) > 1",
            "order_by": f"group_id, {'subject, object' if element_type == 'properties' else 'instance'}"
        })
        
        return disjoint_property_view_definition

    else:
        return None
        

In [1037]:
def view_definition_equivalent_element(graph, elem, table_name, element_type, context_dict, domain = None):
    """
    Creates a SQL view definition that unifies all data from equivalent classes
    or equivalent properties into a single logical view.

    :param graph (rdflib.Graph): RDFLib graph of the ontology.
    :param elem (URIRef): The class or property for which the view is created.
    :param table_name (str): SQL table name of the representative element.
    :param element_type (str): Either "class" or "property".
    :param context_dict (dict): Ontology context containing equivalent groups and mappings.
    :param domain (str | None): Optional domain table name (only needed for properties).
    :return (dict): A structured view definition.
    """
    g = graph
    
    if element_type == "class":
        elem_name = get_class_info(g, elem, context_dict)["sqlName"]
        view_name = elem_name
    else:
        elem_name = get_property_info(g, elem, context_dict)["sqlName"]
        view_name = f"{domain}_{elem_name}_rel"
        
    
    view_comment = (
        f"This view stores the equivalent {element_type} {elem_name}. The corresponding {element_type} table is {table_name}."
    )
    
    equivalent_view_definition = {
        "viewName": view_name,
        "viewComments": view_comment,
        "select": "*",
        "from_": table_name 
    }
    return equivalent_view_definition

In [1038]:
def view_definition_inverse_property(graph, elem, table_name, context_dict, domain, subject_column, object_column):
    """
    Creates a SQL view definition that exposes both directions of an inverse
    property pair in a unified representation.

    :param graph (rdflib.Graph): RDFLib graph of the ontology.
    :param elem (URIRef): The property for which the inverse view is created.
    :param table_name (str): SQL table name of the representative property.
    :param context_dict (dict): Ontology context containing inverse groups and mappings.
    :param domain (str): Name of the domain class/table for this property.
    :param subject_column (str): SQL column storing the subject identifier.
    :param object_column (str): SQL column storing the object identifier.

    :return (dict): A structured view definition.
    """
    g = graph
    
    elem_name = get_property_info(g, elem, context_dict)["sqlName"]
    view_name = f"{domain}_{elem_name}_rel_inv"
    
    view_comment = (
        f"This view stores the inverse property {elem_name}. The corresponding property table is {table_name}."
    )
    
    inverse_view_definition = {
        "viewName": view_name,
        "viewComments": view_comment,
        "select": f"{subject_column}, {object_column}",
        "from_": table_name 
    }
    return inverse_view_definition   

### Tabellen + Attribute

#### Andere Tabellen

Eine Tabelle mit allen Klassen

##### Klassen

In [1039]:
def table_definition_all_classes_table():
    """
    Defines the technical metadata table 'all_classes_sys'.

    This system table lists all ontology classes that were transformed into relational
    tables during the shredding process. Each record represents one class and links its
    semantic identifier (URI) with its generated SQL table name.

    The table serves as a central reference for all class-related system components
    (e.g., trigger installation, subclass relations, equivalent class mappings).

    :retun: Tuple[dict, List[dict]] - Format z. B.:
        (
            {   "tableName": "all_classes_sys",
                "tableAnnotations": ["All classes from the ontology."],
                "columns": [   {"attributeName": "implemented_as",
                                "datatype": "VARCHAR",
                                "comment": ["..."]
                                }, 
                                ...
                            ],                    
            },
            [   
                {   "foreign_key_table": "all_classes_sys",
                    "foreign_key_column": "equivalent_with_table",
                    "target_table": "all_classes_sys",
                    "target_column": "table_name"
                }
            ]
        )
    """
    columns = []
    # equivalent_class_key_definiton = ""
    all_classes_table = "all_classes_sys"


    columns.append({
        "attributeName": "class_id",
        "datatype": "VARCHAR",
        "primaryKey": True,
        "comment": ["Internal and stable identifier of the ontology class used within the Shredder. "
                    "Serves as the technical key to link class-related metadata across system tables."]
    })

    column_names = ["class_uri", "sql_name", "implemented_as", "has_key_properties", "is_disjoint_with"]#, "representative_id"]
    
    column_comments = {
        "class_uri": ["Full URI of the ontology class as defined in the RDF/OWL ontology "
                    "(e.g., http://example.org/ontology#Movie). Preserves the original semantic reference."],
        "sql_name": ["Name of the SQL table generated for this class in the relational schema. Represents the "
                    "physical database object that stores instances of this class."],
        "implemented_as": ["How is the class implemented"],
        "has_key_properties": ["List of key properties that uniquely identify instances of this class."],
        "is_disjoint_with": ["List of all disjoint classes. Classes within a List are all pairwaise disjoint to eachother."],
    }

    for name in column_names:
        column = {
            "attributeName": name,
            "datatype": "VARCHAR",
            "comment": column_comments.get(name),
        }
        columns.append(column)
    
    all_classes_table_definition = {
        "tableName": all_classes_table,
        "tableAnnotations": ["TECHNICAL TABLE: This table lists all classes that were mapping into the relational database. "
                             "Each row represents a unique class included in the transformation."],
        "columns": columns,
    }
    
    return all_classes_table_definition

In [1040]:
def table_definition_sub_class_of():
    """
    Defines the system table 'sub_class_relationship_table_sys' and its foreign-key constraints.

    This table stores explicit 'rdfs:subClassOf' relationships extracted from the ontology.
    It connects the SQL tables representing subclasses and superclasses, enabling
    trigger creation and integrity checks for class hierarchies.

    Additionally, a system trigger ('trg_install_new_sub_class_trigger') is defined
    to automatically install new triggers when subclass relationships are inserted or updated.

    :return (tuple[dict, list[dict], dict]):
        - Table definition for 'sub_class_relationship_table_sys'.
        - List of foreign-key definitions linking subclass/superclass IDs to 'all_classes_sys'.
        - Trigger definition for automatic installation of subclass triggers.
    """ 

    columns = []
    sub_class_key_definition = []
    table_name = "sub_class_relationship_table_sys"

    column_names = ["sub_class", "super_class"]

    column_comments = {
        "sub_class": ["Name of the table corresponding to the subtable"],
        "super_class": ["Name of the table corresponding to the supertable"]
    }
    
    for name in column_names:
        column = {
            "attributeName": name,
            "datatype": "VARCHAR",
            "primaryKey": True,
            "comment": column_comments.get(name), 
            "notNull": True
        }
        columns.append(column)
        
        sub_class_key_definition.append({
            "foreign_key_table": table_name,
            "foreign_key_column": name,
            "target_table": "all_classes_sys",
            "target_column": "class_id",
        })

    
    sub_class_table_definition = {
        "tableName": table_name,
        "columns": columns,
        "tableAnnotations": ["TECHNICAL TABLE: Stores all sub class relationships between generated tables."
                             "Serves as metadata for existing triggers and creating new ones. Does not store "
                             "any instance data."]
    }

    sub_class_trigger_definition = {
       "triggerName": f"trg_install_new_sub_class_trigger_sys",
        "pointInTime": "AFTER",
        "event": ["INSERT", "UPDATE", "DELETE"],
        "target": table_name,
        "validity": "ROW",
        "function": f"install_new_subtable_trigger_sys('class', 'sub_class', 'super_class', {table_name})" 
    }

    return sub_class_table_definition, sub_class_key_definition, sub_class_trigger_definition

In [1041]:
def table_definition_equivalent_class_table():
    """
    Defines the system table 'equivalent_class_table_sys' and its foreign-key constraints.

    This table records all 'owl:equivalentClass' relationships between ontology classes.
    Each row represents a bidirectional equivalence link and ensures semantic alignment
    between multiple class representations in the relational schema.

    The table enables consistency across class mappings, preventing redundant storage
    of equivalent instances and supporting unified querying.

    :return (tuple[dict, list[dict]]):
        - Table definition for 'equivalent_class_table_sys'.
        - List of foreign-key definitions referencing 'all_classes_sys'.
    """
    columns = []
    equivalent_class_key_definiton = []
    table_name = "equivalent_class_table_sys"


    column_names = ["class_id", "equivalent_class", "group_id"]
    
    column_comments = {
        "class_id": ["Internal and stable identifier of the class used within the Shredder. "
                    "Serves as the technical key to link class-related metadata across system tables."],
        "equivalent_class": ["Contains the class_id of another class declared as equivalent (owl:equivalentClass)."],
        "group_id": ["ID to group equivalent classes together. All classes with the same ID belong to one equivalent group."]
    }

    for name in column_names:
        column = {
            "attributeName": name,
            "datatype": "VARCHAR",
            "primaryKey": True,
            "comment": column_comments.get(name),
        }
        
        if name in ["class_id", "equivalent_class"]:
            column["primaryKey"] = True
        
            equivalent_class_key_definiton.append({
                "foreign_key_table": table_name,
                "foreign_key_column": name,
                "target_table": "all_classes_sys",
                "target_column": "class_id",
            })
            
        columns.append(column)
        
    equivalent_classes_table_definition = {
        "tableName": table_name,
        "tableAnnotations": ["TECHNICAL TABLE: This table contains all owl:equivalentClass relationships. " 
                             "Each row links two classes that are declared to be semantically equivalent. "
                             "Used to ensure consistent interpretation and avoid duplication in the relational schema."],
        "columns": columns,
    }
    
    return equivalent_classes_table_definition, equivalent_class_key_definiton

##### Properties

In [1042]:
def table_definition_all_properties_table():
    """
    Defines the technical metadata table 'all_properties_sys'.

    This system table lists all ontology properties that were transformed into relational
    structures (tables, attributes, or views). It connects each property’s semantic URI
    with its corresponding SQL implementation and column names.

    The table serves as a reference for property management, trigger installation,
    and reasoning over property characteristics such as domain, range, or implementation type.

    :retun: Tuple[dict, List[dict]] - Format z. B.:
        (
            {   "tableName": "all_properties_sys",
                "tableAnnotations": ["All properties from the ontology."],
                "columns": [   {"attributeName": "implemented_as",
                                "datatype": "VARCHAR",
                                "comment": ["..."]
                                }, 
                                ...
                            ],                    
            },
            [   
                {   "foreign_key_table": "all_properties_sys",
                    "foreign_key_column": "equivalent_with_property",
                    "target_table": "all_properties_sys",
                    "target_column": "property_name"
                }
            ]
        )
    """
    columns = []
    domain_range_key_definition = []
    all_properties_table = "all_properties_sys"


    columns.append({
        "attributeName": "property_id",
        "datatype": "VARCHAR",
        "primaryKey": True,
        "comment": ["Internal and stable identifier of the ontology property used within the Shredder. "
                    "Serves as the technical key to link property-related metadata across system tables."]
    })
    prop_characteristics = ["functional", "inverse_functional", "symmetricProperty", "asymmetricProperty", "transitiveProperty", "reflexiveProperty", "irreflexiveProperty"]
    column_names = ["property_uri", "sql_name", "implemented_as", "subject_column", "object_column", "domain_class", "range_class", "object_datatype", "is_disjoint_with", "property_chain_axiom"]
    column_names.extend(prop_characteristics)
    
    column_comments = {
        "property_uri": ["Full URI of the ontology property as defined in the RDF/OWL ontology. "
                         "Preserves the original semantic reference."],
        "sql_name": ["Name of the SQL element generated for this property in the relational schema. "
                     "Represents the physical database object that stores instances of this property."],
        "implemented_as": ["How is the property implemented"],
        "subject_column": ["Name of the column that stores the ID of the properties subject."],
        "object_column": ["Name of the column that either stores the ID or the value of the properties object."],
        "object_datatype": ["Datatype of the object column."],
        "domain_class": ["The class that is defined as the domain of this property."],
        "range_class": ["The class that is defined as the range of this property."],
        "is_disjoint_with": ["List of all disjoint properties. Properties within a List are all pairwaise disjoint to eachother."],
        "property_chain_axiom": ["List of all owl:PropertyChainAxiom-Properties."]
    }
    
    for chara in prop_characteristics:
        column_comments[f"{chara}"] = [f"True if the property is {clean_value(chara)}."]
        
    for name in column_names:     
        column = {
            "attributeName": clean_value(name),
            "datatype": "VARCHAR(255)",
            "comment": column_comments.get(name),
        }
        columns.append(column)
    
    all_properties_table_definition = {
        "tableName": all_properties_table,
        "tableAnnotations": ["TECHNICAL TABLE: This table lists all properties that were mapping into the relational database. "
                             "Each row represents a unique property included in the transformation."],
        "columns": columns,
    }
    
    for name in ["domain_class", "range_class"]:
        domain_range_key_definition.append({
            "foreign_key_table": all_properties_table,
            "foreign_key_column": name,
            "target_table": "all_classes_sys",
            "target_column": "class_id",
        })
    
    return all_properties_table_definition, domain_range_key_definition

Für Version 3 von R4 - Erstellt die subClassOf-Metatabelle

In [1043]:
def table_definition_sub_property_of():
    """
    Defines the system table 'sub_property_relationship_table_sys' and its foreign-key constraints.

    This table stores explicit 'rdfs:subPropertyOf' relationships extracted from the ontology.
    It connects the SQL tables that represent sub- and super-properties, allowing the Shredder
    to propagate property inheritance during SQL trigger generation and schema enforcement.

    Additionally, a system trigger ('trg_install_new_sub_property_trigger') is defined
    to automatically install or update subproperty triggers whenever new relationships are inserted.

    :return (tuple[dict, list[dict], dict]):
        - Table definition for 'sub_property_relationship_table_sys'.
        - List of foreign-key definitions linking each sub/super property to 'all_properties_sys'.
        - Trigger definition for dynamic installation of subproperty triggers.

    """ 

    columns = []
    sub_property_key_definition = []
    sub_property_table_name = "sub_property_relationship_table_sys"

    column_names = ["sub_property", "super_property"]

    column_comments = {
        "sub_property": ["Name of the sub property how you can find it in this DB."],
        "super_property": ["Name of the super property how you can find it in this DB."],
    }

    for name in column_names:
        column = {
            "attributeName": name,
            "datatype": "VARCHAR",
            "primaryKey": True,
            "comment": column_comments.get(name),
            "notNull": True
        }
        columns.append(column)

        sub_property_key_definition.append({
            "foreign_key_table": sub_property_table_name,
            "foreign_key_column": name,
            "target_table": "all_properties_sys",
            "target_column": "property_id",
        })

    
    sub_property_table_definition = {
        "tableName": sub_property_table_name,
        "columns": columns,
        "tableAnnotations": ["TECHNICAL TABLE: Stores all sub property relationships between mapped properties."
                             "Serves as metadata for existing triggers and creating new ones. Does not store "
                             "any instance data."]
    }

    sub_property_trigger_definition = {
       "triggerName": f"trg_install_new_sub_property_trigger_sys",
        "pointInTime": "AFTER",
        "event": ["INSERT", "UPDATE", "DELETE"],
        "target": sub_property_table_name,
        "validity": "ROW",
        "function": f"install_new_subtable_trigger_sys('property', 'sub_property', 'super_property', {sub_property_table_name})" 
    }

    return sub_property_table_definition, sub_property_key_definition, sub_property_trigger_definition

In [1044]:
def table_definition_equivalent_property_table():
    """
    Defines the system table 'equivalent_property_table_sys' and its foreign-key constraints.

    This table records all 'owl:equivalentProperty' relationships between ontology properties.
    Each row represents a bidirectional equivalence link, ensuring that properties declared as
    semantically equivalent are consistently interpreted and managed across the generated schema.

    The table supports property unification, joint constraint application,
    and consolidated data access for equivalent properties.

    :return (tuple[dict, list[dict]]):
        - Table definition for 'equivalent_property_table_sys'.
        - List of foreign-key definitions linking properties to 'all_properties_sys'.
    """
    columns = []
    equivalent_property_key_definiton = []
    table_name = "equivalent_property_table_sys"


    column_names = ["property_id", "equivalent_property", "group_id"]
    
    column_comments = {
        "property_id": ["Internal and stable identifier of the property used within the Shredder."],
        "equivalent_with": ["Contains the property_id of another property declared as equivalent (owl:equivalentProperty)."],
        "group_id": ["ID to group equivalent properties together. All proeprties with the same ID belong to one equivalent group."]
    }

    for name in column_names:
        column = {
            "attributeName": name,
            "datatype": "VARCHAR",
            "comment": column_comments.get(name),
        }

        if name in ["property_id", "equivalent_property"]:
            column["primaryKey"] = True
        
            equivalent_property_key_definiton.append({
                "foreign_key_table": table_name,
                "foreign_key_column": name,
                "target_table": "all_properties_sys",
                "target_column": "property_id",
            })
            
        columns.append(column)
        
    equivalent_properties_table_definition = {
        "tableName": table_name,
        "tableAnnotations": ["TECHNICAL TABLE: This table stores all owl:equivalentProperty relationships. " 
                             "Each pair of properties in the table is considered semantically equivalent. "
                             "Used to group equivalent properties and handle them jointly during schema generation."],
        "columns": columns,
    }
    
    return equivalent_properties_table_definition, equivalent_property_key_definiton

##### Allgemein

Erstellt Tabelle zur Darstellung von Instanz-Annotations

In [1045]:
def table_definition_annotation_table():
    """
    Defines the system table 'ontology_annotations_meta', which stores all annotations extracted 
    from the ontology for classes, properties, and individuals.

    This table captures human-readable metadata such as labels, comments, notes, and provenance.
    Each record links an ontology element (subject) to its annotation property and value,
    enabling query-based exploration of semantic metadata directly within SQL.

    The table is used to enrich the relational schema with descriptive context
    and supports documentation, multilingual labels, and data provenance tracking.

    :return (dict): Table definition for 'ontology_annotations_meta' with columns

    :retunrn: dict - Format z. B.:
        {   "tableName": ontology_annotations_meta,
            "tableAnnotations": ["..."],
            "columns": [{   "attributeName": "subject_id",
                            "datatype": "VARCHAR(255)",
                            "comment": ["..."]
                            "primary_key": True
                        }, 
                            ...
                       ],                    
        }    
    """
    columns = []
    table_name = "_annotations_meta"
    
    column_names = ["subject_name", "subject_id", "subject_type", "annotation_property", "annotation_value", "annotation_uri"]

    column_comments = {
        "subject_name": ["Unique name of the element as used in the database."],
        "subject_id": ["Unique name/identifier of of the element. As you can find in the tables all_classes_sys or all_properties_sys"],
        "subject_type": ["Type of the annotated element, e.g. class, property, individual."],
        "annotation_property": ["The property used for the annotation, such as rdfs:label, rdfs:comment, dc:creator, "
                                "or another annotation property."],
        "annotation_uri": ["Full URI of the annotation property."],
        "annotation_value": ["Value of the annotation, eather a literal string (label, comment) or a URI reference to "
                             "another resource."],
    }

    for name in column_names:
        column = {
            "attributeName": name,
            "datatype": "VARCHAR",
            "comment": column_comments.get(name),
        }
        
        # if name in ["subject_id", "subject_type"]:
        #     column["primaryKey"] = True
        
        
        columns.append(column)

    annotation_table_definition = {
        "tableName": table_name,
        "columns": columns,
        "tableAnnotations": ["Stores all semantic annotations extracted from the ontology (labels, comments, notes, "
                             "and other metadata) for classes, properties and individuals. Each record links the "
                             "annotation to its SQL representation (subject_id) and preserves provenance via the "
                             "original ontology URI (source_uri) and origin type (source). This table serves as a "
                             "searchable dictionary of human-readable metadata for ontology-derived database elements, "
                             "supporting multilingual queries and provenance analysis."]
    }
   
    return annotation_table_definition

In [1046]:
def table_definition_has_key_table():
    """
    Defines the SQL table structure for storing owl:hasKey constraints.

    :return: dict - Table definition structure.
    """

    columns = []
    has_key_key_definition = []
    table_name = "_class_keys_meta"
    
    column_names = ["class_id", "key_property_id", "key_group"]

    column_comments = {
        "class_id": ["Unique name of the class, as you can find in the table all_classes_sys."],
        "key_property_id": ["Unique name of the key property, as you can find in the table all_properties_sys."],
        "key_group": ["If a class has multiple keys, this field indicates the group number."],
    }

    for name in column_names:
        column = {
            "attributeName": name,
            "datatype": "VARCHAR",
            "comment": column_comments.get(name),
        }
        
        if name in ["class_id", "key_property_id", "key_group"]:
            column["primaryKey"] = True
        
        
        columns.append(column)

    annotation_table_definition = {
        "tableName": table_name,
        "columns": columns,
        "tableAnnotations": ["Stores all owl:hasKey Triples for classes. Each record links a class to one of its key properties, "
                             "along with a group identifier to distinguish multiple keys. This table serves as metadata "
                             "to identify which properties uniquely identify instances of a class in the relational schema."]
    }
    
    has_key_key_definition.append({
        "foreign_key_table": table_name,
        "foreign_key_column": "class_id",
        "target_table": "all_classes_sys",
        "target_column": "class_id",
    })
    has_key_key_definition.append({
        "foreign_key_table": table_name,
        "foreign_key_column": "key_property_id",
        "target_table": "all_properties_sys",
        "target_column": "property_id",
    })
   
    return annotation_table_definition, has_key_key_definition
    

In [1047]:
def table_definition_deprecated_elements():
    """
    Defines the SQL table structure for storing all deprecated ontology elements.

    :return: dict - Table definition structure.
    """

    columns = []
    table_name = "deprecated_classes_and_properties_sys"
    
    column_names = ["element_id", "element_type", "element_uri"]

    column_comments = {
        "element_id": ["Unique name of the class."],
        "element_uri": ["Full URI of the element."],
        "element_type": ["Type of the element: ether class or property."],
    }

    for name in column_names:
        column = {
            "attributeName": name,
            "datatype": "VARCHAR",
            "comment": column_comments.get(name),
        }
        
        if name == "element_id":
            column["primaryKey"] = True
    
        columns.append(column)

    deprecated_table_definition = {
        "tableName": table_name,
        "columns": columns,
        "tableAnnotations": ["Stores all deprecated classes and propreties."]
    }
       
    return deprecated_table_definition
    
    

#### Klassen- / Propertytabellen

##### Hilfsfunktionen

In [1048]:
def table_definition_property_comments(graph, table_name, table_type, prop_name, 
                                       domain_name, prop_info, context_dict
    ):
    """
    Generates descriptive column comments for a property in the relational schema.

    The result provides rich inline documentation for the generated SQL schema,
    directly explaining the semantic role of each property column or table.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param table_name (str): SQL table in which the property is implemented.
    :param table_type (str): Type of the SQL element (e.g., "table", "view").
    :param prop_name (str): SQL-safe property name.
    :param domain_name (str): SQL name of the domain class of the property.
    :param prop_info (dict): Detailed property metadata (types, restrictions, disjointness, etc.).
    :param context_dict (dict): Global context with equivalent property mappings and metadata.
    :return (list[str]): List of SQL comments explaining the semantic behavior of the property.
    """
    g = graph
    column_comment = []
    
    prop = prop_info["uri"]
    equivalent_property_group = context_dict["equivalent_property_group"].get(prop)
    
    if prop_info["equivalentGroup"]:
        column_comment.append(
            f"{prop_name} is the representative for the equivalent properties: " 
            f"{', '.join(get_property_info(g, equivalent_prop, context_dict)['sqlName'] for equivalent_prop in equivalent_property_group)}\n"
        )
    
    
    if prop_info["reflexiveProperty"]:
        if prop_info["isSingleValued"]:
            column_comment.append(
                f"This column represents the reflexive attribute {prop_name}. "
                "Its value is always equal to the row''s ID. As a GENERATED "
                "column, it is computed automatically and cannot be manually set "
                "during INSERT operations.\n"
            )
        else:
            column_comment.append(
                f"{prop_name} is a reflexive property. Self-references are " 
                f"not stored in the table {domain_name}, but are included only in the " 
                f"view {table_name}_reflexive.\n"
            )

        
    if prop_info["irreflexiveProperty"]:
        column_comment.append(
            f"{prop_name} is a irreflexive property. That means that no " 
            f"self-references are allowed.\n"
        )
                        
    if prop_info["disjointWithGroups"]:
        disjoint_comment = []
        for group in prop_info["disjointWithGroups"]:
            if len(group) > 2:
                disjoint_comment.append(
                    f"{prop_name} is pairwise disjoint with " 
                    f"{', '.join({get_property_info(g, gr, context_dict).get(
                        "equivalentPropertyRepresentative") 
                        or get_property_info(g, gr, context_dict)["sqlName"] 
                        for gr in group if gr != prop})}."
                )
            else:
                disjoint_comment.append(
                    f"{prop_name} is disjoint with "
                    f"{', '.join({get_property_info(g, gr, context_dict).get(
                        "equivalentPropertyRepresentative") 
                        or get_property_info(g, gr, context_dict)["sqlName"] 
                        for gr in group if gr != prop})}."
                )
        disjoint_comment.append(
            "Note: Disjointness is not technically enforced. Violations for disjoint "
            "groups can be inspected via the view disjoint_properies_violations.\n")
        disjoint_comment = ["\n".join(disjoint_comment)]
        column_comment.extend(disjoint_comment)
    
    if prop_info["singleValueTypes"]:
        if "functional" in prop_info["singleValueTypes"] and not prop_info["isSingleValued"]:
            column_comment.append(
                f"Note: {prop_name} is declared as a owl:FunctionalProperty, "
                "but instance data contains multiple values per subject. Therefore, it is "
                "treated as a multi-value property."
            )
        else:
            column_comment.append(
                f"{prop_name} is a functional property with the following functional types: "
                f"{', '.join(prop_info['singleValueTypes'])}"
            )
    
    if prop_info["inversefunctionalTypes"]:
        if "inverse functional" in prop_info["inversefunctionalTypes"] and not prop_info["isInverseFunctional"]:
            column_comment.append(
                f"Note: {prop_name} is declared as a owl:InverseFunctionalProperty, "
                "but instance data contains multiple values per subject. Therefore, it is "
                "treated as a normal property."
            )
        elif prop_info["isInverseFunctional"]:
            column_comment.append(
                f"{prop_name} is a inverse functional property with the following "
                f"inverse functional types: {', '.join(prop_info['inversefunctionalTypes'])}"
            )
            
    return column_comment

In [1049]:
def table_definition_property_tables(graph, column_name, column_type, prop_info, 
                                     table_name, cls_mapping, context_dict, fk_table,
                                     subject_column, table_type = "property", primary_key = True):
    """
    Builds a full column definition for a property-related table.

    This helper function defines a single column (subject or object/value)
    used in property tables. It determines datatypes, constraints, foreign keys,
    and inline comments based on the property's OWL characteristics such as
    Functional, InverseFunctional, Reflexive, or Datatype restrictions.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param column_name (str): SQL column name to be created.
    :param column_type (str): Either "id" (subject) or "value" (object).
    :param prop_info (dict): Extracted ontology metadata for the property.
    :param table_name (str): Name of the SQL table to which the column belongs.
    :param cls_mapping (dict): Mapping of OWL classes to SQL tables.
    :param context_dict (dict): Global ontology context with property metadata.
    :param fk_table (str): Target table for foreign key linkage (domain or range table).
    :param table_type (str, optional): Element type (default "property").
    :param primary_key (bool, optional): If True, marks the column as part of the primary key.
    :return (tuple[dict, list[dict], list[dict]]):
        - Column definition dictionary.
        - Foreign key definition list.
        - Optional view definition list (e.g., for reflexive properties).
    """
    g = graph
    
    column_info = {}
    table_key_definition = []
    table_view_definition = []
    column_comments = []

    prop_name = prop_info["sqlName"]
    prop_domain = prop_info["rdfs:domain"]
    domain_name = cls_mapping.get(prop_domain, {}).get("name")
    
    
    column_info["attributeName"] = column_name
    
    if primary_key:
        column_info["primaryKey"] = True
    
    if fk_table:
        table_key_definition.append({
            "foreign_key_table": table_name,
            "foreign_key_column": column_name,
            "target_table": fk_table,
            "target_column": f"{fk_table}_id",
            })
        
    if column_type == "value":
        datatype = prop_info["recommendedDataType"]["SQLdatatype"]
        if isinstance(datatype, str):
            column_info["datatype"] = datatype
        elif isinstance(datatype, dict):
            column_info["datatype"] = datatype["sql_datatype"]
            if isinstance(datatype["restrictions"], dict):
                column_info["datatype_check"] = [
                    {"restriction": r, "value": v}
                    for r, v in datatype["restrictions"].items()
                ]
            else:
                column_info["datatype_check"] = datatype["restrictions"]
        
        if prop_info["checkConstraints"]:
            column_info["check"] = prop_info["checkConstraints"]
        if prop_info["isInverseFunctional"]:
            column_info["unique"] = True
        
        if prop_info["isSingleValued"]:
            if "key functional" in prop_info["singleValueTypes"]:
                column_info["unique"] = True
            if prop_info["reflexiveProperty"]:
                column_info["reflexiv"] = subject_column
            if prop_info["symmetricProperty"]:
                column_info["unique"] = True
            
        column_comments = table_definition_property_comments(
                g, table_name, table_type, prop_name, domain_name, prop_info, context_dict
            )
    
    elif column_type == "id":
        column_info["datatype"] = prop_info["subject_datatype"]

    column_info["comment"] = column_comments
    
    return column_info, table_key_definition, table_view_definition

##### Hauptfunktionen

In [1050]:
def table_definition_class(graph, cls, cls_mapping, context_dict):
    """
    EGenerates a complete table definition for a given ontology class.

    This function builds the relational structure for each OWL class by
    combining semantic metadata, inheritance (rdfs:subClassOf),
    equivalence (owl:equivalentClass), and disjointness constraints.

    It includes:
      - Primary key generation and datatype selection.
      - Trigger definitions for subclass propagation (Rule R4.4).
      - Attribute inclusion for all single-valued domain properties.
      - Semantic comments summarizing ontology relationships.

    :param graph (rdflib.Graph): RDFLib graph of the ontology.
    :param cls (rdflib.URIRef): Class URI to be transformed.
    :param cls_mapping (dict): Current mapping of OWL classes to SQL tables (extended in place).
    :param context_dict (dict): Global context dictionary with ontology metadata.
    :return (tuple):
        - Table definition dictionary (columns, annotations).
        - Updated cls_mapping.
        - Foreign key definition list.
        - Trigger definition list.
        - View definition list.
    """

    g = graph
    
    table_definition = {}

    columns = []
    table_comments = []


    foreign_key_definition = [] 
    trigger_definition = []
    view_definition = []

    sub_class_trigger_definition = None
    super_class_attributs = None

    cls_info = get_class_info(g, cls, context_dict)

    classes = set()
    super_classes = set()

    if cls not in cls_mapping and cls_info["implementedAs"] == "table":
        equivalent_classes = cls_info["equivalentGroup"]

        if equivalent_classes:
            classes = set(equivalent_classes)
            table_name = cls_info["equivalentClassRepresentative"]
            
            for equivalent_class in equivalent_classes:
                equivalent_class_info = get_class_info(g, equivalent_class, context_dict)
                if equivalent_class_info["rdfs:subClassOf"]:
                    super_classes = super_classes.union(equivalent_class_info["rdfs:subClassOf"])
                
                if equivalent_class == cls:
                    continue
                equivalent_view_defi = view_definition_equivalent_element(g, equivalent_class, table_name, "class", context_dict)
                view_definition.append(equivalent_view_defi)
            
            table_comments.append(
                f"Combined table for equivalent classes: " 
                f"{', '.join(get_class_info(g, equivalent_class, context_dict)['classKey'] for equivalent_class in equivalent_classes)}"
            )    
        else:
            table_name = cls_info["sqlName"]
            classes.add(cls)
            if cls_info["rdfs:subClassOf"]:
                super_classes = super_classes.union(cls_info["rdfs:subClassOf"])
        
        if cls_info["owl:hasKey"]:
            for key_group in cls_info["owl:hasKey"]:
                key_props = ", ".join(get_sql_name(prop, "property", context_dict) for prop in key_group)
                if len(key_group) == 1:
                    word = "property" 
                    word2 = "is"
                else: 
                    word = "properties"
                    word2 = "are"
                table_comments.append(
                    f"The {word} {key_props} {word2} a key of the class and uniquely identify each instance of the class."
                )
        
        id_type = cls_info["primaryKeyType"]

        pk_column = {
            "attributeName": f"{table_name}_id",
            "primaryKey": True,
        }
        if cls_info["checkConstraints"]:
            pk_column["check_class"] = cls_info["checkConstraints"]

        # if id_type == "uri":
        pk_column["datatype"] = "VARCHAR"
        pk_column["comment"] = ["Primary Key: Unique URI from the original Ontology"]
        # else:
        #     pk_column["datatype"] = "SERIAL"
        #     pk_column["comment"] = ["Primary Key: Automatic ID"]

        columns.append(pk_column)
        
        # columns.append({
        #     "attributeName": "label",
        #     "datatype": "VARCHAR",
        #     "comment": ["rdfs:Label"],
        # })

        if super_classes:
            sub_class_trigger_definition = trigger_definition_sub_classes(g, table_name, super_classes, context_dict)
            trigger_definition.extend(sub_class_trigger_definition)

        super_class_attributs = [
            {"attributeName": "source_sys", 
                "datatype": "VARCHAR(8)",
                "comment": ["Says if the entry is ether explicte or derived from the subtables of this table."],             
                "default": "explicit"
            },
            {"attributeName": "sub_refcount_sys", 
                "datatype": "INTEGER",
                "comment": ["Says how many direct subtables also contain this entry."],
                "default": 0
            }
        ]
        
        for clss in classes:
            if get_class_info(g, clss, context_dict)["deprecated"]:
                continue
            attributes = set()
            for prop in get_class_property_info(g, clss, context_dict).get("asDomainOf", []):
                prop_info = get_property_info(g, prop, context_dict)
                if prop_info["implementedAs"] not in ("functional relation", "functional relation (equivalent)"):
                    continue
                attributes.add(prop)
            view_name = None
            if clss != cls:
                view_name = get_class_info(g, clss, context_dict)["sqlName"]
            
            cls_mapping[clss] = {
                "name": table_name,
                "mapped": True if clss == cls else False,
                "attributes": attributes,
                "id_column": f"{table_name}_id",
                "view_name": view_name
            }
        
        for sub_class in cls_info["superClassOf"] or []:
            if get_class_info(g, sub_class, context_dict)["implementedAs"] == "in super table":
                cls_mapping[sub_class] = {
                    "name": table_name,
                    "mapped": False,
                    "attributes": attributes,
                    "id_column": f"{table_name}_id"
                }

        if cls_info["disjointWithGroups"]:
            disjoint_comment = []
            for group in cls_info["disjointWithGroups"]:
                if len(group) > 2:
                    disjoint_comment.append(
                        f"{table_name} is pairwise disjoint with " 
                        f"{', '.join({get_class_info(g, gr, context_dict).get(
                            "equivalentClassRepresentative") 
                            or get_class_info(g, gr, context_dict)["sqlName"] 
                            for gr in group if gr != cls})}."
                    )
                else:
                    disjoint_comment.append(
                        f"{table_name} is disjoint with "
                        f"{', '.join({get_class_info(g, gr, context_dict).get(
                            "equivalentClassRepresentative") 
                            or get_class_info(g, gr, context_dict)["sqlName"] 
                            for gr in group if gr != cls})}."
                    )
            disjoint_comment.append(
                "Note: Disjointness is not technically enforced. Violations for "
                "disjoint groups of two can be inspected via the view disjoint_classes_violations.\n")
            disjoint_comment = ["\n".join(disjoint_comment)]
            table_comments.extend(disjoint_comment)
                    
        if super_class_attributs:
            columns.extend(super_class_attributs)
            
        
        table_definition = {
            "tableName": table_name,
            "tableAnnotations": table_comments if table_comments else None, 
            "columns": columns,
            # "unique_column_combinations": unique_columns if unique_columns else None,
        }  

    return table_definition, cls_mapping, foreign_key_definition, trigger_definition, view_definition

Eigene Tabellen für Properties Allgemein

In [1051]:
def table_definition_properties(graph, prop, cls_mapping, prop_mapping, context_dict):
    """
    Builds the table definition for a property.

    :param graph (rdflib.Graph): RDF graph containing the ontology.
    :param prop (rdflib.URIRef): Property URI to be transformed.
    :param cls_mapping (dict): Mapping of OWL classes to SQL tables.
    :param prop_mapping (dict): Mapping of OWL properties to SQL tables/columns.
    :param context_dict (dict): Context dictionary with ontology metadata.
    :return (tuple):
        - Property table definition.
        - Foreign key definitions.
        - Updated property mapping.
        - Various View definitions.
    """
    g = graph
    equivalent_property_group = context_dict["equivalent_property_group"]
    
    table_definition = {}
    key_definition = []
    view_definition = []
    
    
    prop_info = get_property_info(g, prop, context_dict)
    
    if prop_info["implementedAs"] in ["multivalue relation", "functional relation"]:
                
        columns = []

        prop_domain = prop_info["rdfs:domain"]
        prop_domain_name = cls_mapping.get(prop_domain, {}).get("name")
        
        prop_range = prop_info["rdfs:range"]
        prop_range_name = cls_mapping.get(prop_range, {}).get("name")
        
        table_name = f"{prop_domain_name}_{prop_info["sqlName"]}_rel"
        prop_name = prop_info["sqlName"]
        
        subject_column = f"{prop_domain_name}_id"
        object_column = prop_info["sqlName"]
        
        for props in equivalent_property_group.get(prop, [prop]):
            if props not in prop_mapping:
                view_name = None
                equi_prop_info = get_property_info(g, props, context_dict)
                if equi_prop_info["deprecated"]:
                    continue
                if props != prop:
                    name = equi_prop_info["sqlName"]
                    view_name = f"{prop_domain_name}_{name}"
                
                prop_mapping[props] = {
                    "name": table_name,
                    "attributeName": prop_name,
                    "table": table_name,
                    "implementedAs": "multi value relation",
                    "domain": prop_domain_name,
                    "range": prop_range_name,
                    "mapped": True if props == prop else False,
                    "subject_column": subject_column,
                    "object_column": object_column,
                    "view_name": view_name
                }
            if props != prop:
                equivalent_view_definition = view_definition_equivalent_element(g, props, table_name, "property", context_dict, prop_domain_name)
                view_definition.append(equivalent_view_definition)
            
            if equi_prop_info["implementedAs"] == "functional relation" and prop_info["reflexiveProperty"]:
                prop_mapping[props]["reflexive"] = True
            
              
        for sub_prop in prop_info.get("superPropertyOf") or []:
            sub_prop_info = get_property_info(g, sub_prop, context_dict)
            if sub_prop_info["implementedAs"] == "in super property":
                prop_mapping[sub_prop] = {
                    "name": table_name,
                    "attributeName": prop_name,
                    "table": table_name,
                    "implementedAs": "functional relation",
                    "domain": prop_domain_name,
                    "range": prop_range_name,
                    "mapped": False,
                    "subject_column": subject_column,
                    "object_column": object_column
                }
                if sub_prop_info["implementedAs"] == "functional relation" and prop_info["reflexiveProperty"]:
                    prop_mapping[sub_prop]["reflexive"] = True

        column_info_subject, key_definition_subject, view_definition_subject = table_definition_property_tables(g, subject_column, "id", prop_info, table_name, cls_mapping, context_dict, prop_domain_name, subject_column)
        columns.append(column_info_subject)
        key_definition.extend(key_definition_subject)
        view_definition.extend(view_definition_subject)
        
        column_info_object, key_definition_object, view_definition_object = table_definition_property_tables(g, object_column, "value", prop_info, table_name, cls_mapping, context_dict, prop_range_name, object_column)
        columns.append(column_info_object)
        key_definition.extend(key_definition_object)
        view_definition.extend(view_definition_object)

        columns.extend(
            [{"attributeName": "source_sys", 
                "datatype": "VARCHAR(8)",
                "comment": ["Says if the entry is ether explicte or derived from the subtables of this table."],             
                "default": "explicit",
            },
            {"attributeName": "sub_refcount_sys", 
                "datatype": "INTEGER",
                "comment": ["Says how many subtables also contain this entry."],
                "default": 0
            }]
        )

        if prop_info["implementedAs"] == "multivalue relation":
            table_comments = [
                f"This table stores the multi-value property {prop_name} where a single subject can be linked to multiple "
                "objects. Each row represents one such subject-object pair. These tables are generated "
                "whenever a property allows more than one value per subject."
            ]
            if prop_info["reflexiveProperty"]:
                view_definition.extend(view_definition_reflexive_property(g, prop_name, table_name, prop_domain_name))
        else:
            table_comments = [
                f"This table stores the functional {prop_name} property where a subject can only be linked to one "
                "object. Each row represents one such subject-object pair."
            ]
                
        table_definition = {
            "tableName": table_name,
            "columns": columns,
            "tableAnnotations": table_comments        
        }
        
    return table_definition, key_definition, prop_mapping, view_definition

In [1052]:
def table_definition_inverse_properties(graph, prop, cls_mapping, prop_mapping, context_dict):
    """
    Builds the table definition for inverse properties.

    For each pair of properties connected via 'owl:inverseOf',
    this function generates a shared relational table that represents
    both directions simultaneously. Each subject-object pair implicitly
    implies its inverse counterpart, ensuring data consistency.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param prop (rdflib.URIRef): Property URI that participates in an inverseOf relation.
    :param cls_mapping (dict): Class mapping (used to resolve domains and ranges).
    :param prop_mapping (dict): Property mapping (updated in place).
    :param context_dict (dict): Global ontology context.
    :return (tuple):
        - Inverse property table definition.
        - Foreign key definitions.
        - Updated property mapping.
        - Various view definitions.
    """
    
    g = graph 
    
    inverse_table_definition = {}
    inverse_key_definition = []
    inverse_view_definition = []
    
    
    prop_info = get_property_info(g, prop, context_dict)
    
    if prop not in prop_mapping and prop_info["implementedAs"] == "inverse relation (representative)":
        columns = []
        table_comments = []
        checks = []
        
        rep_prop = prop_info["inverseMap"][0]
        rep_prop_info = get_property_info(g, rep_prop, context_dict)
        rep_prop_name = rep_prop_info["sqlName"]
        
        inv_prop = prop_info["inverseMap"][1]
        inv_prop_info = get_property_info(g, inv_prop, context_dict)
        inv_prop_name = inv_prop_info["sqlName"]
        
        rep_prop_domain = rep_prop_info["rdfs:domain"] # range of inv
        rep_prop_domain_name = cls_mapping.get(rep_prop_domain, {}).get("name")
        inv_prop_domain = inv_prop_info["rdfs:domain"] # range of rep
        inv_prop_domain_name = cls_mapping.get(inv_prop_domain, {}).get("name")

        
        table_name = f"{rep_prop_domain_name}_{rep_prop_name}_rel_inv"
        if rep_prop_domain:
            object_column = f"{inv_prop_domain_name}_id_{rep_prop_name}"
        else:
            object_column = rep_prop_name
        
        if inv_prop_domain:
            subject_column = f"{rep_prop_domain_name}_id_{inv_prop_name}"
        else:
            subject_column = inv_prop_name
        
        column_names = [subject_column, object_column]

        for props in rep_prop_info.get("equivalentGroup") or [rep_prop]:
            if props not in prop_mapping:
                equi_prop_info = get_property_info(g, props, context_dict)
                if equi_prop_info["deprecated"]:
                    continue
                view_name = None
                if props != rep_prop:
                    name = equi_prop_info["sqlName"]
                    view_name = f"{rep_prop_domain_name}_{name}"
                
                prop_mapping[props] = {
                    "name": table_name,
                    "attributeName": column_names,
                    "table": table_name,
                    "implementedAs": "inverse relation",
                    "domain": rep_prop_domain_name,
                    "range": inv_prop_domain_name, 
                    "mapped": True if props == rep_prop else False,
                    "subject_column": subject_column,
                    "object_column": object_column,
                    "view_name": view_name
                }
            
            if props != rep_prop:
                equivalent_view_definition = view_definition_equivalent_element(g, props, table_name, "property", context_dict, rep_prop_domain_name)
                inverse_view_definition.append(equivalent_view_definition)
            if equi_prop_info["implementedAs"] == "functional relation" and rep_prop_info["reflexiveProperty"]:
                prop_mapping[sub_prop]["reflexive"] = True
            
        for sub_prop in rep_prop_info.get("superPropertyOf") or []:
            sub_prop_info = get_property_info(g, sub_prop, context_dict)
            if sub_prop_info["implementedAs"] == "in super property":
                prop_mapping[sub_prop] = {
                    "name": table_name,
                    "attributeName": column_names,
                    "table": table_name,
                    "implementedAs": "inverse relation",
                    "domain": rep_prop_domain_name,
                    "range": inv_prop_domain_name, 
                    "mapped": False,
                    "subject_column": subject_column,
                    "object_column": object_column
                }
                if sub_prop_info["implementedAs"] == "functional relation" and rep_prop_info["reflexiveProperty"]:
                    prop_mapping[sub_prop]["reflexive"] = True
    
        for props in inv_prop_info.get("equivalentGroup") or [inv_prop]:
            if props not in prop_mapping:
                equi_prop_info = get_property_info(g, props, context_dict)
                if equi_prop_info["deprecated"]:
                    continue
                view_name = None
                if props != inv_prop:
                    name = equi_prop_info["sqlName"]
                    view_name = f"{inv_prop_domain_name}_{name}"
                    
                prop_mapping[props] = {
                    "name": table_name,
                    "attributeName": column_names,
                    "table": table_name,
                    "implementedAs": "inverse relation",
                    "domain": inv_prop_domain_name,
                    "range": rep_prop_domain_name,
                    "mapped": True if props == inv_prop else False,
                    "subject_column": object_column,
                    "object_column": subject_column,
                    "view_name": view_name
                }
            if props != inv_prop:
                equivalent_view_definition = view_definition_equivalent_element(g, props, table_name, "property", context_dict, inv_prop_domain_name)
                inverse_view_definition.append(equivalent_view_definition)
            if equi_prop_info["implementedAs"] == "functional relation" and inv_prop_info["reflexiveProperty"]:
                prop_mapping[sub_prop]["reflexive"] = True
                
        for sub_prop in inv_prop_info.get("superPropertyOf") or []:
            sub_prop_info = get_property_info(g, sub_prop, context_dict)
            if sub_prop_info["implementedAs"] == "in super property":
                prop_mapping[sub_prop] = {
                    "name": table_name,
                    "attributeName": column_names,
                    "table": table_name,
                    "implementedAs": "inverse relation",
                    "domain": inv_prop_domain_name,
                    "range": rep_prop_domain_name,
                    "mapped": True if props == inv_prop else False,
                    "subject_column": object_column,
                    "object_column": subject_column
                }
                
                if sub_prop_info["implementedAs"] == "functional relation" and inv_prop_info["reflexiveProperty"]:
                    prop_mapping[sub_prop]["reflexive"] = True
        
        column_info_subject, key_definition_subject, view_definition_subject = table_definition_property_tables(g, column_names[0], "value", inv_prop_info, table_name, cls_mapping, context_dict, rep_prop_domain_name, object_column)
        inverse_key_definition.extend(key_definition_subject)
        inverse_view_definition.extend(view_definition_subject)
        
        column_info_object, key_definition_object, view_definition_object = table_definition_property_tables(g, column_names[1], "value", rep_prop_info, table_name, cls_mapping, context_dict, inv_prop_domain_name, subject_column)
        inverse_key_definition.extend(key_definition_object)
        inverse_view_definition.extend(view_definition_object)

        columns.append(column_info_subject)
        columns.append(column_info_object)
        
        inverse_view_definition.append(view_definition_inverse_property(g, inv_prop, table_name, context_dict, inv_prop_domain_name, object_column, subject_column))
        
        columns.extend(
            [{"attributeName": "source_sys", 
                "datatype": "VARCHAR(8)",
                "comment": ["Says if the entry is ether explicte or derived from the subtables of this table."],             
                "default": "explicit",
            },
            {"attributeName": "sub_refcount_sys", 
                "datatype": "INTEGER",
                "comment": ["Says how many subtables also contain this entry."],
                "default": 0
            }]
        )
        
        inverse_table_definition = {
            "tableName": table_name,
            "columns": columns,
            "tableAnnotations": [
                f"This table represents the inverseOf relation of {rep_prop_name} and {inv_prop_name}. It stores both directions "
                "of a Relation. Each subject-object-pair automatically implies its "
                "inverse to ensure consistency with the ontologys inverseOf relation."]
        }
    
        if rep_prop_info["reflexiveProperty"]:
            inverse_view_definition.extend(view_definition_reflexive_property(g, column_names, table_name, [rep_prop_domain_name, inv_prop_domain_name]))
        
    return inverse_table_definition, inverse_key_definition, prop_mapping, inverse_view_definition
        
        

### Einträge

#### Andere Tabellen

##### Klassen

In [1053]:
def entry_definition_all_classes(graph, class_mapping, context_dict):
    """    
    Generates the row entries for the system table 'all_classes_sys'.

    Each ontology class is represented by one record containing its URI,
    SQL table name, equivalence representative (if any), and implementation type
    (e.g., table, view, or merged class).

    :param graph (rdflib.Graph): RDFLib graph of the ontology.
    :param class_mapping (dict): Mapping of OWL classes to SQL tables.
    :param context_dict (dict): Global context containing equivalent class groups and metadata.
    :return (list[dict]): List of table rows for 'all_classes_sys', e.g.:
        [
            {
                "table_name": "award",
                "class_uri": "http://example.org/ontology/Award",
                "equivalent_with_class": "prize",
                "implemented_as": "table"
            },
            ...
        ]
    """
    g = graph
    
    all_classes_definition = []
    all_classes = context_dict["all_classes"]

    for cls in class_mapping:
        
        cls_info = get_class_info(g, cls, context_dict)
        
        all_key_props = []
        for key_group in cls_info["owl:hasKey"] or []:
            key_props = ", ".join(get_sql_name(prop, "property", context_dict) for prop in key_group)
            all_key_props.append(key_props)
        
        all_disjoint_groups = []
        disjoint_with = context_dict["classes_disjoint_with"].get(cls, set())
        for disjoint_group in disjoint_with:
            if isinstance(disjoint_group, URIRef):
                names = get_sql_name(disjoint_group, "class", context_dict)
            else:
                names = ", ".join(get_sql_name(dg, "class", context_dict) for dg in disjoint_group)
            all_disjoint_groups.append(names)
            
        # Werte-Grundgerüst
        values = {
            "class_id": cls_info["sqlName"],
            "class_uri": cls,
            "sql_name": class_mapping[cls]["name"],
            "implemented_as": cls_info["implementedAs"],
        }
        if all_key_props:
            values["has_key_properties"] = f"[{'], ['.join(all_key_props)}]"
        if all_disjoint_groups:
            values["is_disjoint_with"] = f"[{'], ['.join(all_disjoint_groups)}]"
        
        # Gesamtstruktur erzeugen
        all_classes_definition.append({
            "table": "all_classes_sys",
            "values": values
        })
    
    for cls in all_classes:
        if cls in class_mapping:
            continue
        cls_info = get_class_info(g, cls, context_dict)
        if cls_info["deprecated"]:
            continue
            
        values = {
            "class_id": cls_info["sqlName"],
            "class_uri": cls,
            "implemented_as": cls_info["implementedAs"],
        }
        all_classes_definition.append({
            "table": "all_classes_sys",
            "values": values
        })
    
    return all_classes_definition

In [1054]:
def entry_definition_sub_class_of(graph, class_mapping, context_dict):
    """
    Generates the row entries for the system table 'sub_class_relationship_table_sys'.

    Each explicit 'rdfs:subClassOf' relationship between two ontology classes
    results in one record linking their corresponding SQL table names.
    These entries are later used for trigger generation and semantic propagation.

    :param graph (rdflib.Graph): RDFLib graph of the ontology.
    :param class_mapping (dict): Mapping of OWL classes to SQL tables.
    :param context_dict (dict): Global ontology context.
    :return (list[dict]): List of subclass-superclass entries, e.g.:
        [
            {
                "table": "sub_class_relationship_table_sys",
                "values": {
                    "sub_class": "cast_and_crew",
                    "super_class": "movie_production"
                }
            },
            ...
        ]
    """
    g = graph

    sub_class_of_definition = []
    super_clss = context_dict["sub_class_of_dist"]

    for cls in class_mapping:
        cls_info = get_class_info(g, cls, context_dict)
        
        super_classes = {x for x, dist in super_clss.get(cls, {}).items() if dist == 1}
        for super_cls in super_classes:
            super_class_info = get_class_info(g, super_cls, context_dict)
            values = {
                "sub_class": cls_info["sqlName"],
                "super_class": super_class_info["sqlName"]
            }
    
            sub_class_of_definition.append({
                "table": "sub_class_relationship_table_sys",
                "values": values,
            })
    
    return sub_class_of_definition

In [1055]:
def entry_definition_equivalent_class(graph, class_mapping, context_dict):
    """
    Generates the row entries for the system table 'equivalent_class_table_sys'.

    Each 'owl:equivalentClass' pair in the ontology is stored as a row linking
    two semantically equivalent classes. This ensures that equivalent classes
    can be treated as a single conceptual entity in the relational schema.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param class_mapping (dict): Mapping of OWL classes to SQL table names.
    :param context_dict (dict): Global context containing equivalent class groups.
    :return (list[dict]): List of equivalence entries for 'equivalent_class_table_sys'.
    """
    g = graph
    equivalent_definition = []
    equivalent_group = context_dict["equivalent_class_group"]
    seen = set()
    
    i = 1    
    
    for cls in class_mapping:
        if cls in seen:
            continue        
        equivalent_group_cls = equivalent_group.get(cls, [])

        all_equi_pairs = [(a,b) for a in equivalent_group_cls for b in equivalent_group_cls if a != b]
        
        for sub, obj in all_equi_pairs:
            seen.add(sub)
            seen.add(obj)
            sub_cls_info = get_class_info(g, sub, context_dict)
            obj_cls_info = get_class_info(g, obj, context_dict)
            
            values = {
                "class_id": sub_cls_info["sqlName"],
                "equivalent_class": obj_cls_info["sqlName"],
                "group_id": f"E{i}"
            }
    
            equivalent_definition.append({
                "table": "equivalent_class_table_sys",
                "values": values,
            })
        if all_equi_pairs:
            i += 1
        
    return equivalent_definition

##### Properties

In [1056]:
def entry_definition_all_properties(graph, prop_mapping, context_dict):
    """
    Generates the row entries for the system table 'all_properties_sys'.

    Each ontology property is represented by a record containing its URI,
    SQL implementation (table, attribute, or view), and equivalence group representative.
    This table consolidates all properties used in the relational schema.

    :param graph (rdflib.Graph): RDFLib graph of the ontology.
    :param prop_mapping (dict): Mapping of properties to SQL tables or attributes.
    :param context_dict (dict): Global context with equivalent property groups and metadata.
    :return (list[dict]): List of table rows for 'all_properties_sys', e.g.:
        [
            {
                "property_name": "has_award",
                "property_uri": "http://example.org/ontology/hasAward",
                "equivalent_with_property": "won_award",
                "implemented_as": "table"
            },
            ...
        ]
    """
    g = graph

    all_properties_definition = []

    for prop in prop_mapping:
        if isinstance(prop, BNode):
            continue
        
        prop_info = get_property_info(g, prop, context_dict)
        
        domain_name = get_sql_name(prop_info["rdfs:domain"], "class", context_dict)
        range_name = get_sql_name(prop_info["rdfs:range"], "class", context_dict)
        
        datatype = prop_info["recommendedDataType"]["SQLdatatype"]
        if isinstance(datatype, str):
            object_datatype = datatype
        if isinstance(datatype, dict):
            object_datatype = datatype["sql_datatype"]
        
        all_disjoint_groups = []
        disjoint_with = context_dict["properties_disjoint_with"].get(prop, set())
        for disjoint_group in disjoint_with:
            if isinstance(disjoint_group, URIRef):
                names = get_sql_name(disjoint_group, "property", context_dict)
            else:
                names = ", ".join(get_sql_name(dg, "property", context_dict) for dg in disjoint_group)
            all_disjoint_groups.append(names)
        
        all_chain_groups = []
        property_chain_axiom = prop_info["propertyChainAxiom"] or []
        for chain_group in property_chain_axiom:
            if isinstance(chain_group, URIRef):
                names = get_sql_name(chain_group, "property", context_dict)
            else:
                names = ", ".join(get_sql_name(cg, "property", context_dict) for cg in chain_group)
            all_chain_groups.append(names)
        # Werte-Grundgerüst
        values = {
            "property_id": prop_info["sqlName"],
            "sql_name": prop_mapping.get(prop, {}).get("name"),
            "property_uri": prop,
            "implemented_as": prop_info["implementedAs"],
            "subject_column": prop_mapping.get(prop, {}).get("subject_column"),
            "object_column": prop_mapping.get(prop, {}).get("object_column"),
            "object_datatype": object_datatype,
            }
        
        if prop_info["isSingleValued"]:
            values["functional"] = prop_info["isSingleValued"]
        if prop_info["isInverseFunctional"]:
            values["inverse_functional"] = prop_info["isInverseFunctional"]
        if all_disjoint_groups:
            values["is_disjoint_with"] = f"[{'], ['.join(all_disjoint_groups)}]"
        if all_chain_groups:
            values["property_chain_axiom"] = f"[{'], ['.join(all_chain_groups)}]"
        if range_name:
            values["range_class"] = range_name
        if domain_name:
            values["domain_class"] = domain_name
        
        for char in ["symmetricProperty", "asymmetricProperty", "transitiveProperty", "reflexiveProperty", "irreflexiveProperty"]:
            if prop_info[char]:
                values[clean_value(char)] = prop_info[char]

        all_properties_definition.append({
            "table": "all_properties_sys",
            "values": values
        })
    
    return all_properties_definition

In [1057]:
def entry_definition_sub_property_of(graph, prop_mapping, context_dict):
    """
    Generates the row entries for the system table 'sub_property_relationship_table_sys'.

    Each 'rdfs:subPropertyOf' relationship between two ontology properties
    becomes a record linking the SQL representations of the sub- and super-property.
    These entries guide the propagation of property data via SQL triggers.

    :param graph (rdflib.Graph): RDFLib graph of the ontology.
    :param prop_mapping (dict): Mapping of OWL properties to SQL tables.
    :param context_dict (dict): Global ontology context.
    :return (list[dict]): List of subclass-superclass property entries, e.g.:
        [
            {
                "table": "sub_property_relationship_table_sys",
                "values": {
                    "sub_property": "hasProducer",
                    "super_property": "hasCrewMember"
                }
            },
            ...
        ]
    """
    g = graph

    sub_property_of_definition = []
    super_props = context_dict["sub_property_of_dist"]

    for prop in prop_mapping:
        prop_info = get_property_info(g, prop, context_dict)

        super_properties = {x for x, dist in super_props.get(prop, {}).items() if dist == 1}
        for super_prop in super_properties:
            super_property_info = get_property_info(g, super_prop, context_dict)
            values = {
                "sub_property": prop_info["sqlName"],
                "super_property": super_property_info["sqlName"], 
                
            }
    
            sub_property_of_definition.append({
                "table": "sub_property_relationship_table_sys",
                "values": values,
            })
    
    return sub_property_of_definition

In [1058]:
def entry_definition_equivalent_property(graph, prop_mapping, context_dict):
    """
    Generates the row entries for the system table 'equivalent_property_table_sys'.

    Each 'owl:equivalentProperty' statement is recorded as a row connecting
    two semantically equivalent properties. This enables unified handling of
    equivalent properties within the relational model.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param prop_mapping (dict): Mapping of properties to SQL structures.
    :param context_dict (dict): Global context containing equivalence groups.
    :return (list[dict]): List of equivalence entries for 'equivalent_property_table_sys'.
    """
    g = graph
    equivalent_definition = []
    equivalent_group = context_dict["equivalent_property_group"]
    seen = set()
    
    i = 1    
    
    for prop in prop_mapping:
        if prop in seen:
            continue        
        equivalent_group_prop = equivalent_group.get(prop, [])

        all_equi_pairs = [(a,b) for a in equivalent_group_prop for b in equivalent_group_prop if a != b]
        
        for sub, obj in all_equi_pairs:
            seen.add(sub)
            seen.add(obj)
            sub_prop_info = get_property_info(g, sub, context_dict)
            obj_prop_info = get_property_info(g, obj, context_dict)
            
            values = {
                "property_id": sub_prop_info["sqlName"],
                "equivalent_property": obj_prop_info["sqlName"],
                "group_id": f"E{i}"
            }
    
            equivalent_definition.append({
                "table": "equivalent_property_table_sys",
                "values": values,
            })
        if all_equi_pairs:
            i += 1
    
    return equivalent_definition

##### Allgemein

In [1059]:
def entry_definition_annotation_table(graph, cls_mapping, prop_mapping, context_dict):
    """
    Generates the row entries for the annotation table 'ontology_annotations_meta'.

    Each record corresponds to an annotation triple extracted from the ontology,
    storing the annotated element, the annotation property, its value, and provenance.
    This table acts as a metadata catalog for the generated schema.

    :param graph (rdflib.Graph): RDF graph of the ontology.
    :param cls_mapping (dict): Mapping of ontology classes to SQL tables.
    :param prop_mapping (dict): Mapping of ontology properties to SQL names.
    :param context_dict (dict): Global context containing annotation mappings.
    :return (list[dict]): List of annotation records for 'ontology_annotations_meta'.
    """
    g = graph
    
    annotation_map = context_dict["annotation_map"]
    unique_names = context_dict["unique_names_by_group"]
    
    all_classes = context_dict["all_classes"]
    all_properties = context_dict["all_properties"]
    all_individuals = context_dict["all_individuals"]
    
    all_annotation_props = context_dict["all_annotation_properties"]
    
    all_entry_definition = []
 
    def get_entry_defi(all_elements, mapping, elem_type, all_entry_definition):
        for elem in all_elements:
            elem_annotations = annotation_map[elem_type].get(elem)
            if elem_annotations:
                for anno_prop, value_set in elem_annotations.items():
                    if elem_type == "individual":
                        subject_name = unique_names.get(elem_type).get(elem)
                    else:
                        subject_name = mapping.get(elem, {}).get("view_name", None) or mapping.get(elem, {}).get("name")
                    for single_value in value_set:
                        single_value = str(single_value).replace("'", "''")
                        entry = {
                            "table": "_annotations_meta",
                            "values": {
                                "subject_name": subject_name,
                                "subject_type": elem_type,
                                "subject_id": get_sql_name(elem, elem_type, context_dict),
                                "annotation_property": remove_prefix(anno_prop),
                                "annotation_uri": anno_prop,
                                "annotation_value": single_value
                            }
                        }
                        all_entry_definition.append(entry)

        return all_entry_definition
    
    all_entry_definition = get_entry_defi(all_classes, cls_mapping, "class", all_entry_definition)
    all_entry_definition = get_entry_defi(all_properties, prop_mapping, "property", all_entry_definition)
    all_entry_definition = get_entry_defi(all_individuals, {}, "individual", all_entry_definition)
    
    return all_entry_definition

In [1060]:
def entry_definition_has_key_table(graph, class_mapping, context_dict):
    """
    Generates table entries for the system table storing owl:hasKey constraints.

    For each class that defines one or more key properties via owl:hasKey,
    this function creates the corresponding row entries referencing the
    SQL identifiers of the class and all participating properties.

    :param graph (rdflib.Graph): RDFLib graph of the ontology.
    :param class_mapping (dict): Mapping from class URI → class metadata, including SQL name and id.
    :param context_dict (dict): Ontology context containing hasKey information and property mappings.

    :return: List[dict] - Each entry representing one (class, property) pair.
    """
    g = graph
    
    has_key_definition = []

    
    
    for cls in class_mapping:
        i = 1
        cls_info = get_class_info(g, cls, context_dict)
        
        for key_group in cls_info["owl:hasKey"] or []:
            for prop in key_group:
                prop_info = get_property_info(g, prop, context_dict)
                values = {
                    "class_id": cls_info["sqlName"],
                    "key_property_id": prop_info["sqlName"],
                    "key_group": f"{cls_info["sqlName"]}-Key {i}"
                }
        
                has_key_definition.append({
                    "table": "_class_keys_meta",
                    "values": values,
                })
            i += 1
    
    return has_key_definition

In [1061]:
def entry_definiton_deprecated_elements(graph, context_dict):
    """
    Generates table entries for all deprecated ontology elements.

    :param graph (rdflib.Graph): RDFLib graph of the ontology.
    :param context_dict (dict): Ontology context containing lists or sets of deprecated classes/properties.

    :return: List[dict] - One entry per deprecated class or property.
    """
    g = graph
    
    all_properties = context_dict["all_properties"]
    all_classes = context_dict["all_classes"]
    
    deprecated_definition = []
    
    def get_deprecated_infos(all_elements, type_):
        for elem in all_elements:
            if type_ == "class":
                elem_info = get_class_info(g, elem, context_dict)
            else:
                elem_info = get_property_info(g, elem, context_dict)
            
            if not elem_info["deprecated"]:
                continue
            values = {
                "element_id": elem_info["sqlName"], 
                "element_uri": elem, 
                "element_type": type_
            }
            deprecated_definition.append({
                "table": "deprecated_classes_and_properties_sys",
                "values": values
            })
        return deprecated_definition

    deprecated_definition = get_deprecated_infos(all_properties, "property")
    deprecated_definition =get_deprecated_infos(all_classes, "class")
    return deprecated_definition
    

#### Klassen- / Propertytabellen

Mappt Individuen an Tabellen und ihre Values an die richtigen Attribute.

In [1062]:
def instance_definition(graph, cls_mapping, prop_mapping, context_dict):
    """
    Generates all instance entries for ontology-derived class and property tables.

    This function creates the actual data rows to populate the relational schema.
    For each class implemented as a table and for each mapped property, the
    corresponding individuals and property assertions are translated into SQL rows.

    :param graph (rdflib.Graph): RDF graph containing ontology instances.
    :param cls_mapping (dict): Mapping of OWL classes to SQL tables.
    :param prop_mapping (dict): Mapping of OWL properties to SQL structures.
    :param context_dict (dict): Global context with all ontology metadata.
    :return (list[dict]): List of instance records, e.g.:
        [
            {
                "table": "movie",
                "values": {
                    "id": "Q44578",
                    "label": "Titanic",
                    "sub_refcount": 2,
                    "wiki_data_id": "Q44578"
                },
                "annotations": {...}
            },
            ...
        ]
    """
    g = graph
    
    id_type = context_dict["primary_key_type"]
    all_classes = context_dict["all_classes"]
    
    instances = []
    inserted_instances = set()

    for s, o in g.subject_objects(RDF.type):
        if o not in all_classes:
            continue
        if s in inserted_instances or not isinstance(s, URIRef):
            continue
        inserted_instances.add(s)

        cls_uris = set(g.objects(s, RDF.type))

        seen_tables = []

        for cls in cls_uris:
            if not cls or cls not in cls_mapping or not cls_mapping.get(cls, {}).get("mapped"):
                continue  # ungemappt
            
            table_name = cls_mapping.get(cls, {}).get("name")
            
            if table_name in seen_tables:
                continue
            seen_tables.append(table_name)
           
            values = {}

            # ID setzen
            # if id_type == "uri":
            values[f"{table_name}_id"] = get_sql_name(s, "individual", context_dict)

            # Label setzen
            # label = g.value(s, RDFS.label)
            # if label:
                # values["label"] = remove_prefix(label).replace("'", "''")
            
            # initial sub_refcount_sys setzten
            if get_class_info(g, cls, context_dict)["sub_refcount"]:
                all_sub_refcount = get_class_info(g, cls, context_dict)["sub_refcount"]
                values["sub_refcount_sys"] = all_sub_refcount.get(s, 0)
                
            instance_data = {
                "table": table_name,
                "values": values,
            }

            instances.append(instance_data)

    return instances


Mappt Individuen an die Multivalue Tabellen.

In [1063]:
def instance_definition_properties(graph, cls_mapping, prop_mapping, context_dict):
    """
    Generates all instance entries for property tables.

    :param graph (rdflib.Graph): RDFLib graph containing instance data.
    :param cls_mapping (dict): Mapping of ontology classes to SQL tables.
    :param prop_mapping (dict): Mapping of ontology properties to SQL tables.
    :param context_dict (dict): Global ontology context.
    :return (list[dict]): List of instance entries for functional property tables.
    """
    g = graph
    
    id_type = context_dict["primary_key_type"]
    sub_props = context_dict["super_property_of_dist"]
    
    instances = []
      
    for prop in prop_mapping:
        mapping = prop_mapping.get(prop)
        
        if mapping.get("mapped") and mapping.get("implementedAs") in ["multi value relation", "functional relation" ]:
            table_name = mapping["table"]
            
            insertet_instances = set()
            
            explicit_pairs = list(g.subject_objects(prop))
            derived_pairs = []
            
            sub_properties = {x for x, dist in sub_props.get(prop, {}).items() if dist == 1}
            for sub_prop in sub_properties:
                derived_pairs.extend(list(g.subject_objects(sub_prop)))
        
            all_pairs = explicit_pairs + derived_pairs
            sub_refcount = Counter(derived_pairs)
            
            for s, o in explicit_pairs:
                if (s,o) in insertet_instances:
                    continue
                if isinstance(s, BNode) or (s, RDF.type, OWL.Ontology) in g:
                    continue
                
                insertet_instances.add((s,o))
                
                values = {}
                if id_type == "uri":
                    values[f"{mapping['subject_column']}"] = get_sql_name(s, "individual", context_dict)
                
                if not prop_mapping.get("reflexive", set()):
                    value_column = mapping.get("object_column")
                    if isinstance(o, (Literal, URIRef)):
                        value_clean = remove_prefix(o).replace("'", "''")
                    else:
                        value_clean = str(o).replace("'", "''")
                    values[value_column] = value_clean
                
                pair_sub_refcount = sub_refcount.get((s,o)) or 0
                values["sub_refcount_sys"] = pair_sub_refcount

                instance_data = {
                    "table": table_name,
                    "values": values,
                }   
                
                instances.append(instance_data)

    return instances

In [1064]:
def instance_definition_inverse_properties(graph, cls_mapping, prop_mapping, context_dict):
    """
    Generates instance data for properties implemented as inverse relations.

    For every property pair connected via 'owl:inverseOf', this function extracts
    both directions of the corresponding RDF triples and creates table entries
    representing subject-object and object-subject pairs.

    :param graph (rdflib.Graph): RDFLib graph containing ontology instance data.
    :param cls_mapping (dict): Mapping of ontology classes to SQL tables.
    :param prop_mapping (dict): Mapping of ontology properties to SQL structures.
    :param context_dict (dict): Global context with inverse property definitions.
    :return (list[dict]): List of instance records for inverse property tables.
    """
    g = graph
    
    id_type = context_dict["primary_key_type"]
    sub_props = context_dict["super_property_of_dist"]
    
    instances = []
    seen = set()
    
    # for s, p, o in g:
    for prop in prop_mapping:
        mapping = prop_mapping.get(prop, {})
        if mapping.get("name") in seen:
            continue
        
        if mapping and "inverse relation" in mapping.get("implementedAs", []):
            seen.add(mapping.get("name"))
            
            insertet_instances = set()
            
            table_name = mapping["table"]
            subject = get_property_info(g, prop, context_dict)["inverseMap"][0]
            object = get_property_info(g, prop, context_dict)["inverseMap"][1]

            explicit_pairs = list(g.subject_objects(subject))
            derived_pairs = []
            # id_column = mapping["domain"]
            subject_prop = f"{mapping.get("attributeName")[0]}"
            object_prop = f"{mapping.get("attributeName")[1]}"

            sub_properties = {x for x in sub_props.get(subject, set())}
            for sub_prop in sub_properties:
                derived_pairs.extend(list(g.subject_objects(sub_prop)))
            
            all_pairs = explicit_pairs + derived_pairs
            sub_refcount = Counter(all_pairs)
            
            for s, o in explicit_pairs:
                if (s,o) in insertet_instances:
                    continue
                if isinstance(s, BNode):
                    continue
                insertet_instances.add((s,o))
                
                values = {}
                if id_type == "uri":
                    values[subject_prop] = get_sql_name(s, "individual", context_dict)
                    values[object_prop] = get_sql_name(o, "individual", context_dict)
                
                pair_sub_refcount = sub_refcount.get((s,o))
                values["sub_refcount_sys"] = pair_sub_refcount
            
                instance_data = {
                    "table": table_name,
                    "values": values,
                }

                instances.append(instance_data)

    return instances

### SQL Erstellung

#### Hilfsfunktionen

Generiert aus generate_table_info_for_class() SQL-Code

In [1065]:
def sql_generate_datatype_check(column_name, datatype_definition):
    """
    Converts OWL/RDFS datatype facet restrictions into SQL CHECK constraints.

    Supports constraints such as 'xsd:minInclusive', 'xsd:maxInclusive',
    'xsd:minLength', 'xsd:maxLength', 'xsd:pattern', etc.
    The result can be embedded directly in a CREATE TABLE statement.

    :param column_name (str): Name of the SQL column associated with the datatype.
    :param datatype_definition (list[dict]): List of facet definitions, e.g.:
        [{"restriction": "minInclusive", "value": 0}, {"restriction": "maxInclusive", "value": 100}]
    :return (str): SQL snippet representing the CHECK constraint, e.g.:
        CHECK (column_name BETWEEN 0 AND 100)
    """
    checks = []
    
    for defi in datatype_definition:
        if isinstance(defi, dict):
            
            restriction = defi["restriction"]
            restriction_value = defi.get("value")
        
        # if restriction_value:
            # Vergleichsoperatoren für Zahlen
            if restriction == XSD.minInclusive:
                checks.append(f"{column_name} >= {restriction_value}")
            elif restriction == XSD.maxInclusive:
                checks.append(f"{column_name} <= {restriction_value}")
            elif restriction == XSD.minExclusive:
                checks.append(f"{column_name} > {restriction_value}")
            elif restriction == XSD.maxExclusive:
                checks.append(f"{column_name} < {restriction_value}")

            # String-Längen (nur sinnvoll für VARCHAR, TEXT etc.)
            elif restriction == XSD.length:
                checks.append(f"char_length({column_name}) = {restriction_value}")
            elif restriction == XSD.minLength:
                checks.append(f"char_length({column_name}) >= {restriction_value}")
            elif restriction == XSD.maxLength:
                checks.append(f"char_length({column_name}) <= {restriction_value}")

            # Regex-Muster
            elif restriction == XSD.pattern:
                pattern = str(restriction_value).replace("'", "''")  # SQL-Escaping
                checks.append(f"{column_name} ~ '^{pattern}$'")

            # Optional: Hinweise auf nicht direkt umsetzbare Facetten
            elif restriction == XSD.totalDigits or restriction == XSD.fractionDigits:
                print(f"⚠️ Hinweis: totalDigits/fractionDigits für '{column_name}' erkannt, aber nicht automatisch umsetzbar.")

        # else:
        #     checks.append(f"({column_name} IN ({', '.join(str(val) for val in restrictions_definition)}))")
    if checks:
        return f"CONSTRAINT chk_datatype_{column_name} CHECK ({' AND '.join(checks)})"
    else:
        values = []
        for val in datatype_definition:
            if isinstance(val, str):
                val = f"'{val}'"
                values.append(val)
            else:
                values.append(val)
        return f"CONSTRAINT chk_datatype_{column_name} CHECK ({column_name} IN ({', '.join(values)}))"



In [1066]:
def sql_generate_check_constraint(column, check_attributes, check_definition, table_name, context_dict):
    """
    Generates a SQL CHECK constraint for a given column based on ontology-defined rules.

    This function combines datatype facets, ontology-derived restrictions,
    and explicit value conditions into a single SQL CHECK expression.

    :param column (str): Column name to which the constraint applies.
    :param check_attributes (list[str]): Attribute names involved in the condition.
    :param check_definition (list[dict]): Detailed definition of constraint logic.
    :param table_name (str): SQL table name containing the column.
    :param context_dict (dict): Context dictionary for datatype and property mappings.
    :return (str): SQL CHECK constraint expression ready for inclusion in CREATE TABLE.
    """
    # column_constraints = []
    table_constraints = []
    
    if len(check_attributes) >= 2:
        id_attribute_name = check_attributes[1]
        column_name = check_attributes[0]
    
    # owl:NegativePropertyAssertion
    npa = check_definition.get("negativePropertyAssertion", [])
    if npa:
        npa_check = []
        for neg_assertion in npa:
            
            npa_check.append(f"NOT ({id_attribute_name} = '{get_sql_name(neg_assertion[0], 'individual', context_dict)}' AND {column_name} = '{get_sql_name(neg_assertion[1], 'individual', context_dict)}')")
        npa_check = "\n     AND ".join(npa_check)
        table_constraints.append(f"CONSTRAINT chk_forbidden_pairs_{column} CHECK ({npa_check})")

    # # revlexiveProperty
    # if check_definition.get("reflexiveProperty"):
    #     table_constraints.append(
    #         f"CONSTRAINT chk_reflexive_{column} CHECK ({id_attribute_name} <> {column_name})"
    #     )
    
    # irreflexiveProperty
    if check_definition.get("irreflexiveProperty"):
        table_constraints.append(
            f"CONSTRAINT chk_irreflexive_{column} CHECK ({id_attribute_name} <> {column_name})"
        )

    # asymmetricPropert
    if check_definition.get("asymmetricProperty"):
        table_constraints.append(
            f"CONSTRAINT chk_asymmetric_{column} CHECK ({id_attribute_name} <> {column_name})"
        )
    
    # # PropertyDisjointWith
    # disjoint_with = check_definition.get("disjointWith", [])
    # if disjoint_with:
    #     disjoint_check = []
    #     for disjoint_pair in disjoint_with:
    #         if len(disjoint_pair) > 2:
    #             # TODO CHECK for AllDisjointProperties with more then two properties
    #             continue
    #         disjoint_pair_s = (
    #             get_property_info(g, disjoint_pair[0], context_dict).get("equivalentPropertyRepresentative")
    #             or get_property_info(g, disjoint_pair[0], context_dict).get("sqlName")
    #         )
    #         disjoint_pair_o = (
    #             get_property_info(g, disjoint_pair[1], context_dict).get("equivalentPropertyRepresentative")
    #             or get_property_info(g, disjoint_pair[1], context_dict).get("sqlName")
    #         )
    #         disjoint_check.append(f"({disjoint_pair_s} IS DISTINCT FROM {disjoint_pair_o} OR ({disjoint_pair_s} IS NULL AND {disjoint_pair_o} IS NULL))")
    #     disjoint_check = "\n     AND ".join(disjoint_check)
    #     table_constraints.append(f"CONSTRAINT chk_disjointness_{disjoint_pair_s} CHECK ({disjoint_check})")
    
    # oneOf
    one_of_map = check_definition.get("oneOf")
    if one_of_map:
        for source, instance_groups in one_of_map.items():
            one_of_check = []
            instances = set()
            for group in instance_groups:
                instances.update({get_sql_name(gr, 'individual', context_dict) for gr in group})
            one_of_check = f"{table_name}_id IN ('{"', '".join(instances)}')"
            table_constraints.append(f"CONSTRAINT chk_one_of__{clean_value(source)}__{table_name} CHECK ({one_of_check})")
            
    return ",\n     ".join(table_constraints)

In [1067]:
def sql_create_foreign_key(foreign_key_definition, schema = "public"):
    """
    Translates a list of foreign key definitions into SQL ALTER TABLE statements.

    Each definition describes a reference from one table/column to a target table/column.
    The function produces fully executable SQL syntax for constraint creation.

    :param foreign_key_definition (list[dict]): List of foreign key mappings, e.g.:
    :param schema (str, optional): SQL schema name. Defaults to "public".
    :return (str): SQL code containing one or multiple ALTER TABLE statements.
    """

    if isinstance(schema, list):
        schema_table = schema[0]
        schema_target = schema[1]
    else:
        schema_table = schema
        schema_target = schema
        
    foreign_key_sql = []

    for defi in foreign_key_definition:

        fk = []
        foreign_key_table = defi["foreign_key_table"]
        foreign_key_column = defi["foreign_key_column"]
        target_table = defi["target_table"]
        target_column = defi["target_column"]

        if foreign_key_table != target_table:
            foreign_key_sql.append(
                f"ALTER TABLE {schema_table}.{foreign_key_table}\n"
                f"ADD CONSTRAINT fk_{foreign_key_column}\n" 
                f"FOREIGN KEY ({foreign_key_column})\n"
                f"REFERENCES {schema_target}.{target_table}({target_column});"
                )
        else:
            foreign_key_sql.append(
                f"ALTER TABLE {schema_table}.{foreign_key_table}\n"
                f"ADD CONSTRAINT fk_{foreign_key_column}\n"
                f"FOREIGN KEY ({foreign_key_column})\n"
                f"REFERENCES {schema_target}.{target_table}({target_column})\n"
                f"DEFERRABLE INITIALLY DEFERRED;"
                )
    
    return "\n".join(foreign_key_sql)

#### create table/View/trigger, insert into

CREATE TABLE

In [1068]:
def sql_create_table(table_definition, context_dict, schema = "public"):
    """
    Converts a structured table definition into an executable SQL CREATE TABLE statement.

    Includes:
      - Column datatypes, constraints, and defaults.
      - Primary key and unique constraints.
      - Column and table-level comments.
      - Integration of ontology-derived metadata from context_dict.

    :param table_definition (dict): Structured table specification (e.g., from table_definition_class()).
    :param context_dict (dict): Global context containing datatype mappings and SQL conventions.
    :param schema (str, optional): Target SQL schema. Defaults to "public".
    :return (str): SQL CREATE TABLE statement including COMMENT ON clauses.
    """
    sql_table = []
    
    for defi in table_definition:
        if defi:
            table = defi["tableName"]
            table_comments = []
            table_comments_sql = []
            column_defs = []
            column_comments_sql = []
            # column_comments = {}
            primary_keys = []
            foreign_key_line = []
            unique_constraints = []
            table_check_constraints = []

            table_sql = []
            
            attributes = [column["attributeName"] for column in defi["columns"] if column["attributeName"] not in ["source_sys", "sub_refcount_sys", "label"]]

            for column in defi["columns"]:
                column_comments = {}     

                ###### Attribut Name, Datatype und UNIQUE-Constraint ######
                attribute_name = column["attributeName"]
                column_line = f"{attribute_name} {column['datatype']}"

                if column.get("unique"):
                    column_line += " UNIQUE"
                
                if column.get("default") is not None:
                    default_val = column["default"]
                    if isinstance(default_val, str):
                        default_val = f"'{default_val}'"
                    column_line += f" DEFAULT {default_val}"
                
                if column.get("reflexiv"):
                    column_line += f" GENERATED ALWAYS AS ({column["reflexiv"]}) STORED"

                if column.get("notNull"):
                    column_line += " NOT NULL"
                
                if attribute_name == "source_sys":
                    column_line += " CHECK (source_sys IN ('explicit', 'derived'))"
                
                ###### CHECK Constarints 1 ######
                check_definition = column.get("check", set()) or column.get("check_class", set())
                if check_definition:
                    table_check = sql_generate_check_constraint(attribute_name, attributes, check_definition, table, context_dict)
                    if table_check:
                        table_check_constraints.append(table_check)
                    
                
                ###### Datatype CHECK Constraints ######     
                datatype_check_definition = column.get("datatype_check")
                if datatype_check_definition:
                    datatype_check = sql_generate_datatype_check(attribute_name, datatype_check_definition)
                    # column_line += f" {datatype_check}"
                    table_check_constraints.append(datatype_check)
                column_defs.append(column_line)
    
                ###### Attribut Primary Key 1 ######
                if column.get("primaryKey"):
                    primary_keys.append(attribute_name)
                
                ###### Attribut Comments ######
                if column.get("comment"):
                    column_comments = column.get("comment")
                    comment_text = "\n".join(column_comments)
                    column_comments_sql.append(
                        f"COMMENT ON COLUMN {schema}.{table}.{attribute_name} IS '{comment_text}';"
                    )
            
            ###### Table Primary Key 2 ######
            if primary_keys:
                pk_line = f'PRIMARY KEY ({", ".join(primary_keys)})'
                column_defs.append(pk_line)
            
            if foreign_key_line:
                column_defs.append(", ".join(foreign_key_line))

            ###### Table UNIQE-Constraint ######
            if defi.get("unique_column_combinations"):
                for unique_cols in defi["unique_column_combinations"]:
                    column_defs.append(f"UNIQUE ({', '.join(uc for uc in unique_cols)})")

            ###### CHECK Constarints 2 ###### 
            if table_check_constraints:
                column_defs.append(",\n    ".join(table_check_constraints))


            ###### Table comments ######
            if defi["tableAnnotations"]:
                table_comments = defi["tableAnnotations"]
                comment_text = "\n".join(table_comments)
                column_comments_sql.append(
                    f"COMMENT ON TABLE {schema}.{table} IS '{comment_text}';"
                )

            ###### Create Table ###### 
            table_sql.append(
                f"DROP TABLE IF EXISTS {schema}.{table} CASCADE; \n"
                f"CREATE TABLE {schema}.{table} (\n    " + ",\n    ".join(column_defs) + "\n);"
            )

            table_sql.extend(column_comments_sql)
        
        sql_table.append("\n".join(table_sql))

    return "\n".join(sql_table)

INSERT INTO \
Generiert aus extract_instances() SQL-Code

In [1069]:
def sql_insert_into(instance_definitions, schema = "public"):
    """
    Generates SQL INSERT INTO statements from instance definitions.

    Each entry in 'instance_definitions' represents one relational row derived
    from ontology individuals or property assertions. The resulting SQL inserts
    preserve ontology semantics within the database.

    :param instance_definitions (list[dict]): Structured instance definitions.
    :param schema (str, optional): Target SQL schema. Defaults to "public".
    :return (str): Executable SQL INSERT statements for all instances.
    """
    sql_lines = []
    annotations = []
    

    for defi in instance_definitions:
        table = defi["table"]
        
        attributes = ", ".join(defi["values"].keys())
        values = ", ".join(f"'{str(v)}'" for v in defi["values"].values())
        sql_lines.append(
            f"INSERT INTO {schema}.{table} ({attributes}) VALUES ({values});"
        )

    return "\n".join(sorted(sql_lines))

CREATE VIEW

In [1070]:
def sql_create_view(view_definition, schema = "public"):
    """
    Translates a structured view definition into a SQL CREATE VIEW statement.

    Views are used to represent derived relations (e.g., reflexive or disjointness views)
    and are generated automatically from ontology semantics.

    :param view_definition (dict): Definition of the view including SELECT, FROM, GROUP BY, etc.
    :param schema (str, optional): SQL schema where the view is created. Defaults to "public".
    :return (str): SQL CREATE VIEW statement representing the ontology-derived logic.
    """
    view_sql = []
    
    for defi in view_definition:
        view_name = defi["viewName"]
        table_select = defi["select"]
        from_table = defi.get("from")
        from_table_ = defi.get("from_")
        union = defi.get("union")
        join = defi.get("joinOn")
        where = defi.get("where")
        group_by = defi.get("group_by")
        having = defi.get("having")
        order_by = defi.get("order_by")
        view_comments = defi.get("viewComments")
        
        if union:
            selects = []       
            for i in range(len(table_select)):
                select = []
                select.append(f"SELECT {table_select[i]}")
                if join:
                    select.append(f"FROM {schema}.{from_table[i][0]} JOIN {schema}.{from_table[i][1]}")
                    select.append(f"ON {join[i]}")
                    if i < len(where):
                        select.append(f"WHERE {where[i]}")
                else:
                    select.append(f"FROM {schema}.{from_table[i]}")
                if i < len(where):
                    where_i = where[i].replace("FROM", f"FROM {schema}.")
                    select.append(f"WHERE {where_i}")
                # select = "\n".join(select)
                selects.append("\n".join(select))
            selects = f"\n{union}\n".join(selects)
            
            view_sql.append(
                f"DROP VIEW IF EXISTS {schema}.{view_name}; \n"
                f"CREATE VIEW {schema}.{view_name} AS\n"
                f"{selects};"
            )
            if view_comments:
                view_sql.append(
                    f"COMMENT ON VIEW {schema}.{view_name} IS '{view_comments}';"
                ) 
        else: 
            view_sql.append(
                f"DROP VIEW IF EXISTS {schema}.{view_name}; \n"
                f"CREATE VIEW {schema}.{view_name} AS \n"
                f"SELECT {table_select} \n"
            )
            if from_table:
                view_sql.append(f"FROM {from_table} \n")
            else:
                view_sql.append(f"FROM {schema}.{from_table_} \n")
            if group_by:
                view_sql.append(f"GROUP BY {group_by}\n")
            if having:
                view_sql.append(f"HAVING {having}\n")
            if order_by:
                view_sql.append(f"ORDER BY {order_by}\n")
            view_sql.append(";\n")
             
            if view_comments:
                view_sql.append(f"COMMENT ON VIEW {schema}.{view_name} IS '{view_comments}';\n")
    
    return ''.join(view_sql)

CREATE TRIGGER

In [1071]:
def sql_create_trigger(trigger_definition, schema = "public"):
    """
    Converts structured trigger metadata into executable SQL CREATE TRIGGER statements.

    Each trigger definition specifies:
      - Timing (BEFORE / AFTER)
      - Event (INSERT / UPDATE / DELETE)
      - Target table
      - Invoked trigger function

    Triggers enforce semantic constraints such as subclass propagation,
    property inheritance, or disjointness handling.

    :param trigger_definition (list[dict]): List of trigger definitions.
    :param schema (str, optional): SQL schema for trigger creation. Defaults to "public".
    :return (str): SQL CREATE TRIGGER statements.
    """

    trigger_sql = []
    for trigger in trigger_definition:
        trigger_ = []
        if trigger.get("triggerName") and trigger.get("target") and trigger.get("function"):

            trigger_name = trigger.get("triggerName")
            trigger_point_time = trigger.get("pointInTime")
            trigger_event = trigger.get("event") if isinstance(trigger.get("event"), str) else " OR ".join(trigger.get("event"))
            trigger_target = trigger.get("target")
            trigger_validity = trigger.get("validity")
            trigger_condition = trigger.get("condition")
            trigger_function = trigger.get("function")

            trigger_.append(f"DROP TRIGGER IF EXISTS {trigger_name} ON {schema}.{trigger_target};")
            trigger_.append(f"CREATE TRIGGER {trigger_name}")
            
            if trigger_point_time and trigger_event:
                trigger_.append(f"{trigger_point_time} {trigger_event} ON {schema}.{trigger_target}")

            if trigger_validity:
                trigger_.append(f"FOR EACH {trigger_validity}")
            
            if trigger_condition:
                inner_lists = [f"({' AND '.join(inner)})" for inner in trigger_condition]
                trigger_.append(f"WHEN {' OR '.join(inner_lists)}")
            
            trigger_.append(f"EXECUTE FUNCTION {schema}.{trigger_function};")
        
        trigger_sql.append("\n".join(trigger_))

    if not trigger_sql:
        trigger_sql = ""
    
    return "\n".join(trigger_sql) 


## Ω - Übertragung an PostgreSQL

In [1072]:
schema = make_db_schema_from_uri()
sys_schema = f"{schema}_sys"

db_config = {
    "host": "localhost",
    "database": "Master-Thesis",
    "user": "ragna",
    "password": "123456789",
    "port": "5433", 
}

conn = psycopg2.connect(**db_config)
cursor = conn.cursor()

keywords_for_preferred_lable = ["preferred", "pref", "pf", "prefer", "preferable", "fav", "favour", "favourite"]

context_dict = build_context_dict(g, keywords_for_preferred_lable, schema)
mark("after_context_dict")
all_classes = context_dict["all_classes"]
all_properties = context_dict["all_properties"]

cls_mapping = {}
prop_mapping = {}

sql_prop_table = []
sql_prop_view = []
sql_prop_trigger = []

defi_cls_table = []
defi_prop_table = []
defi_inv_table = []
defi_trigger = []
defi_fk_table = []
defi_view = []

defi_prop_fk_table = []

create_schema = f"CREATE SCHEMA IF NOT EXISTS {schema};"
create_sys_schema = f"CREATE SCHEMA IF NOT EXISTS {sys_schema};"

all_schema = "\n".join([
    create_schema,
    create_sys_schema
])

if all_schema:
    cursor.execute(all_schema)
    conn.commit()
print(f"Schema {schema} Fertig")
mark("after_schema")

drop_source_sub_refcount = (
    f"DROP FUNCTION IF EXISTS {schema}.add_source_sub_refcount() CASCADE ;"
    "DROP EVENT TRIGGER IF EXISTS trg_add_source_sub_refcount;"
    )
cursor.execute(drop_source_sub_refcount)
conn.commit()

# CREATE TABLE
for cls in all_classes:
    table_defi, cls_mapping, fk_defi, trigger_defi, view_defi = table_definition_class(g, cls, cls_mapping, context_dict)
    if table_defi:
        defi_cls_table.append(table_defi)
        defi_trigger.extend(trigger_defi)
        defi_fk_table.extend(fk_defi)
        defi_view.extend(view_defi)
        
sql_cls_table = sql_create_table(defi_cls_table, context_dict, schema)
print("     Klassen Tabellen Fertig")
mark("after_class_tables")

for prop in all_properties:
    prop_table_defi, prop_key_defi, prop_mapping, prop_view_defi = table_definition_properties(g, prop, cls_mapping, prop_mapping, context_dict)
    if prop_table_defi:
        defi_prop_table.append(prop_table_defi)
        defi_fk_table.extend(prop_key_defi)
        defi_view.extend(prop_view_defi)
        continue
    
    inv_table_defi, inv_key_defi, prop_mapping, inv_view_defi = table_definition_inverse_properties(g, prop, cls_mapping, prop_mapping, context_dict)
    if inv_table_defi:
        defi_inv_table.append(inv_table_defi)
        defi_fk_table.extend(inv_key_defi)
        defi_view.extend(inv_view_defi)
        continue
    
sql_prop_table = sql_create_table(defi_prop_table, context_dict, schema)
sql_inv_table = sql_create_table(defi_inv_table, context_dict, schema)
print("     Property Tabellen Fertig")
mark("after_property_tables") 


defi_annotation_table = table_definition_annotation_table()
sql_table_annotation = sql_create_table([defi_annotation_table], context_dict, schema)

defi_has_key_table, defi_fk_has_key = table_definition_has_key_table()
sql_table_has_key = sql_create_table([defi_has_key_table], context_dict, schema)

defi_table_deprecated_elements = table_definition_deprecated_elements()
sql_table_deprecated_elements = sql_create_table([defi_table_deprecated_elements], context_dict, sys_schema)


defi_table_all_classes = table_definition_all_classes_table()
sql_table_all_classes = sql_create_table([defi_table_all_classes], context_dict, sys_schema)

defi_table_sub_class, defi_fk_sub_class, defi_install_sub_class_triggers = table_definition_sub_class_of()
sql_table_sub_class = sql_create_table([defi_table_sub_class], context_dict, sys_schema)

defi_table_equi_class, defi_fk_equi_class = table_definition_equivalent_class_table()
sql_table_equi_class = sql_create_table([defi_table_equi_class], context_dict, sys_schema)


defi_table_all_properties, defi_fk_all_properties = table_definition_all_properties_table()
sql_table_all_properties = sql_create_table([defi_table_all_properties], context_dict, sys_schema)

defi_table_sub_properties, defi_fk_sub_prop, defi_install_sub_prop_triggers = table_definition_sub_property_of()
sql_table_sub_property = sql_create_table([defi_table_sub_properties], context_dict, sys_schema)

defi_table_equi_property, defi_fk_equi_prop = table_definition_equivalent_property_table()
sql_table_equi_property = sql_create_table([defi_table_equi_property], context_dict, sys_schema)
mark("after_meta_tables")

sql_all_tables = "\n".join([
    sql_cls_table, 
    sql_prop_table,
    sql_inv_table,
    sql_table_annotation, 
    sql_table_has_key,
    sql_table_all_classes, 
    sql_table_sub_class,
    sql_table_equi_class,
    sql_table_all_properties,
    sql_table_sub_property,
    sql_table_equi_property,
    sql_table_deprecated_elements
])

if sql_all_tables:
    cursor.execute(sql_all_tables)
    conn.commit()

print("Tabellen Fertig")
mark("after_committing_tables")

# INSERT INTO
defi_instances = instance_definition(g, cls_mapping, prop_mapping, context_dict)
sql_instances = sql_insert_into(defi_instances, schema)
print("     Klassen Instanzen Fertig")
mark("after_class_instance")


defi_prop_instances = instance_definition_properties(g, cls_mapping, prop_mapping, context_dict)
sql_prop_instances = sql_insert_into(defi_prop_instances, schema)

defi_inv_instances = instance_definition_inverse_properties(g, cls_mapping, prop_mapping, context_dict)
sql_inv_instances = sql_insert_into(defi_inv_instances, schema)
print("     Property Instanzen Fertig")
mark("after_property_instance")


defi_annotation_instances = entry_definition_annotation_table(g, cls_mapping, prop_mapping, context_dict)
sql_annotation_instances = sql_insert_into(defi_annotation_instances, schema)

defi_has_key_instances = entry_definition_has_key_table(g, cls_mapping, context_dict)
sql_has_key_instances = sql_insert_into(defi_has_key_instances, schema)

defi_deprecated_elements_instances = entry_definiton_deprecated_elements(g, context_dict)
sql_deprecated_elements_instances = sql_insert_into(defi_deprecated_elements_instances, sys_schema)


defi_all_classes_instances = entry_definition_all_classes(g, cls_mapping, context_dict)
sql_all_classes_instances = sql_insert_into(defi_all_classes_instances, sys_schema)

defi_sub_class_instances = entry_definition_sub_class_of(g, cls_mapping, context_dict)
sql_sub_class_instances = sql_insert_into(defi_sub_class_instances, sys_schema)

defi_equivalent_class_instances = entry_definition_equivalent_class(g, cls_mapping, context_dict)
sql_equivalent_class_instances = sql_insert_into(defi_equivalent_class_instances, sys_schema)


defi_all_propety_instances = entry_definition_all_properties(g, prop_mapping, context_dict)
sql_all_property_instances = sql_insert_into(defi_all_propety_instances, sys_schema)

defi_sub_property_instances = entry_definition_sub_property_of(g, prop_mapping, context_dict)
sql_sub_property_instances = sql_insert_into(defi_sub_property_instances, sys_schema)

defi_equivalent_property_instances = entry_definition_equivalent_property(g, prop_mapping, context_dict)
sql_equivalent_property_instances = sql_insert_into(defi_equivalent_property_instances, sys_schema)
mark("after_meta_instance")

sql_entries = "\n".join([
    "BEGIN;",
    sql_instances,
    sql_prop_instances,
    sql_inv_instances,
    sql_annotation_instances,
    sql_has_key_instances,
    sql_all_classes_instances,
    sql_equivalent_class_instances,
    sql_sub_class_instances,
    sql_all_property_instances,
    sql_sub_property_instances,
    sql_equivalent_property_instances,
    sql_deprecated_elements_instances,
    "COMMIT;"
])

if sql_entries:
    cursor.execute(sql_entries)
    conn.commit()
print("Einträge Fertig")
mark("after_committing_instances")


# CREATE VIEW
sql_view = ""
sql_view_disjoint_properties = ""
sql_view_disjoint_classes = ""

if defi_view:
    sql_view = sql_create_view(defi_view, schema)
    
defi_view_disjoint_properties = view_definition_element_disjoint_with(g, prop_mapping, context_dict, "properties", schema)
if defi_view_disjoint_properties:
    sql_view_disjoint_properties = sql_create_view([defi_view_disjoint_properties], schema)

defi_view_disjoint_classes = view_definition_element_disjoint_with(g, cls_mapping, context_dict, "classes", schema)
if defi_view_disjoint_classes:
    sql_view_disjoint_classes = sql_create_view([defi_view_disjoint_classes], schema)
mark("after_views")

sql_all_views = "".join([
    sql_view,
    sql_view_disjoint_properties,
    sql_view_disjoint_classes
])
# print(sql_all_views)
if sql_all_views:
    cursor.execute(sql_all_views)
    conn.commit()
print("Views Fertig")
mark("after_committing_views")


# CREATE TRIGGER
function_sub_class_trigger = sub_class_trigger_functions(sys_schema, schema)
function_install_new_subclass_trigger = install_new_subclassof_trigger_function(sys_schema, schema)
function_insert_source_sub_ref_count = trigger_definition_insert_source_sub_refcount(schema)
function_sub_property_trigger = sub_property_trigger_functions(sys_schema, schema)
function_symmetric_trigger = symmetric_property_trigger_functions(schema, sys_schema)
function_asymmetric_trigger = asymmetric_property_trigger_functions(schema, sys_schema)


sql_trigger = sql_create_trigger(defi_trigger, schema)
sql_install_new_sub_class_trigger = sql_create_trigger([defi_install_sub_class_triggers], sys_schema)
sql_install_new_sub_property_trigger = sql_create_trigger([defi_install_sub_prop_triggers], sys_schema)

defi_sub_property_trigger = trigger_definition_sub_property(g, prop_mapping, context_dict)
sql_sub_property_trigger = sql_create_trigger(defi_sub_property_trigger, schema)

defi_complex_class_trigger, function_complex_class_trigger = trigger_definition_complex_classes(g, cls_mapping, prop_mapping, context_dict, schema)
sql_complex_class_trigger = sql_create_trigger(defi_complex_class_trigger, schema)

defi_symmetric_property = trigger_definition_symmetric_property(g, prop_mapping, context_dict)
sql_symmetric_property = sql_create_trigger(defi_symmetric_property, schema)

defi_asymmetric_property = trigger_definition_asymmetric_property(g, prop_mapping, context_dict)
sql_asymmetric_property = sql_create_trigger(defi_asymmetric_property, schema)

mark("after_triggers")
# print(function_install_new_subclass_trigger)

all_triggers = "\n\n".join([
    function_sub_class_trigger,
    function_sub_property_trigger,
    function_install_new_subclass_trigger,
    function_insert_source_sub_ref_count,
    function_complex_class_trigger,
    function_symmetric_trigger,
    function_asymmetric_trigger,
    sql_install_new_sub_property_trigger,
    sql_trigger,
    sql_install_new_sub_class_trigger,
    sql_sub_property_trigger,
    sql_complex_class_trigger,
    sql_symmetric_property,
    sql_asymmetric_property,
])

if all_triggers:
    cursor.execute(all_triggers)
    conn.commit()
print("Trigger Fertig")
mark("after_committing_triggers")



# # ALTER FK
# sql_fks = []
# sql_fk_sub_class_table = ""
# sql_defi_fk_equivalent_class_table = ""
# sql_fk_sub_property_table = ""
# sql_defi_fk_equivalent_property_table = ""
# sql_fk_has_key_table = ""

# if defi_fk_table:
#     sql_fk_table = sql_create_foreign_key(defi_fk_table, schema)

# if defi_fk_has_key:
#     sql_fk_has_key_table = sql_create_foreign_key(defi_fk_has_key, [schema, sys_schema])
    
# if defi_fk_sub_class:
#     sql_fk_sub_class_table = sql_create_foreign_key(defi_fk_sub_class, sys_schema)
# if defi_fk_equi_class:
#     sql_fk_equivalent_class_table = sql_create_foreign_key(defi_fk_equi_class, sys_schema)

# if defi_fk_sub_prop:
#     sql_fk_sub_property_table = sql_create_foreign_key(defi_fk_sub_prop, sys_schema)
# if defi_fk_equi_prop:
#     sql_fk_equivalent_property_table = sql_create_foreign_key(defi_fk_equi_prop, sys_schema)
# if defi_fk_all_properties:
#     sql_fk_all_property_table = sql_create_foreign_key(defi_fk_all_properties, sys_schema)
# mark("after_foreign_keys")

# sql_fks = "\n".join([
#     sql_fk_table,
#     sql_fk_has_key_table,
#     sql_fk_equivalent_class_table,
#     sql_fk_sub_class_table,
#     sql_fk_sub_property_table,
#     sql_fk_equivalent_property_table,
#     sql_fk_all_property_table
# ])

# if sql_fks:
#     cursor.execute(sql_fks)
#     conn.commit()
# print("FKs Fertig")
# mark("after_committing_foreign_keys")

conn.close()
print("fertig")

mark("end_total")

Schema ontology Fertig
     Klassen Tabellen Fertig
     Property Tabellen Fertig
Tabellen Fertig
     Klassen Instanzen Fertig
     Property Instanzen Fertig
Einträge Fertig
Views Fertig
Trigger Fertig
fertig


# Evaluation - Part 2

In [1073]:
# ---------- BUILD RUNTIME REPORT ----------
# ---------- RUNTIME REPORT WITH CLEAN LABELS ----------
mark("end_total")
report = [
    f"Ontology: {schema}, {ONTOLOGY_URI}"
    f"Total runtime (full pipeline): {elapsed('start_total', 'end_total'):.2f} s",

    # Phase 1 — Ontology load
    f"Loading ontology + preprocessing: {elapsed('start_total', 'ontology_loaded'):.2f} s",

    # Phase 2 — Initalizing context dict
    f"Context Dictionary creation: {elapsed('ontology_loaded', 'after_context_dict'):.2f} s",
    
    # Phase 2 — Schema creation
    f"Schema creation: {elapsed('after_context_dict', 'after_schema'):.2f} s",

    # Phase 3 — Table creation
    f"Ontology table creation total time: {elapsed('after_schema', 'after_property_tables'):.2f} s",
    f"  Class table creation: {elapsed('after_schema', 'after_class_tables'):.2f} s",
    f"  Property table creation: {elapsed('after_class_tables', 'after_property_tables'):.2f} s",
    f"Metadata table creation (sys-tables): {elapsed('after_property_tables', 'after_meta_tables'):.2f} s",
    f"Commit table creation: {elapsed('after_meta_tables', 'after_committing_tables'):.2f} s",

    # Phase 4 — Instance inserts
    f"Insert all ontology instances: {elapsed('after_committing_tables', 'after_property_instance'):.2f} s",
    f"  Insert class instances: {elapsed('after_committing_tables', 'after_class_instance'):.2f} s",
    f"  Insert property instances: {elapsed('after_class_instance', 'after_property_instance'):.2f} s",
    f"Insert metadata instances: {elapsed('after_property_instance', 'after_meta_instance'):.2f} s",
    f"Commit instance inserts: {elapsed('after_meta_instance', 'after_committing_instances'):.2f} s",

    # Phase 5 — Views
    f"View creation: {elapsed('after_committing_instances', 'after_views'):.2f} s",
    f"Commit view creation: {elapsed('after_views', 'after_committing_views'):.2f} s",

    # Phase 6 — Triggers
    f"Trigger function + trigger creation: {elapsed('after_committing_views', 'after_triggers'):.2f} s",
    f"Commit trigger creation: {elapsed('after_triggers', 'after_committing_triggers'):.2f} s",

    # Phase 7 — Foreign Keys
    # f"Foreign key creation: {elapsed('after_committing_triggers', 'after_foreign_keys'):.2f} s",
    # f"Commit foreign keys: {elapsed('after_foreign_keys', 'after_committing_foreign_keys'):.2f} s",

    # Phase 8 — Finalization
    f"Finalization: {elapsed('after_committing_triggers', 'end_total'):.2f} s",
]


# Append to log file
for line in report:
    append_to_file(LOG_FILE, line)

append_to_file(LOG_FILE, "===== END RUN =====\n")

In [1074]:
def get_size_of_onto_db(conn, schema):
    # Size of Schema in PostgreSQL
    query_size = """
        SELECT
            SUM(pg_total_relation_size(
                quote_ident(table_schema) || '.' || quote_ident(table_name)
                )
            ) AS size_bytes
        FROM information_schema.tables
        WHERE table_schema = %s
           OR table_schema = %s;
    """
    with conn.cursor() as cur:
        cur.execute(query_size, (schema, f"{schema}_sys"))
        size_bytes = cur.fetchone()[0] or 0

    db_size = size_bytes / (1024 ** 2)

    # Size of Ontology as RDF Graph
    with tempfile.NamedTemporaryFile(delete=False, suffix=".nt") as tmp:
        g.serialize(destination=tmp.name, format="nt")
        size_bytes = os.path.getsize(tmp.name)
    os.remove(tmp.name)
    onto_size = size_bytes / (1024 * 1024)
    
    return db_size, onto_size

In [1075]:
def get_number_of_classes(conn, schema, context_dict):
    ##### ONTOLOGY #####
    all_classes = context_dict["all_classes"]
    num_classes_onto = len(all_classes)
    num_classes_impl_onto = len([cls for cls in all_classes if get_class_info(g, cls, context_dict)["implementedAs"] == "table"])
    
    ##### DB #####
    query_classes = f"""
        SELECT count(classes) AS class_count
        FROM (
            SELECT class_id AS classes
            FROM {schema}_sys.all_classes_sys
                UNION ALL
            SELECT element_id AS classes
            FROM {schema}_sys.deprecated_classes_and_properties_sys
            WHERE element_type = 'class'
        );
    """
    with conn.cursor() as cur:
        cur.execute(query_classes)
        num_classes_db = cur.fetchone()[0] or 0
        
    query_class_tables = """
        SELECT COUNT(*) AS table_count
        FROM information_schema.tables
        WHERE table_schema = %s
            AND table_name NOT LIKE '%%_rel%%'
            AND table_name NOT LIKE '%%_meta%%'
            AND table_type = 'BASE TABLE';;
    """
    with conn.cursor() as cur:
        cur.execute(query_class_tables, (schema,))
        num_classes_impl_db = cur.fetchone()[0] or 0
    return num_classes_onto, num_classes_impl_onto, num_classes_db, num_classes_impl_db

In [1076]:
def get_number_of_properties(conn, schema, context_dict):
    ##### ONTO #####
    all_properties = context_dict["all_properties"]
    num_properties_onto = len(all_properties)
    num_properties_impl_onto = len([prop for prop in all_properties if get_property_info(g, prop, context_dict)["implementedAs"] in ("multivalue relation", "functional relation", "inverse relation (representative)")])
    
    ##### DB #####
    query_properties = f"""
        SELECT count(properties) AS property_count
        FROM (
            SELECT property_id AS properties
            FROM {schema}_sys.all_properties_sys
                UNION ALL
            SELECT element_id AS properties
            FROM {schema}_sys.deprecated_classes_and_properties_sys
            WHERE element_type = 'property'
        );
    """
    with conn.cursor() as cur:
        cur.execute(query_properties)
        num_properties_db = cur.fetchone()[0] or 0
    
    query_prop_tables = """
        SELECT COUNT(*) AS table_count
        FROM information_schema.tables
        WHERE table_schema = %s
            AND table_name LIKE '%%_rel%%'
            AND table_name NOT LIKE '%%_meta%%'
            AND table_type = 'BASE TABLE';
    """
    with conn.cursor() as cur:
        cur.execute(query_prop_tables, (schema,))
        num_properties_impl_db = cur.fetchone()[0] or 0

    return num_properties_onto, num_properties_impl_onto, num_properties_db, num_properties_impl_db
 

In [1077]:
# for a in sorted(all_properties):
#     print(get_sql_name(a,"property", context_dict))

# for c in sorted(all_classes):
#     print(get_sql_name(c, "class", context_dict), c)

for a in prop_mapping:
    if prop_mapping[a]["name"] == "album_deezer_album_id_rel":
        print(a,prop_mapping[a]["name"])

In [1078]:
def get_number_of_records_triple(conn, schema, context_dict):
    query_tables = """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = %s
          AND table_type = 'BASE TABLE'
          AND table_name NOT LIKE '%%_meta'
          ;
    """
    all_classes = context_dict["all_classes"]
    all_individuals = context_dict["all_individuals"]
    all_annotation_prop = context_dict["all_annotation_properties"]

    total_rows = 0

    with conn.cursor() as cur:
        cur.execute(query_tables, (schema,))
        tables = cur.fetchall()
        for (table_name,) in tables:
            cur.execute(
                f"SELECT COUNT(*) FROM {schema}.{table_name}"
            )
            total_rows += cur.fetchone()[0]

    total_individual_triple = 0
    # for cls in all_classes:
    #     if get_class_info(g, cls, context_dict)["implementedAs"] == "not at all: deprecated":
    #         continue
    #     for s in g.subjects(RDF.type, cls):
    #         if s not in all_individuals:
    #             continue
    #         total_individual_triple += 1
    # for prop in all_properties:
    #     if get_property_info(g, prop, context_dict)["implementedAs"] == "not at all: deprecated":
    #         continue
    #     for s in g.subjects(prop, None):
    #         if s not in all_individuals:
    #             continue
    #         total_individual_triple +=1
    # for anno_prop in all_annotation_prop:
    #     for s in g.subjects(anno_prop, None):
    #         total_individual_triple +=1
    # for p in [RDFS.label, RDFS.comment, RDFS.seeAlso, RDFS.isDefinedBy, OWL.sameAs, OWL.AllDifferent, OWL.differentFrom]:
    #     for s in g.subjects(p, None):
    #         total_individual_triple +=1
    
    for s, p, o in g.triples((None, None, None)):
        # if p in all_annotation_prop:
        #     total_individual_triple += 1
        #     continue
        # if p in [RDFS.label, RDFS.comment, RDFS.seeAlso, RDFS.isDefinedBy, OWL.sameAs, OWL.AllDifferent, OWL.differentFrom]:
        #     total_individual_triple += 1
        #     continue
        if s not in all_individuals:
            continue
        if p != RDF.type and p not in all_properties:# and p not in all_annotation_prop:
            continue
        if p == RDF.type and o not in all_classes:
            continue
        # if p not in all_annotation_prop:
        if p != RDF.type and p in all_properties and not get_property_info(g, p, context_dict).get("implementedAs", "") in ("multivalue relation", "functional relation", "inverse relation (representative)"):
            continue
        if p == RDF.type and o in all_classes and not get_class_info(g, o, context_dict).get("implementedAs", "") == 'table':
            continue
        total_individual_triple += 1
    
    return total_rows, total_individual_triple

In [1079]:
def result_quantitaniv_analysis(conn, schema, context_dict, csv_path = 'schema_sizes.csv'): 
    
    db_size, onto_size = get_size_of_onto_db(conn, schema)
    num_classes_onto, num_classes_impl_onto, num_classes_db, num_classes_impl_db = get_number_of_classes(conn, schema, context_dict)
    num_properties_onto, num_properties_impl_onto, num_properties_db, num_properties_impl_db = get_number_of_properties(conn, schema, context_dict)
    num_triple_db, num_triple_onto = get_number_of_records_triple(conn, schema, context_dict)
    
    # Append results to CSV
    file_exists = os.path.isfile(csv_path)
    with open(csv_path, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        # Header nur einmal schreiben
        if not file_exists:
            writer.writerow([
                "ontology",
                "size_in_mb_onto",
                "size_in_mb_db",
                "num_classes_onto",
                "num_classes_db",
                "num_classes_implemented_onto",
                "num_classes_implemented_db",
                "num_properties_onto",
                "num_properties_db",
                "num_properties_implemented_onto",
                "num_properties_implemented_db",
                "num_triple_onto",
                "num_triple_db"                
            ])

        writer.writerow([
            schema,
            f"{onto_size:.2f}",
            f"{db_size:.2f}",
            num_classes_onto,
            num_classes_db,
            num_classes_impl_onto,
            num_classes_impl_db,
            num_properties_onto,
            num_properties_db,
            num_properties_impl_onto,
            num_properties_impl_db,
            num_triple_onto,
            num_triple_db,
        ])


db_config = {
    "host": "localhost",
    "database": "Master-Thesis",
    "user": "ragna",
    "password": "123456789",
    "port": "5433", 
}

conn = psycopg2.connect(**db_config)
cursor = conn.cursor()

result_quantitaniv_analysis(conn, schema, context_dict)

In [1080]:
# # Alle Konstrukte
# # 🔧 Pfade
# ONT_DIR = r"C:\Users\ilove\OneDrive\Uni\Master - Philipps Uni\Master Thesis\Ontos für Evaluation"
# OUT_CSV = "axiom_count3.csv"
# EXTS = (".ttl")
# # ONT_DIR = r"C:\Users\ilove\OneDrive\Uni\Master - Philipps Uni\Master Thesis\Ontologien\Zum Testen von Verteilungen von Konzepten\Bio Portal"
# # OUT_CSV = "Bio_Portal_ontology_construct_counts_full3.csv"
# # EXTS = (".owl", ".ttl", ".rdf")

# # --- 1) Definition aller zu zählenden Konstrukte ---

# # a) Dinge, die über rdf:type gezählt werden (Klassen/Konstrukte + Property-Eigenschaften!)
# #    Manche OWL2-Varianten haben alternative Klassennamen (z. B. SymmetricObjectProperty).
# TYPE_MULTI = {
#     "owl:Class": [OWL.Class, RDFS.Class],
#     "owl:AllDisjointClasses": [OWL.AllDisjointClasses],
#     "owl:DeprecatedClass": [OWL.DeprecatedClass],

#     "owl:NegativePropertyAssertion": [OWL.NegativePropertyAssertion],

#     "owl:ObjektProperty": [OWL.ObjectProperty],
#     "owl:DatatypeProperty": [OWL.DatatypeProperty],
#     "rdf:Property": [RDF.Property],
#     "owl:AnnotationProperty": [OWL.AnnotationProperty],
#     "owl:DeprecatedProperty": [OWL.DeprecatedProperty],
#     "owl:AllDisjointProperties": [OWL.AllDisjointProperties],
#     "owl:Restriction": [OWL.Restriction],

#     "owl:FunctionalProperty": [OWL.FunctionalProperty],
#     "owl:InverseFunctionalProperty": [OWL.InverseFunctionalProperty],
#     "owl:SymmetricProperty": [OWL.SymmetricProperty],
#     "owl:AsymmetricProperty": [OWL.AsymmetricProperty],
#     "owl:TransitiveProperty": [OWL.TransitiveProperty],
#     "owl:ReflexiveProperty": [OWL.ReflexiveProperty],
#     "owl:IrreflexiveProperty": [OWL.IrreflexiveProperty],
# }

# # b) Dinge, die als Prädikat auftreten (Triple-Mittelteil)
# PREDICATES = {
#     # Klassen-Axiome (TBox)
#     "rdfs:subClassOf": RDFS.subClassOf,
#     "owl:equivalentClass": OWL.equivalentClass,
#     "owl:disjointWith": OWL.disjointWith,
#     "owl:hasKey": OWL.hasKey,

#     # # Klassen-Konstruktoren
#     "owl:intersectionOf": OWL.intersectionOf,
#     "owl:unionOf": OWL.unionOf,
#     "owl:complementOf": OWL.complementOf,
#     "owl:oneOf": OWL.oneOf, 

#     # # Property Restrictions
#     "owl:someValuesFrom": OWL.someValuesFrom,
#     "owl:allValuesFrom": OWL.allValuesFrom,
#     "owl:hasValue": OWL.hasValue,
#     "owl:minCardinality": OWL.minCardinality,
#     "owl:maxCardinality": OWL.maxCardinality,
#     "owl:cardinality": OWL.cardinality,
#     "owl:minQualifiedCardinality": OWL.minQualifiedCardinality,
#     "owl:maxQualifiedCardinality": OWL.maxQualifiedCardinality,
#     "owl:qualifiedCardinality": OWL.qualifiedCardinality,
#     "owl:hasSelf": OWL.hasSelf,
    
#     "owl:deprecated": OWL.deprecated,

#     # # Property-Axiome (Prädikate)
#     "rdfs:subPropertyOf": RDFS.subPropertyOf,
#     "rdfs:domain": RDFS.domain,
#     "rdfs:range": RDFS.range,
#     "owl:propertyDisjointWith": OWL.propertyDisjointWith,
#     "owl:inverseOf": OWL.inverseOf,
#     "owl:equivalentProperty": OWL.equivalentProperty,
#     "owl:propertyChainAxiom": OWL.propertyChainAxiom,

#     # RDFS (Prädikate)
#     "rdfs:label": RDFS.label,
#     "rdfs:comment": RDFS.comment,
#     "rdfs:seeAlso": RDFS.seeAlso,
#     "rdfs:isDefinedBy": RDFS.isDefinedBy,
#     "owl:differentFrom": OWL.differentFrom,
#     "owl:sameAs": OWL.sameAs,
# }

# def count_in_graph(g):
#     counts = {}

#     for label, class_list in TYPE_MULTI.items():
#         total = 0
#         for cls in class_list:
#             total += sum(1 for _ in g.subjects(RDF.type, cls) if isinstance(_, (URIRef)))
#             total += sum(1 for _ in g.subjects(RDF.type, cls) if isinstance(_, (URIRef, BNode)) and cls in [OWL.AllDisjointClasses, OWL.AllDisjointProperties, OWL.Restriction])
#         counts[label] = total

#     # Prädikat-basierte Zählungen
#     for label, pred in PREDICATES.items():
#         counts[label] = sum(1 for _ in g.triples((None, pred, None)))
#         counts["domain_CE"] = sum(1 for _, p, o in g.triples((None, RDFS.domain, None)) if isinstance(o, BNode))
#         counts["range_CE"] = sum(1 for _, p, o in g.triples((None, RDFS.range, None)) if isinstance(o, BNode))
#         counts["rdf:type"] =  sum(1 for _ in g.triples((None, RDF.type, OWL.NamedIndividual)))
#     return counts

# # --- 2) Dateien zählen & CSV mit allen Rohzahlen schreiben ---
# rows = []

# for fname in sorted(os.listdir(ONT_DIR)):

#     if not fname.lower().endswith(EXTS):
#         continue
#     depths = []
    
#     path = os.path.join(ONT_DIR, fname)
#     a = Graph()
#     a.parse(path)

#     # print("AllDisjointProperties:",
#     #   list(a.subjects(RDF.type, OWL.AllDisjointProperties)))
    
#     c = count_in_graph(a)


#     row = {"file": fname, **c}
#     print(f"    {row}")
#     rows.append(row)

#     # Spaltenreihenfolge
#     columns = (
#         ['file']
#         + ["rdf:type"]
#         + list(TYPE_MULTI.keys())
#         + list(PREDICATES.keys())
#         + ['domain_CE']
#         + ['range_CE']
#     )

# df = pd.DataFrame(rows, columns=columns)
# df.to_csv(OUT_CSV, index=False)
# print(f"✅ Rohzahlen gespeichert: {OUT_CSV}")

# df = pd.read_csv(OUT_CSV)
# N = len(df)

In [1081]:
# def class_expression_depth(g, node):
#     """
#     Rekursiv: Berechnet die maximale Tiefe einer Class Expression.
#     """
#     # Benannte Klasse → Tiefe 0
#     if not isinstance(node, BNode):
#         return 0

#     max_child_depth = 0

#     # --- Restriction ---
#     if (node, RDF.type, OWL.Restriction) in g:
#         for p in [OWL.someValuesFrom, OWL.allValuesFrom,
#                   OWL.hasValue, OWL.onClass]:
#             for _, _, o in g.triples((node, p, None)):
#                 max_child_depth = max(
#                     max_child_depth,
#                     class_expression_depth(g, o)
#                 )
#         return 1 + max_child_depth

#     # --- Boolean Class Constructors ---
#     for op in [OWL.intersectionOf, OWL.unionOf, OWL.complementOf]:
#         coll = g.value(node, op)
#         if coll:
#             depths = [
#                 class_expression_depth(g, item)
#                 for item in Collection(g, coll)
#             ]
#             return 1 + max(depths, default=0)

#     # Fallback: anonymer Knoten
#     return 1

# ONT_DIR = r"C:\Users\ilove\OneDrive\Uni\Master - Philipps Uni\Master Thesis\Ontos für Evaluation"
# OUT_CSV = "class_expression_depth2.csv"
# EXTS = (".ttl")

# rows = []


# for fname in sorted(os.listdir(ONT_DIR)):

#     if not fname.lower().endswith(EXTS):
#         continue
#     depths = []
    
#     path = os.path.join(ONT_DIR, fname)
#     a = Graph()
#     a.parse(path)

#     # subClassOf
#     for _, _, o in a.triples((None, RDFS.subClassOf, None)):
#         if isinstance(o, BNode):
#             depths.append(class_expression_depth(a, o))

#     # equivalentClass
#     for _, _, o in a.triples((None, OWL.equivalentClass, None)):
#         if isinstance(o, BNode):
#             depths.append(class_expression_depth(a, o))

#     # disjointWith
#     for _, _, o in a.triples((None, OWL.disjointWith, None)):
#         if isinstance(o, BNode):
#             depths.append(class_expression_depth(a, o))
        
#     # --- Verteilung ---
#     depth_counter = Counter(depths)

#     row = {"file": fname}

#     for d, cnt in sorted(depth_counter.items()):
#         row[f"depth_{d}"] = cnt

#     if depths:
#         row["max_depth"] = max(depths)
#         row["avg_depth"] = round(sum(depths) / len(depths), 2)
#         row["count_expressions"] = len(depths)
#     else:
#         row["max_depth"] = 0
#         row["avg_depth"] = 0
#         row["count_expressions"] = 0

#     rows.append(row)

# df = pd.DataFrame(rows)
# df.to_csv(OUT_CSV, index=False)
# print(f"✅ gespeichert: {OUT_CSV}")
# df



In [1082]:
# def complex_class_definitions(gh):
#     """Zählt Klassen, die durch komplexe (BNode-basierte) Beschreibungen definiert sind."""
    
#     # Alle expliziten Klassen (rdf:type owl:Class)
#     all_classes = set(c for c in g.subjects(RDF.type, OWL.Class) if isinstance(c, URIRef))
    
#     # --- Kategorien ---
#     subclass_complex = {s for s, _, o in g.triples((None, RDFS.subClassOf, None)) if isinstance(o, BNode)}
#     subclass_simple = {s for s, _, o in g.triples((None, RDFS.subClassOf, None)) if not isinstance(o, BNode)}
#     equivalent_complex = ({s for s, _, o in g.triples((None, OWL.equivalentClass, None)) if isinstance(o, BNode)})
#     equivalent_simple = ({s for s, _, o in g.triples((None, OWL.equivalentClass, None)) if not isinstance(o, BNode)})
#     disjoint_complex = ({s for s, _, o in g.triples((None, OWL.disjointWith, None)) if isinstance(o, BNode)})
#     disjoint_simple = ({s for s, _, o in g.triples((None, OWL.disjointWith, None)) if not isinstance(o, BNode)})

#     # --- Gesamtmenge komplex definierter Klassen ---
#     complex_classes = subclass_complex | equivalent_complex | disjoint_complex

#     simple_classes = {c for c in all_classes if c not in complex_classes and not isinstance(c, BNode)}

#     return {
#         "subClassOf_complex": len(subclass_complex),
#         "subClassOf_simple": len(subclass_simple),
#         "equivalentClass_complex": len(equivalent_complex),
#         "equivalentClass_simple": len(equivalent_simple),
#         "disjoint_complex": len(disjoint_complex),
#         "disjoint_simple": len(disjoint_simple),
#     }

# # --- Verarbeitung aller Ontologien ---
# ONT_DIR = r"C:\Users\ilove\OneDrive\Uni\Master - Philipps Uni\Master Thesis\Ontos für Evaluation"
# OUT_CSV = "complex_vs_simple.csv"
# EXTS = (".ttl")

# rows = []
# idx = 1

# for fname in sorted(os.listdir(ONT_DIR)):
#     if not fname.lower().endswith(EXTS):
#         continue

#     path = os.path.join(ONT_DIR, fname)
#     print(f"{idx}. {fname}")
#     idx += 1

#     try:
#         g = Graph()
#         g.parse(path)
#         stats = complex_class_definitions(g)
#     except Exception as e:
#         print(f"   ⚠️ Fehler beim Parsen: {e}")
#         stats = {k: 0 for k in ["subClassOf_complex", "equivalentClass_complex",
#                                 "disjoint_complex", "complex_total","simple_classes", "total_classes"]}
#         stats["_error"] = str(e)

#     row = {"file": fname, **stats}
#     rows.append(row)

# # --- DataFrame & Summenzeile ---
# df = pd.DataFrame(rows).fillna(0)


# # --- Speichern ---
# df.to_csv(OUT_CSV, index=False)
# print(f"✅ Ergebnisse gespeichert in {OUT_CSV}")
# df.head(15)

# # __TOTAL__ Zeilen gibt's hier nicht; wir nehmen direkt df
# df = pd.read_csv(OUT_CSV)
# N = len(df)